In [ ]:
!pip install linearmodels
from linearmodels.iv import IV2SLS
from linearmodels.iv import IVLIML
import pandas as pd

The below cells all run the same code, just with different IV setups.

First: LJS vote share + turnout (either total or early) covariates turned on in democratic vote share regressions

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import glob
from difflib import get_close_matches
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.formula.api as smf

try:
    from linearmodels.iv import IV2SLS
except ImportError:
    print("[!] 'linearmodels' not found. Please run: pip install linearmodels")
    exit(1)

# ==========================================
# CONFIGURATION & GLOBAL SETUP
# ==========================================

# --- GLOBAL TOGGLES ---
EXCLUDE_HONAM_YEONGNAM       = False
INCLUDE_TURNOUT_IN_VOTESHARE = True   # Toggle Turnout IV in Vote Share Regressions
INCLUDE_LJS_IN_VOTESHARE     = True   # Toggle LJS Vote Share IV in relevant Regressions

# --- DYNAMIC FILENAME SUFFIX ---
if INCLUDE_LJS_IN_VOTESHARE and INCLUDE_TURNOUT_IN_VOTESHARE:
    IV_SUFFIX = "_LJS_turnout_IV_cov"
elif INCLUDE_LJS_IN_VOTESHARE:
    IV_SUFFIX = "_LJS_IV_cov"
elif INCLUDE_TURNOUT_IN_VOTESHARE:
    IV_SUFFIX = "_turnout_IV_cov"
else:
    IV_SUFFIX = "_no_IV_cov"

os.makedirs('output_data', exist_ok=True)
LOG_FILE_PATH = f'output_data/comprehensive_regression_logs{IV_SUFFIX}.txt'

PCA_VARIANCE_THRESHOLD = 0.90
DEMEAN_GROUP_COLS      = ['province_tag']
MIN_COVERAGE           = 0.50

ELECTION_CONFIGS = {
    'pres19': {
        'election_type':  'presidential',
        'census_csv':     '19th_presidential_election_census.csv',
        'result_csv':     '19th_presidential_election_result.csv',
        'apt_csv_glob':   None,
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'자유한국당',
        'third_pattern':  r'유승민|바른정당',
        'label':          '19th Presidential Election (2017)',
        'year':           2017,
        'election_month': 5,
    },
    21: {
        'election_type':  'general',
        'census_csv':     '21st_election_census.csv',
        'result_csv':     '21st_election_result.csv',
        'apt_csv_glob':   '*21st_election_*_apt_price.csv',
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'미래통합당|자유한국당',
        'third_pattern':  None,
        'label':          '21st General Election (2020)',
        'year':           2020,
        'election_month': 4,
    },
    22: {
        'election_type':  'general',
        'census_csv':     '22nd_election_census.csv',
        'result_csv':     '22nd_election_result.csv',
        'apt_csv_glob':   '*22nd_election_*_apt_price.csv',
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'국민의힘',
        'third_pattern':  r'개혁신당',
        'label':          '22nd General Election (2024)',
        'year':           2024,
        'election_month': 4,
    },
    'pres20': {
        'election_type':         'presidential',
        'census_csv':            '20th_presidential_election_census.csv',
        'result_csv':            '20th_presidential_election_result.csv',
        'apt_csv_glob':          '*20th_presidential_election_*_apt_prices.csv',
        'dem_pattern':           r'더불어민주당',
        'con_pattern':           r'국민의힘',
        'third_pattern':         None,
        'label':                 '20th Presidential Election (2022)',
        'year':                  2022,
        'election_month':        3,
    },
    'pres21': {
        'election_type':         'presidential',
        'census_csv':            '21st_presidential_election_census.csv',
        'result_csv':            '21st_presidential_election_result.csv',
        'apt_csv_glob':          '*21st_presidential_election_*_apt_prices.csv',
        'dem_pattern':           r'더불어민주당',
        'con_pattern':           r'국민의힘',
        'third_pattern':         r'이준석|개혁신당',
        'label':                 '21st Presidential Election (2025)',
        'year':                  2025,
        'election_month':        6,
    },
}

SPECIAL_DONG_NAMES_GENERAL = {
    '거소·선상투표', '관외사전투표', '국외부재자투표',
    '국외부재자투표(공관)', '잘못 투입·구분된 투표지',
}
SPECIAL_DONG_NAMES_PRESIDENTIAL = {
    '거소·선상투표', '관외사전투표', '재외투표',
    '잘못 투입·구분된 투표지',
}
GWANNAESA_LABEL = '관내사전투표'
META_CANDIDATES = {'선거인수', '투표수', '무효 투표수', '기권자수'}

PROV_FULL_TO_SHORT = {
    '서울특별시': 'Seoul',  '부산광역시': 'Busan',   '대구광역시': 'Daegu',
    '인천광역시': 'Incheon','광주광역시': 'Gwangju', '대전광역시': 'Daejeon',
    '울산광역시': 'Ulsan',  '세종특별자치시': 'Sejong',
    '경기도': 'Gyeonggi',  '강원도': 'Gangwon',     '강원특별자치도': 'Gangwon',
    '충청북도': 'Chungbuk', '충청남도': 'Chungnam',
    '전라북도': 'Jeonbuk',  '전북특별자치도': 'Jeonbuk', '전라남도': 'Jeonnam',
    '경상북도': 'Gyeongbuk','경상남도': 'Gyeongnam', '제주특별자치도': 'Jeju',
    '서울': 'Seoul',   '부산': 'Busan',    '대구': 'Daegu',
    '인천': 'Incheon', '광주': 'Gwangju',  '대전': 'Daejeon',
    '울산': 'Ulsan',
}

SGG_TO_PROVINCE_EMP = {
    '가평군':'Gyeonggi','고양시':'Gyeonggi','과천시':'Gyeonggi','광명시':'Gyeonggi',
    '광주시':'Gyeonggi','구리시':'Gyeonggi','군포시':'Gyeonggi','김포시':'Gyeonggi',
    '남양주시':'Gyeonggi','동두천시':'Gyeonggi','부천시':'Gyeonggi','성남시':'Gyeonggi',
    '수원시':'Gyeonggi','시흥시':'Gyeonggi','안산시':'Gyeonggi','안성시':'Gyeonggi',
    '안양시':'Gyeonggi','양주시':'Gyeonggi','양평군':'Gyeonggi','여주군':'Gyeonggi',
    '여주시':'Gyeonggi','연천군':'Gyeonggi','오산시':'Gyeonggi','용인시':'Gyeonggi',
    '의왕시':'Gyeonggi','의정부시':'Gyeonggi','이천시':'Gyeonggi','파주시':'Gyeonggi',
    '평택시':'Gyeonggi','포천시':'Gyeonggi','하남시':'Gyeonggi','화성시':'Gyeonggi',
    '강릉시':'Gangwon','고성군':'Gangwon','동해시':'Gangwon','삼척시':'Gangwon',
    '속초시':'Gangwon','양구군':'Gangwon','양양군':'Gangwon','영월군':'Gangwon',
    '원주시':'Gangwon','인제군':'Gangwon','정선군':'Gangwon','철원군':'Gangwon',
    '춘천시':'Gangwon','태백시':'Gangwon','평창군':'Gangwon','홍천군':'Gangwon',
    '화천군':'Gangwon','횡성군':'Gangwon','괴산군':'Chungbuk','단양군':'Chungbuk',
    '보은군':'Chungbuk','영동군':'Chungbuk','옥천군':'Chungbuk','음성군':'Chungbuk',
    '제천시':'Chungbuk','증평군':'Chungbuk','진천군':'Chungbuk','청원군':'Chungbuk',
    '청주시':'Chungbuk','충주시':'Chungbuk','계룡시':'Chungnam','공주시':'Chungnam',
    '금산군':'Chungnam','논산시':'Chungnam','당진시':'Chungnam','보령시':'Chungnam',
    '부여군':'Chungnam','서산시':'Chungnam','서천군':'Chungnam','아산시':'Chungnam',
    '연기군':'Chungnam','예산군':'Chungnam','청양군':'Chungnam','천안시':'Chungnam',
    '태안군':'Chungnam','홍성군':'Chungnam','고창군':'Jeonbuk','군산시':'Jeonbuk',
    '김제시':'Jeonbuk','남원시':'Jeonbuk','무주군':'Jeonbuk','부안군':'Jeonbuk',
    '순창군':'Jeonbuk','완주군':'Jeonbuk','익산시':'Jeonbuk','임실군':'Jeonbuk',
    '장수군':'Jeonbuk','전주시':'Jeonbuk','정읍시':'Jeonbuk','진안군':'Jeonbuk',
    '강진군':'Jeonnam','고흥군':'Jeonnam','곡성군':'Jeonnam','구례군':'Jeonnam',
    '나주시':'Jeonnam','담양군':'Jeonnam','목포시':'Jeonnam','무안군':'Jeonnam',
    '보성군':'Jeonnam','순천시':'Jeonnam','신안군':'Jeonnam','여수시':'Jeonnam',
    '영광군':'Jeonnam','영암군':'Jeonnam','완도군':'Jeonnam','장성군':'Jeonnam',
    '장흥군':'Jeonnam','진도군':'Jeonnam','함평군':'Jeonnam','화순군':'Jeonnam',
    '광양시':'Jeonnam','해남군':'Jeonnam','경산시':'Gyeongbuk','경주시':'Gyeongbuk',
    '고령군':'Gyeongbuk','구미시':'Gyeongbuk','김천시':'Gyeongbuk','문경시':'Gyeongbuk',
    '봉화군':'Gyeongbuk','상주시':'Gyeongbuk','성주군':'Gyeongbuk','안동시':'Gyeongbuk',
    '영덕군':'Gyeongbuk','영양군':'Gyeongbuk','영주시':'Gyeongbuk','영천시':'Gyeongbuk',
    '예천군':'Gyeongbuk','울릉군':'Gyeongbuk','울진군':'Gyeongbuk','의성군':'Gyeongbuk',
    '청도군':'Gyeongbuk','청송군':'Gyeongbuk','칠곡군':'Gyeongbuk','포항시':'Gyeongbuk',
    '거제시':'Gyeongnam','거창군':'Gyeongnam','김해시':'Gyeongnam','남해군':'Gyeongnam',
    '밀양시':'Gyeongnam','사천시':'Gyeongnam','산청군':'Gyeongnam','양산시':'Gyeongnam',
    '의령군':'Gyeongnam','진주시':'Gyeongnam','창녕군':'Gyeongnam','창원시':'Gyeongnam',
    '통영시':'Gyeongnam','하동군':'Gyeongnam','함안군':'Gyeongnam','함양군':'Gyeongnam',
    '합천군':'Gyeongnam','서귀포시':'Jeju','제주시':'Jeju',
}

PARTISAN_REGION_PROVINCES = {
    'Honam':   {'Jeonnam', 'Jeonbuk', 'Gwangju', '전남', '전북', '광주'},
    'Yeongnam':{'Gyeongbuk', 'Gyeongnam', 'Daegu', 'Busan', 'Ulsan', '경북', '경남', '대구', '부산', '울산'},
}

AGE_GENDER_COLS = [
    'pct_m_1824', 'pct_m_2529', 'pct_m_3034', 'pct_m_3539', 'pct_m_4044', 'pct_m_4549',
    'pct_m_5054', 'pct_m_5559', 'pct_m_6064', 'pct_m_6569', 'pct_m_70plus',
    'pct_f_1824', 'pct_f_2529', 'pct_f_3034', 'pct_f_3539', 'pct_f_4044', 'pct_f_4549',
    'pct_f_5054', 'pct_f_5559', 'pct_f_6064', 'pct_f_6569', 'pct_f_70plus',
]

EMPLOYMENT_FEATURE_COLS = [
    'emp_men_total', 'emp_men_1529', 'emp_men_3049', 'emp_men_5064', 'emp_men_65plus',
    'emp_wmn_total', 'emp_wmn_1529', 'emp_wmn_3049', 'emp_wmn_5064', 'emp_wmn_65plus',
    'occ_wage_share', 'occ_regular_share', 'occ_nonwage_share',
    'job_professional', 'job_clerical', 'job_service_sales',
    'job_skilled_machine', 'job_simple_labor', 'job_farming',
    'ind_agriculture', 'ind_manufacturing', 'ind_construction',
    'ind_retail_food', 'ind_transport_finance', 'ind_services',
    'nonp_male_share', 'nonp_young_share', 'nonp_middle_share', 'nonp_old_share',
]

# ==========================================
# FILE WRITER HELPER
# ==========================================
def log_to_file(text: str, mode='a'):
    print(text)
    with open(LOG_FILE_PATH, mode, encoding='utf-8') as f:
        f.write(text + "\n")

# ==========================================
# NORMALISATION & IO UTILITIES
# ==========================================
def _read_csv_auto(path: str, **kwargs) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding='utf-8', **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='cp949', **kwargs)

def _to_numeric_safe(series: pd.Series) -> pd.Series:
    return (series.astype(str).str.strip()
            .replace(['-', '.', '', 'nan'], np.nan)
            .pipe(pd.to_numeric, errors='coerce'))

def normalize_dong_name(name: str) -> str:
    if not isinstance(name, str): return ""
    name = re.sub(r'\(.*?\)', '', name).strip().replace('.', '·')
    name = re.sub(r'제(\d)', r'\1', name)
    name = re.sub(r'·\d+', '', name)
    name = re.sub(r'(\d+)(동|읍|면)$', r'\2', name)
    return re.sub(r'\s+', ' ', name)

def split_admin_tokens(name: str) -> list:
    tokens, buf = [], []
    for ch in name:
        buf.append(ch)
        if ch in '시군구' and len(buf) >= 2:
            tokens.append(''.join(buf))
            buf = []
    if buf: tokens.append(''.join(buf))
    return [t for t in tokens if t]

def normalize_sigungu(name: str) -> list:
    if not isinstance(name, str): return []
    name = re.sub(r'\(.*?\)', '', name).strip()
    if not name: return []
    tokens = split_admin_tokens(name)
    if not tokens:
        stripped = re.sub(r'[시군구갑을병정무]$', '', name).strip()
        return [stripped] if stripped else []
    si_gun_count = sum(1 for t in tokens if t[-1] in '시군' and len(t) >= 2)
    gu_count     = sum(1 for t in tokens if t[-1] == '구'  and len(t) >= 2)
    ordered = tokens if (si_gun_count >= 2 or (si_gun_count == 0 and gu_count >= 2)) else list(reversed(tokens))
    candidates = []
    for t in ordered:
        key = re.sub(r'[시군구]$', '', t).strip()
        if key and key not in candidates: candidates.append(key)
    return candidates

def get_election_periods(month: int, year: int) -> dict:
    if month <= 6:
        prev = year - 1
        return {'half_periods': [f'{prev}.1/2', f'{prev}.2/2'], 'grdp_year': prev}
    else:
        prev = year - 1
        return {'half_periods': [f'{prev}.2/2', f'{year}.1/2'], 'grdp_year': prev}

def parse_employment_sgg(region_name: str) -> tuple:
    if not isinstance(region_name, str): return ('', '')
    parts = region_name.strip().split(None, 1)
    if not parts: return ('', '')
    prov = parts[0]
    if len(parts) == 1:
        prov_eng = SGG_TO_PROVINCE_EMP.get(prov, '')
        primary  = re.sub(r'[시군구]$', '', prov).strip()
        return (prov_eng, primary)
    sgg_raw  = parts[1]
    sgg_norm = normalize_sigungu(sgg_raw)
    primary  = sgg_norm[0] if sgg_norm else re.sub(r'[시군구]$', '', sgg_raw).strip()
    prov_eng = PROV_FULL_TO_SHORT.get(prov, prov)
    return (prov_eng, primary)

# ==========================================
# 1. DEMOGRAPHIC, ASSET & STATION DATA LOADERS
# ==========================================
def load_polling_stations(elec_id) -> pd.DataFrame:
    # Look strictly for normal-day station files to proxy physical density friction
    paths_to_try = [
        f"stations_normal_{elec_id}.csv",
        f"polling_stations_{elec_id}.csv",
        f"stations_normal_20240410.csv" if str(elec_id) == '22' else f"stations_normal_{elec_id}.csv"
    ]

    df = pd.DataFrame()
    for path in paths_to_try:
        if os.path.exists(path):
            df = _read_csv_auto(path)
            break

    if df.empty: return df

    if 'province_tag' not in df.columns and 'sdName' in df.columns:
        df['province_tag'] = df['sdName'].map(PROV_FULL_TO_SHORT).fillna(df['sdName'])
        def extract_sgg(x):
            cands = normalize_sigungu(x)
            return cands[0] if cands else re.sub(r'[시군구]$', '', str(x)).strip()
        df['primary_sgg'] = df['wiwName'].apply(extract_sgg)
        df['dong_norm']   = df['emdName'].apply(normalize_dong_name)

        mask_seoul = df['province_tag'] == 'Seoul'
        mask_g3 = df['primary_sgg'].isin(['강남', '서초', '송파'])
        df.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
        df.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    if 'normal_station_count' not in df.columns:
        if 'station_count' in df.columns:
            df = df.rename(columns={'station_count': 'normal_station_count'})
        else:
            df['normal_station_count'] = 1

    return df[['province_tag', 'primary_sgg', 'dong_norm', 'normal_station_count']]


def _detect_year_prefix(df: pd.DataFrame) -> str:
    for col in df.columns:
        m = re.match(r'(\d{4}년\d{2}월)_계_총인구수', col)
        if m: return m.group(1)
    raise ValueError("Cannot detect census year prefix.")

def load_census_csv(csv_path: str) -> pd.DataFrame:
    if not csv_path or not os.path.exists(csv_path): return pd.DataFrame()
    try:
        df = _read_csv_auto(csv_path, low_memory=False)
        prefix = _detect_year_prefix(df)
        voting_age_cols = ([f"{prefix}_계_{a}세" for a in range(18, 100)] + [f"{prefix}_계_100세 이상"])
        cols_4059 = [f"{prefix}_계_{a}세" for a in range(40, 60)]
        all_target_cols = list(voting_age_cols)
        for g in ['남', '여']:
            for a in range(18, 100):
                all_target_cols.append(f"{prefix}_{g}_{a}세")
            all_target_cols.append(f"{prefix}_{g}_100세 이상")

        for col in set(all_target_cols):
            if col in df.columns:
                df[col] = (df[col].astype(str).str.replace(',', '', regex=False).pipe(pd.to_numeric, errors='coerce').fillna(0))

        df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)
        df = df[df['total_voting_pop'] > 0].copy()

        ranges = [(18,25,'1824'),(25,30,'2529'),(30,35,'3034'),(35,40,'3539'),
                  (40,45,'4044'),(45,50,'4549'),(50,55,'5054'),(55,60,'5559'),
                  (60,65,'6064'),(65,70,'6569')]

        for g, g_str in [('남','m'), ('여','f')]:
            for r_start, r_end, r_str in ranges:
                cols = [f"{prefix}_{g}_{a}세" for a in range(r_start, r_end)]
                df[f'pct_{g_str}_{r_str}'] = (df[[c for c in cols if c in df.columns]].sum(axis=1) / df['total_voting_pop'])
            cols_70 = ([f"{prefix}_{g}_{a}세" for a in range(70, 100)] + [f"{prefix}_{g}_100세 이상"])
            df[f'pct_{g_str}_70plus'] = (df[[c for c in cols_70 if c in df.columns]].sum(axis=1) / df['total_voting_pop'])

        df['demographic_propensity'] = (df[[c for c in cols_4059 if c in df.columns]].sum(axis=1) / df['total_voting_pop'])

        def extract_census_keys(admin_name):
            if not isinstance(admin_name, str): return [], ""
            clean = re.sub(r'\(.*?\)', '', admin_name).strip()
            parts = clean.split()
            dong_norm = normalize_dong_name(parts[-1]) if parts else ""
            sgg_cands = []
            for token in reversed(parts[:-1]):
                for c in normalize_sigungu(token):
                    if c not in sgg_cands: sgg_cands.append(c)
            return sgg_cands, dong_norm

        rows = []
        for _, row in df.iterrows():
            sgg_cands, dong_norm = extract_census_keys(row['행정구역'])
            row_dict = {
                'sgg_candidates': sgg_cands,
                'primary_sgg':    sgg_cands[0] if sgg_cands else "",
                'dong_norm':      dong_norm,
                'demographic_propensity': row['demographic_propensity'],
            }
            for col in AGE_GENDER_COLS:
                row_dict[col] = row.get(col, np.nan)
            rows.append(row_dict)

        return pd.DataFrame(rows)
    except Exception as e:
        return pd.DataFrame()

def load_apt_csv(glob_pattern: str, elec_id: str) -> pd.DataFrame:
    if not glob_pattern: return pd.DataFrame()

    processed_path = f"output_data/processed_apt_data_{elec_id}.csv"

    if os.path.exists(processed_path):
        try:
            return _read_csv_auto(processed_path)
        except Exception:
            pass

    file_list = glob.glob(glob_pattern)
    if not file_list: return pd.DataFrame()

    df_list = []
    for file in file_list:
        try:
            df_list.append(_read_csv_auto(file, skiprows=15))
        except Exception: pass
    if not df_list: return pd.DataFrame()

    df = pd.concat(df_list, ignore_index=True)
    try:
        df['거래금액(만원)'] = pd.to_numeric(df['거래금액(만원)'].astype(str).str.replace(',', '').str.strip(), errors='coerce')
        df['전용면적(㎡)']  = pd.to_numeric(df['전용면적(㎡)'], errors='coerce')
        df['price_per_sqm']  = df['거래금액(만원)'] / df['전용면적(㎡)']

        def parse_loc(x):
            parts = str(x).split()
            prov  = PROV_FULL_TO_SHORT.get(parts[0], parts[0]) if parts else ""
            sgg   = normalize_sigungu(parts[1])[0] if len(parts) > 2 and normalize_sigungu(parts[1]) else ""
            dong  = normalize_dong_name(parts[-1]) if parts else ""
            return pd.Series([prov, sgg, dong])

        df[['prov','sgg','dong_norm']] = df['시군구'].apply(parse_loc)

        mask_seoul = df['prov'] == 'Seoul'
        mask_g3 = df['sgg'].isin(['강남', '서초', '송파'])
        df.loc[mask_seoul & mask_g3, 'prov'] = 'Seoul (Gangnam3gu)'
        df.loc[mask_seoul & ~mask_g3, 'prov'] = 'Seoul (Non-Gangnam3gu)'

        apt_agg = (df.groupby(['prov','sgg','dong_norm'])['price_per_sqm'].median().reset_index()
                   .rename(columns={'price_per_sqm': 'median_apt_price_sqm'}))

        apt_agg.to_csv(processed_path, index=False)
        return apt_agg
    except Exception as e:
        return pd.DataFrame()

def _avg_half_periods(df: pd.DataFrame, base_cols: list) -> pd.Series:
    available = [c for c in base_cols if c in df.columns]
    if not available: return pd.Series(np.nan, index=df.index)
    return df[available].apply(pd.to_numeric, errors='coerce').mean(axis=1)

def load_employment_sgg(election_key, **kwargs) -> pd.DataFrame:
    cfg    = ELECTION_CONFIGS[election_key if type(election_key)==str else int(election_key)]
    month  = cfg['election_month']
    year   = cfg['year']
    periods_info = get_election_periods(month, year)
    periods      = periods_info['half_periods']

    def read_emp(path):
        if not os.path.exists(path): return pd.DataFrame()
        df = _read_csv_auto(path, low_memory=False)
        return df[[c for c in df.columns if not str(c).startswith('Unnamed')]]

    age_map = {'total':'계','1529':'15 - 29세','3049':'30 - 49세',
               '5064':'50 - 64세','65plus':'65세이상'}

    def extract_gender_emp(csv_path, gender_prefix):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df_rate = df[df['항목'].str.strip() == '고용률 (%)'].copy()
        records = {}
        for suf, kor_age in age_map.items():
            sub = df_rate[df_rate['연령별'].str.strip() == kor_age].copy()
            if sub.empty: continue
            val_col = f'{gender_prefix}_{suf}'
            sub[val_col] = _avg_half_periods(sub, periods)
            prov_sgg = sub['행정구역별'].apply(parse_employment_sgg)
            sub['province_tag'] = prov_sgg.apply(lambda x: x[0])
            sub['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
            records[val_col] = sub.groupby(['province_tag','primary_sgg'])[val_col].mean()
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_men = extract_gender_emp(kwargs.get('men_csv', 'men_employment_data.csv'), 'emp_men')
    df_wmn = extract_gender_emp(kwargs.get('women_csv', 'women_employment_data.csv'), 'emp_wmn')

    def extract_occ_type(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df_total = df[df['종사상지위별'].str.strip() == '계'].copy()
        df_total['_total'] = _avg_half_periods(df_total, periods)
        prov_sgg = df_total['행정구역별'].apply(parse_employment_sgg)
        df_total['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df_total['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df_total.set_index(['province_tag','primary_sgg'])['_total']
        records = {}
        for col_name, kor_cat in {'occ_wage_share':'임금근로자', 'occ_regular_share':'- 상용근로자', 'occ_nonwage_share':'비임금근로자'}.items():
            sub = df[df['종사상지위별'].str.strip() == kor_cat].copy()
            if sub.empty: continue
            sub['_val'] = _avg_half_periods(sub, periods)
            prov_sgg2 = sub['행정구역별'].apply(parse_employment_sgg)
            sub['province_tag'] = prov_sgg2.apply(lambda x: x[0])
            sub['primary_sgg']  = prov_sgg2.apply(lambda x: x[1])
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_occ_type = extract_occ_type(kwargs.get('occ_type_csv', 'occupation_type_employment_data.csv'))

    def extract_job_shares(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df[df['항목'].str.strip() == '취업자[천명]'].copy()
        df['_val'] = _avg_half_periods(df, periods)
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df[df['직업별'].str.strip() == '계'].set_index(['province_tag','primary_sgg'])['_val']
        records = {}
        for col_name, kor_cat in {'job_professional': '관리자, 전문가 및 관련종사자', 'job_clerical': '사무 종사자', 'job_service_sales': '서비스·판매 종사자', 'job_skilled_machine':'기능·기계조작·조립 종사자', 'job_simple_labor': '단순노무 종사자', 'job_farming': '농림어업 숙련종사자'}.items():
            sub = df[df['직업별'].str.strip() == kor_cat].copy()
            if sub.empty: continue
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_job = extract_job_shares(kwargs.get('occ_csv', 'occupation_employment_data.csv'))

    def extract_ind_shares(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df[df['항목'].str.strip() == '취업자[천명]'].copy()
        df['_val'] = _avg_half_periods(df, periods)
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df[df['산업별'].str.strip() == '계'].set_index(['province_tag','primary_sgg'])['_val']
        records = {}
        for col_name, kor_cat in {'ind_agriculture': '농업, 임업 및 어업 (A)', 'ind_manufacturing': '광·제조업(B,C)', 'ind_construction': '건설업 (F) ', 'ind_retail_food': '도소매·숙박음식업(G,I)', 'ind_transport_finance': '전기·운수·통신·금융(D,H,J,K)', 'ind_services': '사업·개인·공공서비스 및 기타(E,L~U)'}.items():
            sub = df[df['산업별'].str.strip() == kor_cat.strip()].copy()
            if sub.empty:
                match = get_close_matches(kor_cat.strip(), df['산업별'].str.strip().unique().tolist(), n=1, cutoff=0.85)
                if match: sub = df[df['산업별'].str.strip() == match[0]].copy()
            if sub.empty: continue
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_ind = extract_ind_shares(kwargs.get('ind_csv', 'industry_employment_data.csv'))

    def extract_nonpart(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df.iloc[1:].copy()
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        sub_suffixes = {'total':'', 'male':'.1', 'young':'.3', 'middle':'.4'}
        period_vals  = {k: [] for k in sub_suffixes}
        for period in periods:
            for k, suf in sub_suffixes.items():
                if period + suf in df.columns: period_vals[k].append(_to_numeric_safe(df[period + suf]))
        def mean_series(lst): return pd.concat(lst, axis=1).mean(axis=1) if lst else pd.Series(np.nan, index=df.index)
        total  = mean_series(period_vals['total'])
        result = df[['province_tag','primary_sgg']].copy()
        result['nonp_male_share']   = mean_series(period_vals['male'])   / total.replace(0, np.nan)
        result['nonp_young_share']  = mean_series(period_vals['young'])  / total.replace(0, np.nan)
        result['nonp_middle_share'] = mean_series(period_vals['middle']) / total.replace(0, np.nan)
        return (result.groupby(['province_tag','primary_sgg'])[['nonp_male_share','nonp_young_share','nonp_middle_share']].mean().reset_index())

    df_nonp = extract_nonpart(kwargs.get('nonp_csv', 'non_participant_employment_data.csv'))

    merge_key = ['province_tag','primary_sgg']
    frames = [f for f in [df_men, df_wmn, df_occ_type, df_job, df_ind, df_nonp] if not f.empty]
    if not frames: return pd.DataFrame(columns=merge_key + EMPLOYMENT_FEATURE_COLS)

    df_emp = frames[0]
    for frame in frames[1:]: df_emp = df_emp.merge(frame, on=merge_key, how='outer')
    for col in EMPLOYMENT_FEATURE_COLS:
        if col not in df_emp.columns: df_emp[col] = np.nan

    if not df_emp.empty:
        mask_seoul = df_emp['province_tag'] == 'Seoul'
        mask_g3 = df_emp['primary_sgg'].isin(['강남', '서초', '송파'])
        df_emp.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
        df_emp.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    return df_emp

# ==========================================
# 2. ELECTION CSV LOADER & MATCHER
# ==========================================
def load_election_csv(csv_path: str, dem_pattern: str, con_pattern: str,
                      third_pattern: str = None, election_type: str = 'general'):
    try:
        df = _read_csv_auto(csv_path, low_memory=False)
    except Exception: return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if election_type == 'presidential':
        df = df.rename(columns={'구시군명': '선거구명', '읍면동명': '법정읍면동명'})
        special_dong_names = SPECIAL_DONG_NAMES_PRESIDENTIAL
    else:
        special_dong_names = SPECIAL_DONG_NAMES_GENERAL

    df['득표수']    = pd.to_numeric(df['득표수'], errors='coerce').fillna(0).astype(int)
    df['is_dem']   = df['후보자'].str.contains(dem_pattern,   case=False, na=False)

    if election_type == 'general':
        sejong_gap_kim = (df['선거구명'].astype(str).str.replace(' ', '').str.contains('세종.*갑', na=False)) & \
                         (df['후보자'].astype(str).str.contains('김종민', na=False))
        df.loc[sejong_gap_kim, 'is_dem'] = True

    df['is_con']   = df['후보자'].str.contains(con_pattern,   case=False, na=False)
    df['is_third'] = df['후보자'].str.contains(third_pattern, case=False, na=False) if third_pattern else False
    df['is_meta']  = df['후보자'].isin(META_CANDIDATES)
    df['is_early'] = df['투표구명'] == GWANNAESA_LABEL

    dong_key  = ['시도명','선거구명','법정읍면동명']
    const_key = ['시도명','선거구명']

    def sgg_cands(name):
        if not isinstance(name, str): return []
        if '_' in name: return normalize_sigungu(name.split('_', 1)[1])
        return normalize_sigungu(re.sub(r'[갑을병정무]$', '', name).strip())

    df_geo   = df[~df['법정읍면동명'].isin(special_dong_names)].copy()
    df_votes = df_geo[~df_geo['is_meta']].copy()

    gn_dem = df_votes[df_votes['is_dem'] & df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_dem')
    gn_tot = df_votes[df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_total')
    gn_third = df_votes[df_votes['is_third'] & df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_third')

    sd_dem = df_votes[df_votes['is_dem'] & ~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_dem')
    sd_tot = df_votes[~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_total')
    sd_third = df_votes[df_votes['is_third'] & ~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_third')

    sum_people_dong = (df_geo[~df_geo['is_early'] & (df_geo['후보자'] == '선거인수')].groupby(dong_key)['득표수'].sum().reset_index(name='sum_people'))
    sum_vote_geo    = (df_geo[df_geo['후보자'] == '투표수'].groupby(dong_key)['득표수'].sum().reset_index(name='sum_vote_geo'))

    df_dong = gn_dem.copy()
    for frame in (gn_tot, gn_third, sd_dem, sd_tot, sd_third, sum_people_dong, sum_vote_geo):
        df_dong = df_dong.merge(frame, on=dong_key, how='outer')
    df_dong = df_dong.fillna(0)

    gn_ppl_dong = (df_geo[df_geo['is_early'] & (df_geo['후보자'] == '선거인수')].groupby(dong_key)['득표수'].sum().reset_index(name='_gn_ppl'))
    df_dong = df_dong.merge(gn_ppl_dong, on=dong_key, how='left')
    df_dong['sum_people'] = df_dong['sum_people'] + df_dong['_gn_ppl'].fillna(0)
    df_dong.drop(columns=['_gn_ppl'], inplace=True)

    df_dong['sgg_candidates'] = df_dong['선거구명'].apply(sgg_cands)
    df_dong['primary_sgg']    = df_dong['sgg_candidates'].apply(lambda x: x[0] if x else "")
    df_dong['dong_norm']      = df_dong['법정읍면동명'].apply(normalize_dong_name)
    df_dong['province_tag']   = df_dong['시도명'].map(PROV_FULL_TO_SHORT).fillna(df_dong['시도명'])
    df_dong['area2_name']     = df_dong['선거구명']

    mask_seoul = df_dong['province_tag'] == 'Seoul'
    mask_g3 = df_dong['primary_sgg'].isin(['강남', '서초', '송파'])
    df_dong.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
    df_dong.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    df_gw   = df[df['법정읍면동명'].isin(special_dong_names)]
    df_gw_v = df_gw[~df_gw['is_meta']]

    go_dem_c  = df_gw_v[df_gw_v['is_dem']].groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_dem')
    go_tot_c  = df_gw_v.groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_total')
    go_turn_c = (df_gw[df_gw['후보자'] == '투표수'].groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_turnout'))

    df_const = df_dong.groupby(const_key)['sum_people'].sum().reset_index(name='sum_people')
    for frame in (go_dem_c, go_tot_c, go_turn_c):
        df_const = df_const.merge(frame, on=const_key, how='left')
    df_const = df_const.fillna(0)

    return df_dong, df_const, pd.DataFrame()

def merge_dong_with_covariates(df_election: pd.DataFrame, df_census: pd.DataFrame,
                                df_apt: pd.DataFrame, df_emp: pd.DataFrame) -> pd.DataFrame:
    if not df_census.empty:
        census_lookup = {}
        census_by_sgg = {}
        for _, row in df_census.iterrows():
            dnorm = row['dong_norm']
            covs  = {'demographic_propensity': row['demographic_propensity']}
            for c in AGE_GENDER_COLS: covs[c] = row.get(c, np.nan)
            for sgg in row['sgg_candidates']:
                census_lookup[(sgg, dnorm)] = covs
                census_by_sgg.setdefault(sgg, []).append(dnorm)

        results = []
        for _, row in df_election.iterrows():
            covs, dk, sgc = None, row['dong_norm'], row['sgg_candidates'] if isinstance(row['sgg_candidates'], list) else [row['primary_sgg']]
            if (row['primary_sgg'], dk) in census_lookup: covs = census_lookup[(row['primary_sgg'], dk)]
            if covs is None and '·' in dk:
                if (row['primary_sgg'], dk.replace('·', '')) in census_lookup: covs = census_lookup[(row['primary_sgg'], dk.replace('·', ''))]
            if covs is None:
                for sgg in sgc[1:]:
                    if (sgg, dk) in census_lookup: covs = census_lookup[(sgg, dk)]; break
            if covs is None:
                for sgg in sgc:
                    pool = census_by_sgg.get(sgg, [])
                    if pool:
                        m = get_close_matches(dk, pool, n=1, cutoff=0.82)
                        if m and (sgg, m[0]) in census_lookup: covs = census_lookup[(sgg, m[0])]; break
            if covs is None: covs = {k: np.nan for k in ['demographic_propensity'] + AGE_GENDER_COLS}
            rd = row.to_dict(); rd.update(covs)
            results.append(rd)
        df_out = pd.DataFrame(results)
    else:
        df_out = df_election.copy()

    if not df_apt.empty:
        df_out = df_out.merge(df_apt, left_on=['province_tag','primary_sgg','dong_norm'], right_on=['prov','sgg','dong_norm'], how='left')
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out.groupby('primary_sgg')['median_apt_price_sqm'].transform('median'))
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out.groupby('province_tag')['median_apt_price_sqm'].transform('median'))
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out['median_apt_price_sqm'].median())
        df_out['log_apt_price'] = np.log1p(df_out['median_apt_price_sqm'])
    else:
        df_out['log_apt_price'] = np.nan

    if not df_emp.empty:
        df_out = df_out.merge(df_emp[['province_tag','primary_sgg'] + EMPLOYMENT_FEATURE_COLS], on=['province_tag','primary_sgg'], how='left')
        for col in EMPLOYMENT_FEATURE_COLS:
            if col not in df_out.columns: df_out[col] = np.nan
            df_out[col] = df_out[col].fillna(df_out.groupby('province_tag')[col].transform('median')).fillna(df_out[col].median())
    else:
        for col in EMPLOYMENT_FEATURE_COLS: df_out[col] = np.nan

    return df_out[df_out['pct_f_4044'].notna()].copy()

def prep_election_data(df_dong_raw: pd.DataFrame, df_const_raw: pd.DataFrame,
                       df_census: pd.DataFrame, df_apt: pd.DataFrame,
                       df_emp: pd.DataFrame, df_stations: pd.DataFrame, elec_id: str) -> pd.DataFrame:
    dm = merge_dong_with_covariates(df_dong_raw, df_census, df_apt, df_emp)
    dm = dm[(dm['in_precinct_early_total'] > 50) & (dm['same_day_total'] > 50)].copy()

    no_dem = (dm['in_precinct_early_dem'] == 0) & (dm['same_day_dem'] == 0)
    if dm[no_dem].shape[0] > 0:
        dm = dm[~no_dem].copy()

    const_sums = dm.groupby(['시도명', '선거구명'])[['in_precinct_early_total', 'same_day_total']].transform('sum')
    dm['dong_weight'] = (dm['in_precinct_early_total'] + dm['same_day_total']) / (const_sums['in_precinct_early_total'] + const_sums['same_day_total']).replace(0, np.nan)

    df_const_sub = df_const_raw[['시도명', '선거구명', 'out_precinct_early_dem', 'out_precinct_early_total', 'out_precinct_early_turnout']].drop_duplicates()
    dm = dm.merge(df_const_sub, on=['시도명', '선거구명'], how='left')

    dm['out_precinct_alloc_dem']     = dm['out_precinct_early_dem'].fillna(0) * dm['dong_weight'].fillna(0)
    dm['out_precinct_alloc_tot']     = dm['out_precinct_early_total'].fillna(0) * dm['dong_weight'].fillna(0)
    dm['out_precinct_alloc_turnout'] = dm['out_precinct_early_turnout'].fillna(0) * dm['dong_weight'].fillna(0)

    dm['vote_share'] = ((dm['in_precinct_early_dem'] + dm['same_day_dem'] + dm['out_precinct_alloc_dem']) /
                        (dm['in_precinct_early_total'] + dm['same_day_total'] + dm['out_precinct_alloc_tot'])).fillna(0)
    dm['turnout'] = (dm['sum_vote_geo'] + dm['out_precinct_alloc_turnout']) / (dm['sum_people'] + dm['out_precinct_alloc_turnout']).replace(0, np.nan)
    dm['early_turnout'] = (dm['in_precinct_early_total'] + dm['out_precinct_alloc_tot']) / (dm['sum_people'] + dm['out_precinct_alloc_turnout']).replace(0, np.nan)
    dm['early_dem_share'] = (dm['in_precinct_early_dem'] + dm['out_precinct_alloc_dem']) / (dm['in_precinct_early_total'] + dm['out_precinct_alloc_tot']).replace(0, np.nan)
    dm['third_vote_share'] = ((dm['in_precinct_early_third'] + dm['same_day_third']) / (dm['in_precinct_early_total'] + dm['same_day_total'])).fillna(0)

    # ---------------------------------------------------------
    # INSTRUMENT INTEGRATION: voters_per_station
    # ---------------------------------------------------------
    if not df_stations.empty:
        dm = dm.merge(df_stations, on=['province_tag', 'primary_sgg', 'dong_norm'], how='left')
        dm['normal_station_count'] = dm['normal_station_count'].fillna(1)
    else:
        dm['normal_station_count'] = 1

    dm['voters_per_station'] = dm['sum_people'] / dm['normal_station_count']

    # Strict Local Demeaning (Province + Primary SGG Level)
    dm['voters_per_station_dm'] = dm['voters_per_station'] - dm.groupby(['province_tag', 'primary_sgg'])['voters_per_station'].transform('mean')

    if dm['voters_per_station_dm'].std(skipna=True) > 0:
        dm['voters_per_station_dm'] = StandardScaler().fit_transform(dm[['voters_per_station_dm']].fillna(0))
    else:
        dm['voters_per_station_dm'] = 0.0

    dm['election_id'] = str(elec_id)
    return dm

# ==========================================
# 4. PCA COLLINEARITY PIPELINE
# ==========================================
def reduce_pca(df: pd.DataFrame, feature_cols: list, threshold: float, name_label: str) -> tuple:
    available = [c for c in feature_cols if c in df.columns and df[c].notna().mean() >= MIN_COVERAGE]
    if len(available) < 2: return df.copy(), [], None, available

    valid_mask = df[available].notna().all(axis=1)
    if valid_mask.sum() < 30: return df.copy(), [], None, available

    X = df.loc[valid_mask, available].values.astype(float)
    X_scaled = StandardScaler().fit_transform(X)

    pca_probe = PCA().fit(X_scaled)
    cumvar = np.cumsum(pca_probe.explained_variance_ratio_)
    n_components = min(int(np.searchsorted(cumvar, threshold)) + 1, len(available))

    pca_final = PCA(n_components=n_components)
    components = pca_final.fit_transform(X_scaled)

    pc_cols = [f'pc{i+1}' for i in range(n_components)]
    df_out = df.copy()
    for col in pc_cols: df_out[col] = np.nan
    df_out.loc[valid_mask, pc_cols] = components

    loadings = pd.DataFrame(pca_final.components_.T, columns=pc_cols, index=available)
    out_path = f'output_data/pca_loadings_{name_label}.csv'
    loadings.to_csv(out_path)
    log_to_file(f"  -> Saved PCA Loadings for {name_label} to {out_path}")

    return df_out, pc_cols, pca_final, available

def prepare_continuous_covariates(df: pd.DataFrame, raw_age_cols: list, raw_emp_cols: list, label: str) -> tuple:
    combined_raw = list(raw_age_cols) + list(raw_emp_cols)
    if 'log_apt_price' in df.columns: combined_raw.append('log_apt_price')

    df, pc_cols, pca_model, used_features = reduce_pca(df, combined_raw, PCA_VARIANCE_THRESHOLD, label)
    pc_cols = [c for c in pc_cols if df[c].notna().mean() >= MIN_COVERAGE]

    dm_cols = []
    if pc_cols:
        active_groups = [g for g in DEMEAN_GROUP_COLS if g in df.columns]
        for col in pc_cols:
            df[f'{col}_dm'] = df[col] - df.groupby(active_groups)[col].transform('mean')
            dm_cols.append(f'{col}_dm')

    dm_cols = [c for c in dm_cols if df[c].std(skipna=True) > 1e-10]

    if dm_cols:
        valid_mask = df[dm_cols].notna().all(axis=1)
        if valid_mask.sum() > len(dm_cols) + 5:
            df.loc[valid_mask, dm_cols] = StandardScaler().fit_transform(df.loc[valid_mask, dm_cols])

    return df, dm_cols, pca_model, used_features


# ==========================================
# 5. REGRESSIONS (WLS & IV)
# ==========================================
def run_single_analysis(df_single: pd.DataFrame, elec_id, dm_cols: list):
    label = f"Election_{elec_id}"
    header = f"\n{'='*70}\n  SINGLE ELECTION ANALYSIS: {label}\n{'='*70}"
    log_to_file(header)

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share', 'sum_people',
                'area2_name', 'province_tag', 'voters_per_station_dm'] + dm_cols
    df_sub = df_single.dropna(subset=req_cols).copy()

    df_sub.to_csv(f'output_data/regression_data_{label}{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    if df_sub.empty:
        log_to_file(f" [!] Not enough data to run analysis for {label}.")
        return

    exog_vars = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag)"

    # 1. Turnout Regressions
    for target in ['turnout', 'early_turnout']:
        log_to_file(f"\n--- Sub-Target: {target} ---")
        exog_str = exog_vars + " + voters_per_station_dm"

        formula = f"{target} ~ {exog_str}"
        model = smf.wls(formula, data=df_sub, weights=df_sub['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_sub['area2_name']})
        log_to_file(f"[Formula]: {formula}")
        log_to_file(model.summary().as_text())

    # 2. Total Dem Vote Share & Early Dem Share Regressions
    for main_target in ['vote_share', 'early_dem_share']:
        turnout_vars = ['early_turnout', 'turnout'] if INCLUDE_TURNOUT_IN_VOTESHARE else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (Instrumenting {turnout_var})\n{'='*55}")
                formula_vs = f"{main_target} ~ {exog_vars} + [{turnout_var} ~ voters_per_station_dm]"
                model_vs = IV2SLS.from_formula(formula_vs, df_sub, weights=df_sub['sum_people'])
                res_vs = model_vs.fit(cov_type='robust')

                log_to_file(f"\n[Formula]: {formula_vs}")
                if hasattr(res_vs, 'first_stage') and res_vs.first_stage is not None:
                    log_to_file("\n[First Stage F-Stats]")
                    for endog in res_vs.first_stage.diagnostics.index:
                        f_stat = res_vs.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file("\n[Second Stage Summary]")
                log_to_file(res_vs.summary.as_text())
            else:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (No Turnout Covariate)\n{'='*55}")
                formula_vs = f"{main_target} ~ {exog_vars}"
                model_vs = smf.wls(formula_vs, data=df_sub, weights=df_sub['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_sub['area2_name']})
                log_to_file(f"\n[Formula]: {formula_vs}")
                log_to_file(model_vs.summary().as_text())


def run_iv_causal_analysis(df_pres21: pd.DataFrame, df_pres19: pd.DataFrame, dm_cols: list):
    header = f"\n{'='*70}\n  EXACTLY IDENTIFIED IV (2SLS) ANALYSIS (21st Pres)\n{'='*70}"
    log_to_file(header)

    instrument_df = df_pres19[['province_tag', 'primary_sgg', 'dong_norm', 'third_vote_share']].drop_duplicates()
    instrument_df = instrument_df.rename(columns={'third_vote_share': 'yoo_vote_share_19'})

    df_iv = df_pres21.rename(columns={'third_vote_share': 'LJS_vote_share'})
    df_iv = df_iv.merge(instrument_df, on=['province_tag', 'primary_sgg', 'dong_norm'], how='inner')

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share',
                'LJS_vote_share', 'yoo_vote_share_19', 'sum_people', 'voters_per_station_dm', 'area2_name'] + dm_cols
    df_iv = df_iv.dropna(subset=req_cols).copy()

    df_iv.to_csv(f'output_data/regression_data_iv_pres21{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    exog_vars = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag)"

    # 1. Turnout Regressions
    for target in ['turnout', 'early_turnout']:
        log_to_file(f"\n\n{'='*55}\n Target: {target}\n{'='*55}")

        exog_str = exog_vars + " + voters_per_station_dm"

        if INCLUDE_LJS_IN_VOTESHARE:
            formula = f"{target} ~ {exog_str} + [LJS_vote_share ~ yoo_vote_share_19]"
            res = IV2SLS.from_formula(formula, df_iv, weights=df_iv['sum_people']).fit(cov_type='robust')
            log_to_file(f"\n[Formula]: {formula}")

            if hasattr(res, 'first_stage') and res.first_stage is not None:
                log_to_file("\n[First Stage F-Stats]")
                for endog in res.first_stage.diagnostics.index:
                    f_stat = res.first_stage.diagnostics.loc[endog, 'f.stat']
                    log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
            log_to_file("\n[Second Stage Summary]")
            log_to_file(res.summary.as_text())
        else:
            formula = f"{target} ~ {exog_str}"
            res = smf.wls(formula, data=df_iv, weights=df_iv['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_iv['area2_name']})
            log_to_file(f"\n[Formula]: {formula}")
            log_to_file(res.summary().as_text())

    # 2. Vote Share & Early Dem Share Regressions (Dynamic string building based on toggles)
    for main_target in ['vote_share', 'early_dem_share']:
        turnout_vars = ['early_turnout', 'turnout'] if INCLUDE_TURNOUT_IN_VOTESHARE else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (Turnout Covariate: {turnout_var})\n{'='*55}")
            else:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (No Turnout Covariate)\n{'='*55}")

            endogs, instrs = [], []

            if INCLUDE_LJS_IN_VOTESHARE:
                endogs.append('LJS_vote_share')
                instrs.append('yoo_vote_share_19')
            if INCLUDE_TURNOUT_IN_VOTESHARE and turnout_var:
                endogs.append(turnout_var)
                instrs.append('voters_per_station_dm')

            if endogs:
                formula = f"{main_target} ~ {exog_vars} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res = IV2SLS.from_formula(formula, df_iv, weights=df_iv['sum_people']).fit(cov_type='robust')

                log_to_file(f"\n[Formula]: {formula}")
                if hasattr(res, 'first_stage') and res.first_stage is not None:
                    log_to_file("\n[First Stage F-Stats]")
                    for endog in res.first_stage.diagnostics.index:
                        f_stat = res.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file("\n[Second Stage Summary]")
                log_to_file(res.summary.as_text())
            else:
                formula = f"{main_target} ~ {exog_vars}"
                res = smf.wls(formula, data=df_iv, weights=df_iv['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_iv['area2_name']})
                log_to_file(f"\n[Formula]: {formula}")
                log_to_file(res.summary().as_text())

    return df_iv


def run_pooled_iv_panel(df_pooled: pd.DataFrame, df_pres19: pd.DataFrame, dm_cols: list):
    header = f"\n{'='*70}\n  POOLED IV PANEL REGRESSIONS\n{'='*70}"
    log_to_file(header)

    df_panel = df_pooled.copy()

    inst_df = df_pres19[['province_tag', 'primary_sgg', 'dong_norm', 'third_vote_share']].drop_duplicates()
    inst_df = inst_df.rename(columns={'third_vote_share': 'base_yoo_pr'})

    df_panel = df_panel.merge(inst_df, on=['province_tag', 'primary_sgg', 'dong_norm'], how='inner')

    df_panel['LJS_vote_share'] = np.where(df_panel['election_id'] == 'pres21', df_panel['third_vote_share'], 0)
    df_panel['active_yoo_pr'] = np.where(df_panel['election_id'] == 'pres21', df_panel['base_yoo_pr'], 0)
    df_panel['prov_elec'] = df_panel['province_tag'] + "_" + df_panel['election_id'].astype(str)

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share',
                'LJS_vote_share', 'active_yoo_pr', 'sum_people', 'election_id',
                'province_tag', 'prov_elec', 'voters_per_station_dm', 'area2_name'] + dm_cols
    df_panel = df_panel.dropna(subset=req_cols).copy()

    df_panel.to_csv(f'output_data/regression_data_pooled_panel{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    exog_vars_add = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag) + C(election_id)"
    exog_vars_int = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(prov_elec)"

    for target in ['turnout', 'early_turnout', 'early_dem_share', 'vote_share']:
        log_to_file(f"\n\n{'='*55}\n PANEL TARGET: {target}\n{'='*55}")

        exog_add_base = exog_vars_add
        exog_int_base = exog_vars_int

        if target in ['turnout', 'early_turnout']:
            exog_add_base += " + voters_per_station_dm"
            exog_int_base += " + voters_per_station_dm"

        turnout_vars = ['early_turnout', 'turnout'] if (target in ['vote_share', 'early_dem_share'] and INCLUDE_TURNOUT_IN_VOTESHARE) else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n--- Sub-Target for {target} (Turnout Covariate: {turnout_var}) ---")

            endogs, instrs = [], []

            if INCLUDE_LJS_IN_VOTESHARE:
                endogs.append('LJS_vote_share')
                instrs.append('active_yoo_pr')

            if target in ['vote_share', 'early_dem_share'] and INCLUDE_TURNOUT_IN_VOTESHARE and turnout_var:
                endogs.append(turnout_var)
                instrs.append('voters_per_station_dm')

            # Execute Spec 1 (Additive)
            log_to_file(f"\n[Spec 1: Additive Province & Election FEs]")
            if endogs:
                formula_add = f"{target} ~ {exog_add_base} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res_add = IV2SLS.from_formula(formula_add, df_panel, weights=df_panel['sum_people']).fit(cov_type='robust')
                log_to_file(f"[Formula]: {formula_add}")
                if hasattr(res_add, 'first_stage') and res_add.first_stage is not None:
                    log_to_file("[First Stage F-Stats]")
                    for endog in res_add.first_stage.diagnostics.index:
                        f_stat = res_add.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file(res_add.summary.as_text())
            else:
                formula_add = f"{target} ~ {exog_add_base}"
                res_add = smf.wls(formula_add, data=df_panel, weights=df_panel['sum_people']).fit(cov_type='HC3')
                log_to_file(f"[Formula]: {formula_add}")
                log_to_file(res_add.summary().as_text())

            # Execute Spec 2 (Interacted)
            log_to_file(f"\n[Spec 2: Interacted Province x Election FEs]")
            if endogs:
                formula_int = f"{target} ~ {exog_int_base} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res_int = IV2SLS.from_formula(formula_int, df_panel, weights=df_panel['sum_people']).fit(cov_type='robust')
                log_to_file(f"[Formula]: {formula_int}")
                if hasattr(res_int, 'first_stage') and res_int.first_stage is not None:
                    log_to_file("[First Stage F-Stats]")
                    for endog in res_int.first_stage.diagnostics.index:
                        f_stat = res_int.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file(res_int.summary.as_text())
            else:
                formula_int = f"{target} ~ {exog_int_base}"
                res_int = smf.wls(formula_int, data=df_panel, weights=df_panel['sum_people']).fit(cov_type='HC3')
                log_to_file(f"[Formula]: {formula_int}")
                log_to_file(res_int.summary().as_text())


# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    with open(LOG_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write("COMPREHENSIVE REGRESSION & DATA LOG\n")
        f.write("="*60 + "\n")

    all_dm = []

    for elec_id in ['pres19', 21, 22, 'pres20', 'pres21']:
        cfg = ELECTION_CONFIGS[elec_id]
        log_to_file(f"\n--- Loading Data for {cfg['label']} ---")

        df_census = load_census_csv(cfg.get('census_csv'))
        df_apt = load_apt_csv(cfg.get('apt_csv_glob'), elec_id)
        df_emp = load_employment_sgg(election_key=elec_id)
        df_stations = load_polling_stations(elec_id)

        log_to_file(f"  > Census Data: {'Loaded successfully' if not df_census.empty else 'NO DATA FOUND'}")

        apt_cache_path = f"output_data/processed_apt_data_{elec_id}.csv"
        if not df_apt.empty:
            if os.path.exists(apt_cache_path) and os.path.getmtime(apt_cache_path) > 0:
                log_to_file(f"  > APT Price Data: Loaded from Cache ({apt_cache_path})")
            else:
                log_to_file(f"  > APT Price Data: Parsed from Raw and Saved to Cache")
        else:
            log_to_file(f"  > APT Price Data: NO DATA FOUND")

        log_to_file(f"  > Employment Data: {'Loaded successfully' if not df_emp.empty else 'NO DATA FOUND'}")
        log_to_file(f"  > Polling Stations: {'Loaded successfully' if not df_stations.empty else 'NO DATA FOUND'}")

        df_dong, df_const, _ = load_election_csv(
            cfg['result_csv'],
            dem_pattern=cfg['dem_pattern'],
            con_pattern=cfg['con_pattern'],
            third_pattern=cfg.get('third_pattern'),
            election_type=cfg['election_type'],
        )

        if EXCLUDE_HONAM_YEONGNAM:
            exclude_provs = PARTISAN_REGION_PROVINCES['Honam'] | PARTISAN_REGION_PROVINCES['Yeongnam']
            short_exclude = {PROV_FULL_TO_SHORT.get(p, p) for p in exclude_provs} | exclude_provs
            df_dong  = df_dong[~df_dong['province_tag'].isin(short_exclude)]
            df_const = df_const[~df_const['province_tag'].isin(short_exclude)]

        if not df_dong.empty and not df_const.empty:
            dm = prep_election_data(df_dong, df_const, df_census, df_apt, df_emp, df_stations, elec_id)
            all_dm.append(dm)
            log_to_file(f"  > Processed {len(dm)} distinct dong-level observations.")

    df_raw_pooled = pd.concat(all_dm, ignore_index=True)
    raw_age_cols = [c for c in AGE_GENDER_COLS if c in df_raw_pooled.columns]
    raw_emp_cols = [c for c in EMPLOYMENT_FEATURE_COLS if c in df_raw_pooled.columns]

    # ---------------------------------------------------------
    # PART A: ISOLATED SINGLE-ELECTION PIPELINE
    # ---------------------------------------------------------
    log_to_file("\n\n" + "#"*70 + "\n PART A: ISOLATED SINGLE-ELECTION PIPELINE\n" + "#"*70)

    for elec in [21, 22, 'pres20']:
        df_single_raw = df_raw_pooled[df_raw_pooled['election_id'] == str(elec)].copy()
        if not df_single_raw.empty:
            df_single_pca, dm_cols_single, _, _ = prepare_continuous_covariates(df_single_raw, raw_age_cols, raw_emp_cols, label=f"isolated_{elec}")
            run_single_analysis(df_single_pca, elec, dm_cols_single)

    df_pres21_raw = df_raw_pooled[df_raw_pooled['election_id'] == 'pres21'].copy()
    df_pres19_raw = df_raw_pooled[df_raw_pooled['election_id'] == 'pres19'].copy()

    if not df_pres21_raw.empty and not df_pres19_raw.empty:
        df_pres21_pca, dm_cols_iv, _, _ = prepare_continuous_covariates(df_pres21_raw, raw_age_cols, raw_emp_cols, label="isolated_pres21")
        run_iv_causal_analysis(df_pres21_pca, df_pres19_raw, dm_cols_iv)

    # ---------------------------------------------------------
    # PART B: POOLED ELECTION ANALYSIS
    # ---------------------------------------------------------
    log_to_file("\n\n" + "#"*70 + "\n PART B: POOLED PANEL ELECTION PIPELINE\n" + "#"*70)

    df_pooled_pca, dm_cols_pooled, _, _ = prepare_continuous_covariates(df_raw_pooled, raw_age_cols, raw_emp_cols, label="pooled_panel")

    if not df_pres19_raw.empty:
        run_pooled_iv_panel(df_pooled_pca, df_pres19_raw, dm_cols_pooled)

    print(f"\n[+] All tasks completed. Full statistical logs exported to -> {LOG_FILE_PATH}")


--- Loading Data for 19th Presidential Election (2017) ---


/tmp/ipykernel_4025/1350488833.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: NO DATA FOUND
  > Employment Data: Loaded successfully
  > Polling Stations: NO DATA FOUND
  > Processed 3537 distinct dong-level observations.

--- Loading Data for 21st General Election (2020) ---


/tmp/ipykernel_4025/1350488833.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_21.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 5519 distinct dong-level observations.

--- Loading Data for 22nd General Election (2024) ---


...

  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_22.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 4513 distinct dong-level observations.

--- Loading Data for 20th Presidential Election (2022) ---


/tmp/ipykernel_4025/1350488833.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_pres20.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 5976 distinct dong-level observations.

--- Loading Data for 21st Presidential Election (2025) ---


/tmp/ipykernel_4025/1350488833.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_pres21.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 6050 distinct dong-level observations.


######################################################################
 PART A: ISOLATED SINGLE-ELECTION PIPELINE
######################################################################
  -> Saved PCA Loadings for isolated_21 to output_data/pca_loadings_isolated_21.csv

  SINGLE ELECTION ANALYSIS: Election_21

--- Sub-Target: turnout ---
[Formula]: turnout ~ pc1_dm + pc2_dm + pc3_dm + pc4_dm + pc5_dm + pc6_dm + pc7_dm + pc8_dm + pc9_dm + pc10_dm + pc11_dm + C(province_tag) + voters_per_station_dm
                            WLS Regression Results                            
Dep. Variable:                turnout   R-squared:                       0.291
Model:                            WLS   Adj. R-squared:                  0.288
Metho

LJS vote share covariate turned on in democratic vote share regressions

NO turnout covariate in democratic vote share regressions

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import glob
from difflib import get_close_matches
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.formula.api as smf

try:
    from linearmodels.iv import IV2SLS
except ImportError:
    print("[!] 'linearmodels' not found. Please run: pip install linearmodels")
    exit(1)

# ==========================================
# CONFIGURATION & GLOBAL SETUP
# ==========================================

# --- GLOBAL TOGGLES ---
EXCLUDE_HONAM_YEONGNAM       = False
INCLUDE_TURNOUT_IN_VOTESHARE = False   # Toggle Turnout IV in Vote Share Regressions
INCLUDE_LJS_IN_VOTESHARE     = True   # Toggle LJS Vote Share IV in relevant Regressions

# --- DYNAMIC FILENAME SUFFIX ---
if INCLUDE_LJS_IN_VOTESHARE and INCLUDE_TURNOUT_IN_VOTESHARE:
    IV_SUFFIX = "_LJS_turnout_IV_cov"
elif INCLUDE_LJS_IN_VOTESHARE:
    IV_SUFFIX = "_LJS_IV_cov"
elif INCLUDE_TURNOUT_IN_VOTESHARE:
    IV_SUFFIX = "_turnout_IV_cov"
else:
    IV_SUFFIX = "_no_IV_cov"

os.makedirs('output_data', exist_ok=True)
LOG_FILE_PATH = f'output_data/comprehensive_regression_logs{IV_SUFFIX}.txt'

PCA_VARIANCE_THRESHOLD = 0.90
DEMEAN_GROUP_COLS      = ['province_tag']
MIN_COVERAGE           = 0.50

ELECTION_CONFIGS = {
    'pres19': {
        'election_type':  'presidential',
        'census_csv':     '19th_presidential_election_census.csv',
        'result_csv':     '19th_presidential_election_result.csv',
        'apt_csv_glob':   None,
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'자유한국당',
        'third_pattern':  r'유승민|바른정당',
        'label':          '19th Presidential Election (2017)',
        'year':           2017,
        'election_month': 5,
    },
    21: {
        'election_type':  'general',
        'census_csv':     '21st_election_census.csv',
        'result_csv':     '21st_election_result.csv',
        'apt_csv_glob':   '*21st_election_*_apt_price.csv',
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'미래통합당|자유한국당',
        'third_pattern':  None,
        'label':          '21st General Election (2020)',
        'year':           2020,
        'election_month': 4,
    },
    22: {
        'election_type':  'general',
        'census_csv':     '22nd_election_census.csv',
        'result_csv':     '22nd_election_result.csv',
        'apt_csv_glob':   '*22nd_election_*_apt_price.csv',
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'국민의힘',
        'third_pattern':  r'개혁신당',
        'label':          '22nd General Election (2024)',
        'year':           2024,
        'election_month': 4,
    },
    'pres20': {
        'election_type':         'presidential',
        'census_csv':            '20th_presidential_election_census.csv',
        'result_csv':            '20th_presidential_election_result.csv',
        'apt_csv_glob':          '*20th_presidential_election_*_apt_prices.csv',
        'dem_pattern':           r'더불어민주당',
        'con_pattern':           r'국민의힘',
        'third_pattern':         None,
        'label':                 '20th Presidential Election (2022)',
        'year':                  2022,
        'election_month':        3,
    },
    'pres21': {
        'election_type':         'presidential',
        'census_csv':            '21st_presidential_election_census.csv',
        'result_csv':            '21st_presidential_election_result.csv',
        'apt_csv_glob':          '*21st_presidential_election_*_apt_prices.csv',
        'dem_pattern':           r'더불어민주당',
        'con_pattern':           r'국민의힘',
        'third_pattern':         r'이준석|개혁신당',
        'label':                 '21st Presidential Election (2025)',
        'year':                  2025,
        'election_month':        6,
    },
}

SPECIAL_DONG_NAMES_GENERAL = {
    '거소·선상투표', '관외사전투표', '국외부재자투표',
    '국외부재자투표(공관)', '잘못 투입·구분된 투표지',
}
SPECIAL_DONG_NAMES_PRESIDENTIAL = {
    '거소·선상투표', '관외사전투표', '재외투표',
    '잘못 투입·구분된 투표지',
}
GWANNAESA_LABEL = '관내사전투표'
META_CANDIDATES = {'선거인수', '투표수', '무효 투표수', '기권자수'}

PROV_FULL_TO_SHORT = {
    '서울특별시': 'Seoul',  '부산광역시': 'Busan',   '대구광역시': 'Daegu',
    '인천광역시': 'Incheon','광주광역시': 'Gwangju', '대전광역시': 'Daejeon',
    '울산광역시': 'Ulsan',  '세종특별자치시': 'Sejong',
    '경기도': 'Gyeonggi',  '강원도': 'Gangwon',     '강원특별자치도': 'Gangwon',
    '충청북도': 'Chungbuk', '충청남도': 'Chungnam',
    '전라북도': 'Jeonbuk',  '전북특별자치도': 'Jeonbuk', '전라남도': 'Jeonnam',
    '경상북도': 'Gyeongbuk','경상남도': 'Gyeongnam', '제주특별자치도': 'Jeju',
    '서울': 'Seoul',   '부산': 'Busan',    '대구': 'Daegu',
    '인천': 'Incheon', '광주': 'Gwangju',  '대전': 'Daejeon',
    '울산': 'Ulsan',
}

SGG_TO_PROVINCE_EMP = {
    '가평군':'Gyeonggi','고양시':'Gyeonggi','과천시':'Gyeonggi','광명시':'Gyeonggi',
    '광주시':'Gyeonggi','구리시':'Gyeonggi','군포시':'Gyeonggi','김포시':'Gyeonggi',
    '남양주시':'Gyeonggi','동두천시':'Gyeonggi','부천시':'Gyeonggi','성남시':'Gyeonggi',
    '수원시':'Gyeonggi','시흥시':'Gyeonggi','안산시':'Gyeonggi','안성시':'Gyeonggi',
    '안양시':'Gyeonggi','양주시':'Gyeonggi','양평군':'Gyeonggi','여주군':'Gyeonggi',
    '여주시':'Gyeonggi','연천군':'Gyeonggi','오산시':'Gyeonggi','용인시':'Gyeonggi',
    '의왕시':'Gyeonggi','의정부시':'Gyeonggi','이천시':'Gyeonggi','파주시':'Gyeonggi',
    '평택시':'Gyeonggi','포천시':'Gyeonggi','하남시':'Gyeonggi','화성시':'Gyeonggi',
    '강릉시':'Gangwon','고성군':'Gangwon','동해시':'Gangwon','삼척시':'Gangwon',
    '속초시':'Gangwon','양구군':'Gangwon','양양군':'Gangwon','영월군':'Gangwon',
    '원주시':'Gangwon','인제군':'Gangwon','정선군':'Gangwon','철원군':'Gangwon',
    '춘천시':'Gangwon','태백시':'Gangwon','평창군':'Gangwon','홍천군':'Gangwon',
    '화천군':'Gangwon','횡성군':'Gangwon','괴산군':'Chungbuk','단양군':'Chungbuk',
    '보은군':'Chungbuk','영동군':'Chungbuk','옥천군':'Chungbuk','음성군':'Chungbuk',
    '제천시':'Chungbuk','증평군':'Chungbuk','진천군':'Chungbuk','청원군':'Chungbuk',
    '청주시':'Chungbuk','충주시':'Chungbuk','계룡시':'Chungnam','공주시':'Chungnam',
    '금산군':'Chungnam','논산시':'Chungnam','당진시':'Chungnam','보령시':'Chungnam',
    '부여군':'Chungnam','서산시':'Chungnam','서천군':'Chungnam','아산시':'Chungnam',
    '연기군':'Chungnam','예산군':'Chungnam','청양군':'Chungnam','천안시':'Chungnam',
    '태안군':'Chungnam','홍성군':'Chungnam','고창군':'Jeonbuk','군산시':'Jeonbuk',
    '김제시':'Jeonbuk','남원시':'Jeonbuk','무주군':'Jeonbuk','부안군':'Jeonbuk',
    '순창군':'Jeonbuk','완주군':'Jeonbuk','익산시':'Jeonbuk','임실군':'Jeonbuk',
    '장수군':'Jeonbuk','전주시':'Jeonbuk','정읍시':'Jeonbuk','진안군':'Jeonbuk',
    '강진군':'Jeonnam','고흥군':'Jeonnam','곡성군':'Jeonnam','구례군':'Jeonnam',
    '나주시':'Jeonnam','담양군':'Jeonnam','목포시':'Jeonnam','무안군':'Jeonnam',
    '보성군':'Jeonnam','순천시':'Jeonnam','신안군':'Jeonnam','여수시':'Jeonnam',
    '영광군':'Jeonnam','영암군':'Jeonnam','완도군':'Jeonnam','장성군':'Jeonnam',
    '장흥군':'Jeonnam','진도군':'Jeonnam','함평군':'Jeonnam','화순군':'Jeonnam',
    '광양시':'Jeonnam','해남군':'Jeonnam','경산시':'Gyeongbuk','경주시':'Gyeongbuk',
    '고령군':'Gyeongbuk','구미시':'Gyeongbuk','김천시':'Gyeongbuk','문경시':'Gyeongbuk',
    '봉화군':'Gyeongbuk','상주시':'Gyeongbuk','성주군':'Gyeongbuk','안동시':'Gyeongbuk',
    '영덕군':'Gyeongbuk','영양군':'Gyeongbuk','영주시':'Gyeongbuk','영천시':'Gyeongbuk',
    '예천군':'Gyeongbuk','울릉군':'Gyeongbuk','울진군':'Gyeongbuk','의성군':'Gyeongbuk',
    '청도군':'Gyeongbuk','청송군':'Gyeongbuk','칠곡군':'Gyeongbuk','포항시':'Gyeongbuk',
    '거제시':'Gyeongnam','거창군':'Gyeongnam','김해시':'Gyeongnam','남해군':'Gyeongnam',
    '밀양시':'Gyeongnam','사천시':'Gyeongnam','산청군':'Gyeongnam','양산시':'Gyeongnam',
    '의령군':'Gyeongnam','진주시':'Gyeongnam','창녕군':'Gyeongnam','창원시':'Gyeongnam',
    '통영시':'Gyeongnam','하동군':'Gyeongnam','함안군':'Gyeongnam','함양군':'Gyeongnam',
    '합천군':'Gyeongnam','서귀포시':'Jeju','제주시':'Jeju',
}

PARTISAN_REGION_PROVINCES = {
    'Honam':   {'Jeonnam', 'Jeonbuk', 'Gwangju', '전남', '전북', '광주'},
    'Yeongnam':{'Gyeongbuk', 'Gyeongnam', 'Daegu', 'Busan', 'Ulsan', '경북', '경남', '대구', '부산', '울산'},
}

AGE_GENDER_COLS = [
    'pct_m_1824', 'pct_m_2529', 'pct_m_3034', 'pct_m_3539', 'pct_m_4044', 'pct_m_4549',
    'pct_m_5054', 'pct_m_5559', 'pct_m_6064', 'pct_m_6569', 'pct_m_70plus',
    'pct_f_1824', 'pct_f_2529', 'pct_f_3034', 'pct_f_3539', 'pct_f_4044', 'pct_f_4549',
    'pct_f_5054', 'pct_f_5559', 'pct_f_6064', 'pct_f_6569', 'pct_f_70plus',
]

EMPLOYMENT_FEATURE_COLS = [
    'emp_men_total', 'emp_men_1529', 'emp_men_3049', 'emp_men_5064', 'emp_men_65plus',
    'emp_wmn_total', 'emp_wmn_1529', 'emp_wmn_3049', 'emp_wmn_5064', 'emp_wmn_65plus',
    'occ_wage_share', 'occ_regular_share', 'occ_nonwage_share',
    'job_professional', 'job_clerical', 'job_service_sales',
    'job_skilled_machine', 'job_simple_labor', 'job_farming',
    'ind_agriculture', 'ind_manufacturing', 'ind_construction',
    'ind_retail_food', 'ind_transport_finance', 'ind_services',
    'nonp_male_share', 'nonp_young_share', 'nonp_middle_share', 'nonp_old_share',
]

# ==========================================
# FILE WRITER HELPER
# ==========================================
def log_to_file(text: str, mode='a'):
    print(text)
    with open(LOG_FILE_PATH, mode, encoding='utf-8') as f:
        f.write(text + "\n")

# ==========================================
# NORMALISATION & IO UTILITIES
# ==========================================
def _read_csv_auto(path: str, **kwargs) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding='utf-8', **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='cp949', **kwargs)

def _to_numeric_safe(series: pd.Series) -> pd.Series:
    return (series.astype(str).str.strip()
            .replace(['-', '.', '', 'nan'], np.nan)
            .pipe(pd.to_numeric, errors='coerce'))

def normalize_dong_name(name: str) -> str:
    if not isinstance(name, str): return ""
    name = re.sub(r'\(.*?\)', '', name).strip().replace('.', '·')
    name = re.sub(r'제(\d)', r'\1', name)
    name = re.sub(r'·\d+', '', name)
    name = re.sub(r'(\d+)(동|읍|면)$', r'\2', name)
    return re.sub(r'\s+', ' ', name)

def split_admin_tokens(name: str) -> list:
    tokens, buf = [], []
    for ch in name:
        buf.append(ch)
        if ch in '시군구' and len(buf) >= 2:
            tokens.append(''.join(buf))
            buf = []
    if buf: tokens.append(''.join(buf))
    return [t for t in tokens if t]

def normalize_sigungu(name: str) -> list:
    if not isinstance(name, str): return []
    name = re.sub(r'\(.*?\)', '', name).strip()
    if not name: return []
    tokens = split_admin_tokens(name)
    if not tokens:
        stripped = re.sub(r'[시군구갑을병정무]$', '', name).strip()
        return [stripped] if stripped else []
    si_gun_count = sum(1 for t in tokens if t[-1] in '시군' and len(t) >= 2)
    gu_count     = sum(1 for t in tokens if t[-1] == '구'  and len(t) >= 2)
    ordered = tokens if (si_gun_count >= 2 or (si_gun_count == 0 and gu_count >= 2)) else list(reversed(tokens))
    candidates = []
    for t in ordered:
        key = re.sub(r'[시군구]$', '', t).strip()
        if key and key not in candidates: candidates.append(key)
    return candidates

def get_election_periods(month: int, year: int) -> dict:
    if month <= 6:
        prev = year - 1
        return {'half_periods': [f'{prev}.1/2', f'{prev}.2/2'], 'grdp_year': prev}
    else:
        prev = year - 1
        return {'half_periods': [f'{prev}.2/2', f'{year}.1/2'], 'grdp_year': prev}

def parse_employment_sgg(region_name: str) -> tuple:
    if not isinstance(region_name, str): return ('', '')
    parts = region_name.strip().split(None, 1)
    if not parts: return ('', '')
    prov = parts[0]
    if len(parts) == 1:
        prov_eng = SGG_TO_PROVINCE_EMP.get(prov, '')
        primary  = re.sub(r'[시군구]$', '', prov).strip()
        return (prov_eng, primary)
    sgg_raw  = parts[1]
    sgg_norm = normalize_sigungu(sgg_raw)
    primary  = sgg_norm[0] if sgg_norm else re.sub(r'[시군구]$', '', sgg_raw).strip()
    prov_eng = PROV_FULL_TO_SHORT.get(prov, prov)
    return (prov_eng, primary)

# ==========================================
# 1. DEMOGRAPHIC, ASSET & STATION DATA LOADERS
# ==========================================
def load_polling_stations(elec_id) -> pd.DataFrame:
    # Look strictly for normal-day station files to proxy physical density friction
    paths_to_try = [
        f"stations_normal_{elec_id}.csv",
        f"polling_stations_{elec_id}.csv",
        f"stations_normal_20240410.csv" if str(elec_id) == '22' else f"stations_normal_{elec_id}.csv"
    ]

    df = pd.DataFrame()
    for path in paths_to_try:
        if os.path.exists(path):
            df = _read_csv_auto(path)
            break

    if df.empty: return df

    if 'province_tag' not in df.columns and 'sdName' in df.columns:
        df['province_tag'] = df['sdName'].map(PROV_FULL_TO_SHORT).fillna(df['sdName'])
        def extract_sgg(x):
            cands = normalize_sigungu(x)
            return cands[0] if cands else re.sub(r'[시군구]$', '', str(x)).strip()
        df['primary_sgg'] = df['wiwName'].apply(extract_sgg)
        df['dong_norm']   = df['emdName'].apply(normalize_dong_name)

        mask_seoul = df['province_tag'] == 'Seoul'
        mask_g3 = df['primary_sgg'].isin(['강남', '서초', '송파'])
        df.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
        df.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    if 'normal_station_count' not in df.columns:
        if 'station_count' in df.columns:
            df = df.rename(columns={'station_count': 'normal_station_count'})
        else:
            df['normal_station_count'] = 1

    return df[['province_tag', 'primary_sgg', 'dong_norm', 'normal_station_count']]


def _detect_year_prefix(df: pd.DataFrame) -> str:
    for col in df.columns:
        m = re.match(r'(\d{4}년\d{2}월)_계_총인구수', col)
        if m: return m.group(1)
    raise ValueError("Cannot detect census year prefix.")

def load_census_csv(csv_path: str) -> pd.DataFrame:
    if not csv_path or not os.path.exists(csv_path): return pd.DataFrame()
    try:
        df = _read_csv_auto(csv_path, low_memory=False)
        prefix = _detect_year_prefix(df)
        voting_age_cols = ([f"{prefix}_계_{a}세" for a in range(18, 100)] + [f"{prefix}_계_100세 이상"])
        cols_4059 = [f"{prefix}_계_{a}세" for a in range(40, 60)]
        all_target_cols = list(voting_age_cols)
        for g in ['남', '여']:
            for a in range(18, 100):
                all_target_cols.append(f"{prefix}_{g}_{a}세")
            all_target_cols.append(f"{prefix}_{g}_100세 이상")

        for col in set(all_target_cols):
            if col in df.columns:
                df[col] = (df[col].astype(str).str.replace(',', '', regex=False).pipe(pd.to_numeric, errors='coerce').fillna(0))

        df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)
        df = df[df['total_voting_pop'] > 0].copy()

        ranges = [(18,25,'1824'),(25,30,'2529'),(30,35,'3034'),(35,40,'3539'),
                  (40,45,'4044'),(45,50,'4549'),(50,55,'5054'),(55,60,'5559'),
                  (60,65,'6064'),(65,70,'6569')]

        for g, g_str in [('남','m'), ('여','f')]:
            for r_start, r_end, r_str in ranges:
                cols = [f"{prefix}_{g}_{a}세" for a in range(r_start, r_end)]
                df[f'pct_{g_str}_{r_str}'] = (df[[c for c in cols if c in df.columns]].sum(axis=1) / df['total_voting_pop'])
            cols_70 = ([f"{prefix}_{g}_{a}세" for a in range(70, 100)] + [f"{prefix}_{g}_100세 이상"])
            df[f'pct_{g_str}_70plus'] = (df[[c for c in cols_70 if c in df.columns]].sum(axis=1) / df['total_voting_pop'])

        df['demographic_propensity'] = (df[[c for c in cols_4059 if c in df.columns]].sum(axis=1) / df['total_voting_pop'])

        def extract_census_keys(admin_name):
            if not isinstance(admin_name, str): return [], ""
            clean = re.sub(r'\(.*?\)', '', admin_name).strip()
            parts = clean.split()
            dong_norm = normalize_dong_name(parts[-1]) if parts else ""
            sgg_cands = []
            for token in reversed(parts[:-1]):
                for c in normalize_sigungu(token):
                    if c not in sgg_cands: sgg_cands.append(c)
            return sgg_cands, dong_norm

        rows = []
        for _, row in df.iterrows():
            sgg_cands, dong_norm = extract_census_keys(row['행정구역'])
            row_dict = {
                'sgg_candidates': sgg_cands,
                'primary_sgg':    sgg_cands[0] if sgg_cands else "",
                'dong_norm':      dong_norm,
                'demographic_propensity': row['demographic_propensity'],
            }
            for col in AGE_GENDER_COLS:
                row_dict[col] = row.get(col, np.nan)
            rows.append(row_dict)

        return pd.DataFrame(rows)
    except Exception as e:
        return pd.DataFrame()

def load_apt_csv(glob_pattern: str, elec_id: str) -> pd.DataFrame:
    if not glob_pattern: return pd.DataFrame()

    processed_path = f"output_data/processed_apt_data_{elec_id}.csv"

    if os.path.exists(processed_path):
        try:
            return _read_csv_auto(processed_path)
        except Exception:
            pass

    file_list = glob.glob(glob_pattern)
    if not file_list: return pd.DataFrame()

    df_list = []
    for file in file_list:
        try:
            df_list.append(_read_csv_auto(file, skiprows=15))
        except Exception: pass
    if not df_list: return pd.DataFrame()

    df = pd.concat(df_list, ignore_index=True)
    try:
        df['거래금액(만원)'] = pd.to_numeric(df['거래금액(만원)'].astype(str).str.replace(',', '').str.strip(), errors='coerce')
        df['전용면적(㎡)']  = pd.to_numeric(df['전용면적(㎡)'], errors='coerce')
        df['price_per_sqm']  = df['거래금액(만원)'] / df['전용면적(㎡)']

        def parse_loc(x):
            parts = str(x).split()
            prov  = PROV_FULL_TO_SHORT.get(parts[0], parts[0]) if parts else ""
            sgg   = normalize_sigungu(parts[1])[0] if len(parts) > 2 and normalize_sigungu(parts[1]) else ""
            dong  = normalize_dong_name(parts[-1]) if parts else ""
            return pd.Series([prov, sgg, dong])

        df[['prov','sgg','dong_norm']] = df['시군구'].apply(parse_loc)

        mask_seoul = df['prov'] == 'Seoul'
        mask_g3 = df['sgg'].isin(['강남', '서초', '송파'])
        df.loc[mask_seoul & mask_g3, 'prov'] = 'Seoul (Gangnam3gu)'
        df.loc[mask_seoul & ~mask_g3, 'prov'] = 'Seoul (Non-Gangnam3gu)'

        apt_agg = (df.groupby(['prov','sgg','dong_norm'])['price_per_sqm'].median().reset_index()
                   .rename(columns={'price_per_sqm': 'median_apt_price_sqm'}))

        apt_agg.to_csv(processed_path, index=False)
        return apt_agg
    except Exception as e:
        return pd.DataFrame()

def _avg_half_periods(df: pd.DataFrame, base_cols: list) -> pd.Series:
    available = [c for c in base_cols if c in df.columns]
    if not available: return pd.Series(np.nan, index=df.index)
    return df[available].apply(pd.to_numeric, errors='coerce').mean(axis=1)

def load_employment_sgg(election_key, **kwargs) -> pd.DataFrame:
    cfg    = ELECTION_CONFIGS[election_key if type(election_key)==str else int(election_key)]
    month  = cfg['election_month']
    year   = cfg['year']
    periods_info = get_election_periods(month, year)
    periods      = periods_info['half_periods']

    def read_emp(path):
        if not os.path.exists(path): return pd.DataFrame()
        df = _read_csv_auto(path, low_memory=False)
        return df[[c for c in df.columns if not str(c).startswith('Unnamed')]]

    age_map = {'total':'계','1529':'15 - 29세','3049':'30 - 49세',
               '5064':'50 - 64세','65plus':'65세이상'}

    def extract_gender_emp(csv_path, gender_prefix):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df_rate = df[df['항목'].str.strip() == '고용률 (%)'].copy()
        records = {}
        for suf, kor_age in age_map.items():
            sub = df_rate[df_rate['연령별'].str.strip() == kor_age].copy()
            if sub.empty: continue
            val_col = f'{gender_prefix}_{suf}'
            sub[val_col] = _avg_half_periods(sub, periods)
            prov_sgg = sub['행정구역별'].apply(parse_employment_sgg)
            sub['province_tag'] = prov_sgg.apply(lambda x: x[0])
            sub['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
            records[val_col] = sub.groupby(['province_tag','primary_sgg'])[val_col].mean()
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_men = extract_gender_emp(kwargs.get('men_csv', 'men_employment_data.csv'), 'emp_men')
    df_wmn = extract_gender_emp(kwargs.get('women_csv', 'women_employment_data.csv'), 'emp_wmn')

    def extract_occ_type(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df_total = df[df['종사상지위별'].str.strip() == '계'].copy()
        df_total['_total'] = _avg_half_periods(df_total, periods)
        prov_sgg = df_total['행정구역별'].apply(parse_employment_sgg)
        df_total['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df_total['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df_total.set_index(['province_tag','primary_sgg'])['_total']
        records = {}
        for col_name, kor_cat in {'occ_wage_share':'임금근로자', 'occ_regular_share':'- 상용근로자', 'occ_nonwage_share':'비임금근로자'}.items():
            sub = df[df['종사상지위별'].str.strip() == kor_cat].copy()
            if sub.empty: continue
            sub['_val'] = _avg_half_periods(sub, periods)
            prov_sgg2 = sub['행정구역별'].apply(parse_employment_sgg)
            sub['province_tag'] = prov_sgg2.apply(lambda x: x[0])
            sub['primary_sgg']  = prov_sgg2.apply(lambda x: x[1])
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_occ_type = extract_occ_type(kwargs.get('occ_type_csv', 'occupation_type_employment_data.csv'))

    def extract_job_shares(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df[df['항목'].str.strip() == '취업자[천명]'].copy()
        df['_val'] = _avg_half_periods(df, periods)
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df[df['직업별'].str.strip() == '계'].set_index(['province_tag','primary_sgg'])['_val']
        records = {}
        for col_name, kor_cat in {'job_professional': '관리자, 전문가 및 관련종사자', 'job_clerical': '사무 종사자', 'job_service_sales': '서비스·판매 종사자', 'job_skilled_machine':'기능·기계조작·조립 종사자', 'job_simple_labor': '단순노무 종사자', 'job_farming': '농림어업 숙련종사자'}.items():
            sub = df[df['직업별'].str.strip() == kor_cat].copy()
            if sub.empty: continue
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_job = extract_job_shares(kwargs.get('occ_csv', 'occupation_employment_data.csv'))

    def extract_ind_shares(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df[df['항목'].str.strip() == '취업자[천명]'].copy()
        df['_val'] = _avg_half_periods(df, periods)
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df[df['산업별'].str.strip() == '계'].set_index(['province_tag','primary_sgg'])['_val']
        records = {}
        for col_name, kor_cat in {'ind_agriculture': '농업, 임업 및 어업 (A)', 'ind_manufacturing': '광·제조업(B,C)', 'ind_construction': '건설업 (F) ', 'ind_retail_food': '도소매·숙박음식업(G,I)', 'ind_transport_finance': '전기·운수·통신·금융(D,H,J,K)', 'ind_services': '사업·개인·공공서비스 및 기타(E,L~U)'}.items():
            sub = df[df['산업별'].str.strip() == kor_cat.strip()].copy()
            if sub.empty:
                match = get_close_matches(kor_cat.strip(), df['산업별'].str.strip().unique().tolist(), n=1, cutoff=0.85)
                if match: sub = df[df['산업별'].str.strip() == match[0]].copy()
            if sub.empty: continue
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_ind = extract_ind_shares(kwargs.get('ind_csv', 'industry_employment_data.csv'))

    def extract_nonpart(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df.iloc[1:].copy()
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        sub_suffixes = {'total':'', 'male':'.1', 'young':'.3', 'middle':'.4'}
        period_vals  = {k: [] for k in sub_suffixes}
        for period in periods:
            for k, suf in sub_suffixes.items():
                if period + suf in df.columns: period_vals[k].append(_to_numeric_safe(df[period + suf]))
        def mean_series(lst): return pd.concat(lst, axis=1).mean(axis=1) if lst else pd.Series(np.nan, index=df.index)
        total  = mean_series(period_vals['total'])
        result = df[['province_tag','primary_sgg']].copy()
        result['nonp_male_share']   = mean_series(period_vals['male'])   / total.replace(0, np.nan)
        result['nonp_young_share']  = mean_series(period_vals['young'])  / total.replace(0, np.nan)
        result['nonp_middle_share'] = mean_series(period_vals['middle']) / total.replace(0, np.nan)
        return (result.groupby(['province_tag','primary_sgg'])[['nonp_male_share','nonp_young_share','nonp_middle_share']].mean().reset_index())

    df_nonp = extract_nonpart(kwargs.get('nonp_csv', 'non_participant_employment_data.csv'))

    merge_key = ['province_tag','primary_sgg']
    frames = [f for f in [df_men, df_wmn, df_occ_type, df_job, df_ind, df_nonp] if not f.empty]
    if not frames: return pd.DataFrame(columns=merge_key + EMPLOYMENT_FEATURE_COLS)

    df_emp = frames[0]
    for frame in frames[1:]: df_emp = df_emp.merge(frame, on=merge_key, how='outer')
    for col in EMPLOYMENT_FEATURE_COLS:
        if col not in df_emp.columns: df_emp[col] = np.nan

    if not df_emp.empty:
        mask_seoul = df_emp['province_tag'] == 'Seoul'
        mask_g3 = df_emp['primary_sgg'].isin(['강남', '서초', '송파'])
        df_emp.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
        df_emp.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    return df_emp

# ==========================================
# 2. ELECTION CSV LOADER & MATCHER
# ==========================================
def load_election_csv(csv_path: str, dem_pattern: str, con_pattern: str,
                      third_pattern: str = None, election_type: str = 'general'):
    try:
        df = _read_csv_auto(csv_path, low_memory=False)
    except Exception: return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if election_type == 'presidential':
        df = df.rename(columns={'구시군명': '선거구명', '읍면동명': '법정읍면동명'})
        special_dong_names = SPECIAL_DONG_NAMES_PRESIDENTIAL
    else:
        special_dong_names = SPECIAL_DONG_NAMES_GENERAL

    df['득표수']    = pd.to_numeric(df['득표수'], errors='coerce').fillna(0).astype(int)
    df['is_dem']   = df['후보자'].str.contains(dem_pattern,   case=False, na=False)

    if election_type == 'general':
        sejong_gap_kim = (df['선거구명'].astype(str).str.replace(' ', '').str.contains('세종.*갑', na=False)) & \
                         (df['후보자'].astype(str).str.contains('김종민', na=False))
        df.loc[sejong_gap_kim, 'is_dem'] = True

    df['is_con']   = df['후보자'].str.contains(con_pattern,   case=False, na=False)
    df['is_third'] = df['후보자'].str.contains(third_pattern, case=False, na=False) if third_pattern else False
    df['is_meta']  = df['후보자'].isin(META_CANDIDATES)
    df['is_early'] = df['투표구명'] == GWANNAESA_LABEL

    dong_key  = ['시도명','선거구명','법정읍면동명']
    const_key = ['시도명','선거구명']

    def sgg_cands(name):
        if not isinstance(name, str): return []
        if '_' in name: return normalize_sigungu(name.split('_', 1)[1])
        return normalize_sigungu(re.sub(r'[갑을병정무]$', '', name).strip())

    df_geo   = df[~df['법정읍면동명'].isin(special_dong_names)].copy()
    df_votes = df_geo[~df_geo['is_meta']].copy()

    gn_dem = df_votes[df_votes['is_dem'] & df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_dem')
    gn_tot = df_votes[df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_total')
    gn_third = df_votes[df_votes['is_third'] & df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_third')

    sd_dem = df_votes[df_votes['is_dem'] & ~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_dem')
    sd_tot = df_votes[~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_total')
    sd_third = df_votes[df_votes['is_third'] & ~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_third')

    sum_people_dong = (df_geo[~df_geo['is_early'] & (df_geo['후보자'] == '선거인수')].groupby(dong_key)['득표수'].sum().reset_index(name='sum_people'))
    sum_vote_geo    = (df_geo[df_geo['후보자'] == '투표수'].groupby(dong_key)['득표수'].sum().reset_index(name='sum_vote_geo'))

    df_dong = gn_dem.copy()
    for frame in (gn_tot, gn_third, sd_dem, sd_tot, sd_third, sum_people_dong, sum_vote_geo):
        df_dong = df_dong.merge(frame, on=dong_key, how='outer')
    df_dong = df_dong.fillna(0)

    gn_ppl_dong = (df_geo[df_geo['is_early'] & (df_geo['후보자'] == '선거인수')].groupby(dong_key)['득표수'].sum().reset_index(name='_gn_ppl'))
    df_dong = df_dong.merge(gn_ppl_dong, on=dong_key, how='left')
    df_dong['sum_people'] = df_dong['sum_people'] + df_dong['_gn_ppl'].fillna(0)
    df_dong.drop(columns=['_gn_ppl'], inplace=True)

    df_dong['sgg_candidates'] = df_dong['선거구명'].apply(sgg_cands)
    df_dong['primary_sgg']    = df_dong['sgg_candidates'].apply(lambda x: x[0] if x else "")
    df_dong['dong_norm']      = df_dong['법정읍면동명'].apply(normalize_dong_name)
    df_dong['province_tag']   = df_dong['시도명'].map(PROV_FULL_TO_SHORT).fillna(df_dong['시도명'])
    df_dong['area2_name']     = df_dong['선거구명']

    mask_seoul = df_dong['province_tag'] == 'Seoul'
    mask_g3 = df_dong['primary_sgg'].isin(['강남', '서초', '송파'])
    df_dong.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
    df_dong.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    df_gw   = df[df['법정읍면동명'].isin(special_dong_names)]
    df_gw_v = df_gw[~df_gw['is_meta']]

    go_dem_c  = df_gw_v[df_gw_v['is_dem']].groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_dem')
    go_tot_c  = df_gw_v.groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_total')
    go_turn_c = (df_gw[df_gw['후보자'] == '투표수'].groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_turnout'))

    df_const = df_dong.groupby(const_key)['sum_people'].sum().reset_index(name='sum_people')
    for frame in (go_dem_c, go_tot_c, go_turn_c):
        df_const = df_const.merge(frame, on=const_key, how='left')
    df_const = df_const.fillna(0)

    return df_dong, df_const, pd.DataFrame()

def merge_dong_with_covariates(df_election: pd.DataFrame, df_census: pd.DataFrame,
                                df_apt: pd.DataFrame, df_emp: pd.DataFrame) -> pd.DataFrame:
    if not df_census.empty:
        census_lookup = {}
        census_by_sgg = {}
        for _, row in df_census.iterrows():
            dnorm = row['dong_norm']
            covs  = {'demographic_propensity': row['demographic_propensity']}
            for c in AGE_GENDER_COLS: covs[c] = row.get(c, np.nan)
            for sgg in row['sgg_candidates']:
                census_lookup[(sgg, dnorm)] = covs
                census_by_sgg.setdefault(sgg, []).append(dnorm)

        results = []
        for _, row in df_election.iterrows():
            covs, dk, sgc = None, row['dong_norm'], row['sgg_candidates'] if isinstance(row['sgg_candidates'], list) else [row['primary_sgg']]
            if (row['primary_sgg'], dk) in census_lookup: covs = census_lookup[(row['primary_sgg'], dk)]
            if covs is None and '·' in dk:
                if (row['primary_sgg'], dk.replace('·', '')) in census_lookup: covs = census_lookup[(row['primary_sgg'], dk.replace('·', ''))]
            if covs is None:
                for sgg in sgc[1:]:
                    if (sgg, dk) in census_lookup: covs = census_lookup[(sgg, dk)]; break
            if covs is None:
                for sgg in sgc:
                    pool = census_by_sgg.get(sgg, [])
                    if pool:
                        m = get_close_matches(dk, pool, n=1, cutoff=0.82)
                        if m and (sgg, m[0]) in census_lookup: covs = census_lookup[(sgg, m[0])]; break
            if covs is None: covs = {k: np.nan for k in ['demographic_propensity'] + AGE_GENDER_COLS}
            rd = row.to_dict(); rd.update(covs)
            results.append(rd)
        df_out = pd.DataFrame(results)
    else:
        df_out = df_election.copy()

    if not df_apt.empty:
        df_out = df_out.merge(df_apt, left_on=['province_tag','primary_sgg','dong_norm'], right_on=['prov','sgg','dong_norm'], how='left')
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out.groupby('primary_sgg')['median_apt_price_sqm'].transform('median'))
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out.groupby('province_tag')['median_apt_price_sqm'].transform('median'))
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out['median_apt_price_sqm'].median())
        df_out['log_apt_price'] = np.log1p(df_out['median_apt_price_sqm'])
    else:
        df_out['log_apt_price'] = np.nan

    if not df_emp.empty:
        df_out = df_out.merge(df_emp[['province_tag','primary_sgg'] + EMPLOYMENT_FEATURE_COLS], on=['province_tag','primary_sgg'], how='left')
        for col in EMPLOYMENT_FEATURE_COLS:
            if col not in df_out.columns: df_out[col] = np.nan
            df_out[col] = df_out[col].fillna(df_out.groupby('province_tag')[col].transform('median')).fillna(df_out[col].median())
    else:
        for col in EMPLOYMENT_FEATURE_COLS: df_out[col] = np.nan

    return df_out[df_out['pct_f_4044'].notna()].copy()

def prep_election_data(df_dong_raw: pd.DataFrame, df_const_raw: pd.DataFrame,
                       df_census: pd.DataFrame, df_apt: pd.DataFrame,
                       df_emp: pd.DataFrame, df_stations: pd.DataFrame, elec_id: str) -> pd.DataFrame:
    dm = merge_dong_with_covariates(df_dong_raw, df_census, df_apt, df_emp)
    dm = dm[(dm['in_precinct_early_total'] > 50) & (dm['same_day_total'] > 50)].copy()

    no_dem = (dm['in_precinct_early_dem'] == 0) & (dm['same_day_dem'] == 0)
    if dm[no_dem].shape[0] > 0:
        dm = dm[~no_dem].copy()

    const_sums = dm.groupby(['시도명', '선거구명'])[['in_precinct_early_total', 'same_day_total']].transform('sum')
    dm['dong_weight'] = (dm['in_precinct_early_total'] + dm['same_day_total']) / (const_sums['in_precinct_early_total'] + const_sums['same_day_total']).replace(0, np.nan)

    df_const_sub = df_const_raw[['시도명', '선거구명', 'out_precinct_early_dem', 'out_precinct_early_total', 'out_precinct_early_turnout']].drop_duplicates()
    dm = dm.merge(df_const_sub, on=['시도명', '선거구명'], how='left')

    dm['out_precinct_alloc_dem']     = dm['out_precinct_early_dem'].fillna(0) * dm['dong_weight'].fillna(0)
    dm['out_precinct_alloc_tot']     = dm['out_precinct_early_total'].fillna(0) * dm['dong_weight'].fillna(0)
    dm['out_precinct_alloc_turnout'] = dm['out_precinct_early_turnout'].fillna(0) * dm['dong_weight'].fillna(0)

    dm['vote_share'] = ((dm['in_precinct_early_dem'] + dm['same_day_dem'] + dm['out_precinct_alloc_dem']) /
                        (dm['in_precinct_early_total'] + dm['same_day_total'] + dm['out_precinct_alloc_tot'])).fillna(0)
    dm['turnout'] = (dm['sum_vote_geo'] + dm['out_precinct_alloc_turnout']) / (dm['sum_people'] + dm['out_precinct_alloc_turnout']).replace(0, np.nan)
    dm['early_turnout'] = (dm['in_precinct_early_total'] + dm['out_precinct_alloc_tot']) / (dm['sum_people'] + dm['out_precinct_alloc_turnout']).replace(0, np.nan)
    dm['early_dem_share'] = (dm['in_precinct_early_dem'] + dm['out_precinct_alloc_dem']) / (dm['in_precinct_early_total'] + dm['out_precinct_alloc_tot']).replace(0, np.nan)
    dm['third_vote_share'] = ((dm['in_precinct_early_third'] + dm['same_day_third']) / (dm['in_precinct_early_total'] + dm['same_day_total'])).fillna(0)

    # ---------------------------------------------------------
    # INSTRUMENT INTEGRATION: voters_per_station
    # ---------------------------------------------------------
    if not df_stations.empty:
        dm = dm.merge(df_stations, on=['province_tag', 'primary_sgg', 'dong_norm'], how='left')
        dm['normal_station_count'] = dm['normal_station_count'].fillna(1)
    else:
        dm['normal_station_count'] = 1

    dm['voters_per_station'] = dm['sum_people'] / dm['normal_station_count']

    # Strict Local Demeaning (Province + Primary SGG Level)
    dm['voters_per_station_dm'] = dm['voters_per_station'] - dm.groupby(['province_tag', 'primary_sgg'])['voters_per_station'].transform('mean')

    if dm['voters_per_station_dm'].std(skipna=True) > 0:
        dm['voters_per_station_dm'] = StandardScaler().fit_transform(dm[['voters_per_station_dm']].fillna(0))
    else:
        dm['voters_per_station_dm'] = 0.0

    dm['election_id'] = str(elec_id)
    return dm

# ==========================================
# 4. PCA COLLINEARITY PIPELINE
# ==========================================
def reduce_pca(df: pd.DataFrame, feature_cols: list, threshold: float, name_label: str) -> tuple:
    available = [c for c in feature_cols if c in df.columns and df[c].notna().mean() >= MIN_COVERAGE]
    if len(available) < 2: return df.copy(), [], None, available

    valid_mask = df[available].notna().all(axis=1)
    if valid_mask.sum() < 30: return df.copy(), [], None, available

    X = df.loc[valid_mask, available].values.astype(float)
    X_scaled = StandardScaler().fit_transform(X)

    pca_probe = PCA().fit(X_scaled)
    cumvar = np.cumsum(pca_probe.explained_variance_ratio_)
    n_components = min(int(np.searchsorted(cumvar, threshold)) + 1, len(available))

    pca_final = PCA(n_components=n_components)
    components = pca_final.fit_transform(X_scaled)

    pc_cols = [f'pc{i+1}' for i in range(n_components)]
    df_out = df.copy()
    for col in pc_cols: df_out[col] = np.nan
    df_out.loc[valid_mask, pc_cols] = components

    loadings = pd.DataFrame(pca_final.components_.T, columns=pc_cols, index=available)
    out_path = f'output_data/pca_loadings_{name_label}.csv'
    loadings.to_csv(out_path)
    log_to_file(f"  -> Saved PCA Loadings for {name_label} to {out_path}")

    return df_out, pc_cols, pca_final, available

def prepare_continuous_covariates(df: pd.DataFrame, raw_age_cols: list, raw_emp_cols: list, label: str) -> tuple:
    combined_raw = list(raw_age_cols) + list(raw_emp_cols)
    if 'log_apt_price' in df.columns: combined_raw.append('log_apt_price')

    df, pc_cols, pca_model, used_features = reduce_pca(df, combined_raw, PCA_VARIANCE_THRESHOLD, label)
    pc_cols = [c for c in pc_cols if df[c].notna().mean() >= MIN_COVERAGE]

    dm_cols = []
    if pc_cols:
        active_groups = [g for g in DEMEAN_GROUP_COLS if g in df.columns]
        for col in pc_cols:
            df[f'{col}_dm'] = df[col] - df.groupby(active_groups)[col].transform('mean')
            dm_cols.append(f'{col}_dm')

    dm_cols = [c for c in dm_cols if df[c].std(skipna=True) > 1e-10]

    if dm_cols:
        valid_mask = df[dm_cols].notna().all(axis=1)
        if valid_mask.sum() > len(dm_cols) + 5:
            df.loc[valid_mask, dm_cols] = StandardScaler().fit_transform(df.loc[valid_mask, dm_cols])

    return df, dm_cols, pca_model, used_features


# ==========================================
# 5. REGRESSIONS (WLS & IV)
# ==========================================
def run_single_analysis(df_single: pd.DataFrame, elec_id, dm_cols: list):
    label = f"Election_{elec_id}"
    header = f"\n{'='*70}\n  SINGLE ELECTION ANALYSIS: {label}\n{'='*70}"
    log_to_file(header)

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share', 'sum_people',
                'area2_name', 'province_tag', 'voters_per_station_dm'] + dm_cols
    df_sub = df_single.dropna(subset=req_cols).copy()

    df_sub.to_csv(f'output_data/regression_data_{label}{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    if df_sub.empty:
        log_to_file(f" [!] Not enough data to run analysis for {label}.")
        return

    exog_vars = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag)"

    # 1. Turnout Regressions
    for target in ['turnout', 'early_turnout']:
        log_to_file(f"\n--- Sub-Target: {target} ---")
        exog_str = exog_vars + " + voters_per_station_dm"

        formula = f"{target} ~ {exog_str}"
        model = smf.wls(formula, data=df_sub, weights=df_sub['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_sub['area2_name']})
        log_to_file(f"[Formula]: {formula}")
        log_to_file(model.summary().as_text())

    # 2. Total Dem Vote Share & Early Dem Share Regressions
    for main_target in ['vote_share', 'early_dem_share']:
        turnout_vars = ['early_turnout', 'turnout'] if INCLUDE_TURNOUT_IN_VOTESHARE else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (Instrumenting {turnout_var})\n{'='*55}")
                formula_vs = f"{main_target} ~ {exog_vars} + [{turnout_var} ~ voters_per_station_dm]"
                model_vs = IV2SLS.from_formula(formula_vs, df_sub, weights=df_sub['sum_people'])
                res_vs = model_vs.fit(cov_type='robust')

                log_to_file(f"\n[Formula]: {formula_vs}")
                if hasattr(res_vs, 'first_stage') and res_vs.first_stage is not None:
                    log_to_file("\n[First Stage F-Stats]")
                    for endog in res_vs.first_stage.diagnostics.index:
                        f_stat = res_vs.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file("\n[Second Stage Summary]")
                log_to_file(res_vs.summary.as_text())
            else:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (No Turnout Covariate)\n{'='*55}")
                formula_vs = f"{main_target} ~ {exog_vars}"
                model_vs = smf.wls(formula_vs, data=df_sub, weights=df_sub['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_sub['area2_name']})
                log_to_file(f"\n[Formula]: {formula_vs}")
                log_to_file(model_vs.summary().as_text())


def run_iv_causal_analysis(df_pres21: pd.DataFrame, df_pres19: pd.DataFrame, dm_cols: list):
    header = f"\n{'='*70}\n  EXACTLY IDENTIFIED IV (2SLS) ANALYSIS (21st Pres)\n{'='*70}"
    log_to_file(header)

    instrument_df = df_pres19[['province_tag', 'primary_sgg', 'dong_norm', 'third_vote_share']].drop_duplicates()
    instrument_df = instrument_df.rename(columns={'third_vote_share': 'yoo_vote_share_19'})

    df_iv = df_pres21.rename(columns={'third_vote_share': 'LJS_vote_share'})
    df_iv = df_iv.merge(instrument_df, on=['province_tag', 'primary_sgg', 'dong_norm'], how='inner')

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share',
                'LJS_vote_share', 'yoo_vote_share_19', 'sum_people', 'voters_per_station_dm', 'area2_name'] + dm_cols
    df_iv = df_iv.dropna(subset=req_cols).copy()

    df_iv.to_csv(f'output_data/regression_data_iv_pres21{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    exog_vars = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag)"

    # 1. Turnout Regressions
    for target in ['turnout', 'early_turnout']:
        log_to_file(f"\n\n{'='*55}\n Target: {target}\n{'='*55}")

        exog_str = exog_vars + " + voters_per_station_dm"

        if INCLUDE_LJS_IN_VOTESHARE:
            formula = f"{target} ~ {exog_str} + [LJS_vote_share ~ yoo_vote_share_19]"
            res = IV2SLS.from_formula(formula, df_iv, weights=df_iv['sum_people']).fit(cov_type='robust')
            log_to_file(f"\n[Formula]: {formula}")

            if hasattr(res, 'first_stage') and res.first_stage is not None:
                log_to_file("\n[First Stage F-Stats]")
                for endog in res.first_stage.diagnostics.index:
                    f_stat = res.first_stage.diagnostics.loc[endog, 'f.stat']
                    log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
            log_to_file("\n[Second Stage Summary]")
            log_to_file(res.summary.as_text())
        else:
            formula = f"{target} ~ {exog_str}"
            res = smf.wls(formula, data=df_iv, weights=df_iv['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_iv['area2_name']})
            log_to_file(f"\n[Formula]: {formula}")
            log_to_file(res.summary().as_text())

    # 2. Vote Share & Early Dem Share Regressions (Dynamic string building based on toggles)
    for main_target in ['vote_share', 'early_dem_share']:
        turnout_vars = ['early_turnout', 'turnout'] if INCLUDE_TURNOUT_IN_VOTESHARE else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (Turnout Covariate: {turnout_var})\n{'='*55}")
            else:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (No Turnout Covariate)\n{'='*55}")

            endogs, instrs = [], []

            if INCLUDE_LJS_IN_VOTESHARE:
                endogs.append('LJS_vote_share')
                instrs.append('yoo_vote_share_19')
            if INCLUDE_TURNOUT_IN_VOTESHARE and turnout_var:
                endogs.append(turnout_var)
                instrs.append('voters_per_station_dm')

            if endogs:
                formula = f"{main_target} ~ {exog_vars} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res = IV2SLS.from_formula(formula, df_iv, weights=df_iv['sum_people']).fit(cov_type='robust')

                log_to_file(f"\n[Formula]: {formula}")
                if hasattr(res, 'first_stage') and res.first_stage is not None:
                    log_to_file("\n[First Stage F-Stats]")
                    for endog in res.first_stage.diagnostics.index:
                        f_stat = res.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file("\n[Second Stage Summary]")
                log_to_file(res.summary.as_text())
            else:
                formula = f"{main_target} ~ {exog_vars}"
                res = smf.wls(formula, data=df_iv, weights=df_iv['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_iv['area2_name']})
                log_to_file(f"\n[Formula]: {formula}")
                log_to_file(res.summary().as_text())

    return df_iv


def run_pooled_iv_panel(df_pooled: pd.DataFrame, df_pres19: pd.DataFrame, dm_cols: list):
    header = f"\n{'='*70}\n  POOLED IV PANEL REGRESSIONS\n{'='*70}"
    log_to_file(header)

    df_panel = df_pooled.copy()

    inst_df = df_pres19[['province_tag', 'primary_sgg', 'dong_norm', 'third_vote_share']].drop_duplicates()
    inst_df = inst_df.rename(columns={'third_vote_share': 'base_yoo_pr'})

    df_panel = df_panel.merge(inst_df, on=['province_tag', 'primary_sgg', 'dong_norm'], how='inner')

    df_panel['LJS_vote_share'] = np.where(df_panel['election_id'] == 'pres21', df_panel['third_vote_share'], 0)
    df_panel['active_yoo_pr'] = np.where(df_panel['election_id'] == 'pres21', df_panel['base_yoo_pr'], 0)
    df_panel['prov_elec'] = df_panel['province_tag'] + "_" + df_panel['election_id'].astype(str)

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share',
                'LJS_vote_share', 'active_yoo_pr', 'sum_people', 'election_id',
                'province_tag', 'prov_elec', 'voters_per_station_dm', 'area2_name'] + dm_cols
    df_panel = df_panel.dropna(subset=req_cols).copy()

    df_panel.to_csv(f'output_data/regression_data_pooled_panel{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    exog_vars_add = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag) + C(election_id)"
    exog_vars_int = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(prov_elec)"

    for target in ['turnout', 'early_turnout', 'early_dem_share', 'vote_share']:
        log_to_file(f"\n\n{'='*55}\n PANEL TARGET: {target}\n{'='*55}")

        exog_add_base = exog_vars_add
        exog_int_base = exog_vars_int

        if target in ['turnout', 'early_turnout']:
            exog_add_base += " + voters_per_station_dm"
            exog_int_base += " + voters_per_station_dm"

        turnout_vars = ['early_turnout', 'turnout'] if (target in ['vote_share', 'early_dem_share'] and INCLUDE_TURNOUT_IN_VOTESHARE) else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n--- Sub-Target for {target} (Turnout Covariate: {turnout_var}) ---")

            endogs, instrs = [], []

            if INCLUDE_LJS_IN_VOTESHARE:
                endogs.append('LJS_vote_share')
                instrs.append('active_yoo_pr')

            if target in ['vote_share', 'early_dem_share'] and INCLUDE_TURNOUT_IN_VOTESHARE and turnout_var:
                endogs.append(turnout_var)
                instrs.append('voters_per_station_dm')

            # Execute Spec 1 (Additive)
            log_to_file(f"\n[Spec 1: Additive Province & Election FEs]")
            if endogs:
                formula_add = f"{target} ~ {exog_add_base} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res_add = IV2SLS.from_formula(formula_add, df_panel, weights=df_panel['sum_people']).fit(cov_type='robust')
                log_to_file(f"[Formula]: {formula_add}")
                if hasattr(res_add, 'first_stage') and res_add.first_stage is not None:
                    log_to_file("[First Stage F-Stats]")
                    for endog in res_add.first_stage.diagnostics.index:
                        f_stat = res_add.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file(res_add.summary.as_text())
            else:
                formula_add = f"{target} ~ {exog_add_base}"
                res_add = smf.wls(formula_add, data=df_panel, weights=df_panel['sum_people']).fit(cov_type='HC3')
                log_to_file(f"[Formula]: {formula_add}")
                log_to_file(res_add.summary().as_text())

            # Execute Spec 2 (Interacted)
            log_to_file(f"\n[Spec 2: Interacted Province x Election FEs]")
            if endogs:
                formula_int = f"{target} ~ {exog_int_base} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res_int = IV2SLS.from_formula(formula_int, df_panel, weights=df_panel['sum_people']).fit(cov_type='robust')
                log_to_file(f"[Formula]: {formula_int}")
                if hasattr(res_int, 'first_stage') and res_int.first_stage is not None:
                    log_to_file("[First Stage F-Stats]")
                    for endog in res_int.first_stage.diagnostics.index:
                        f_stat = res_int.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file(res_int.summary.as_text())
            else:
                formula_int = f"{target} ~ {exog_int_base}"
                res_int = smf.wls(formula_int, data=df_panel, weights=df_panel['sum_people']).fit(cov_type='HC3')
                log_to_file(f"[Formula]: {formula_int}")
                log_to_file(res_int.summary().as_text())


# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    with open(LOG_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write("COMPREHENSIVE REGRESSION & DATA LOG\n")
        f.write("="*60 + "\n")

    all_dm = []

    for elec_id in ['pres19', 21, 22, 'pres20', 'pres21']:
        cfg = ELECTION_CONFIGS[elec_id]
        log_to_file(f"\n--- Loading Data for {cfg['label']} ---")

        df_census = load_census_csv(cfg.get('census_csv'))
        df_apt = load_apt_csv(cfg.get('apt_csv_glob'), elec_id)
        df_emp = load_employment_sgg(election_key=elec_id)
        df_stations = load_polling_stations(elec_id)

        log_to_file(f"  > Census Data: {'Loaded successfully' if not df_census.empty else 'NO DATA FOUND'}")

        apt_cache_path = f"output_data/processed_apt_data_{elec_id}.csv"
        if not df_apt.empty:
            if os.path.exists(apt_cache_path) and os.path.getmtime(apt_cache_path) > 0:
                log_to_file(f"  > APT Price Data: Loaded from Cache ({apt_cache_path})")
            else:
                log_to_file(f"  > APT Price Data: Parsed from Raw and Saved to Cache")
        else:
            log_to_file(f"  > APT Price Data: NO DATA FOUND")

        log_to_file(f"  > Employment Data: {'Loaded successfully' if not df_emp.empty else 'NO DATA FOUND'}")
        log_to_file(f"  > Polling Stations: {'Loaded successfully' if not df_stations.empty else 'NO DATA FOUND'}")

        df_dong, df_const, _ = load_election_csv(
            cfg['result_csv'],
            dem_pattern=cfg['dem_pattern'],
            con_pattern=cfg['con_pattern'],
            third_pattern=cfg.get('third_pattern'),
            election_type=cfg['election_type'],
        )

        if EXCLUDE_HONAM_YEONGNAM:
            exclude_provs = PARTISAN_REGION_PROVINCES['Honam'] | PARTISAN_REGION_PROVINCES['Yeongnam']
            short_exclude = {PROV_FULL_TO_SHORT.get(p, p) for p in exclude_provs} | exclude_provs
            df_dong  = df_dong[~df_dong['province_tag'].isin(short_exclude)]
            df_const = df_const[~df_const['province_tag'].isin(short_exclude)]

        if not df_dong.empty and not df_const.empty:
            dm = prep_election_data(df_dong, df_const, df_census, df_apt, df_emp, df_stations, elec_id)
            all_dm.append(dm)
            log_to_file(f"  > Processed {len(dm)} distinct dong-level observations.")

    df_raw_pooled = pd.concat(all_dm, ignore_index=True)
    raw_age_cols = [c for c in AGE_GENDER_COLS if c in df_raw_pooled.columns]
    raw_emp_cols = [c for c in EMPLOYMENT_FEATURE_COLS if c in df_raw_pooled.columns]

    # ---------------------------------------------------------
    # PART A: ISOLATED SINGLE-ELECTION PIPELINE
    # ---------------------------------------------------------
    log_to_file("\n\n" + "#"*70 + "\n PART A: ISOLATED SINGLE-ELECTION PIPELINE\n" + "#"*70)

    for elec in [21, 22, 'pres20']:
        df_single_raw = df_raw_pooled[df_raw_pooled['election_id'] == str(elec)].copy()
        if not df_single_raw.empty:
            df_single_pca, dm_cols_single, _, _ = prepare_continuous_covariates(df_single_raw, raw_age_cols, raw_emp_cols, label=f"isolated_{elec}")
            run_single_analysis(df_single_pca, elec, dm_cols_single)

    df_pres21_raw = df_raw_pooled[df_raw_pooled['election_id'] == 'pres21'].copy()
    df_pres19_raw = df_raw_pooled[df_raw_pooled['election_id'] == 'pres19'].copy()

    if not df_pres21_raw.empty and not df_pres19_raw.empty:
        df_pres21_pca, dm_cols_iv, _, _ = prepare_continuous_covariates(df_pres21_raw, raw_age_cols, raw_emp_cols, label="isolated_pres21")
        run_iv_causal_analysis(df_pres21_pca, df_pres19_raw, dm_cols_iv)

    # ---------------------------------------------------------
    # PART B: POOLED ELECTION ANALYSIS
    # ---------------------------------------------------------
    log_to_file("\n\n" + "#"*70 + "\n PART B: POOLED PANEL ELECTION PIPELINE\n" + "#"*70)

    df_pooled_pca, dm_cols_pooled, _, _ = prepare_continuous_covariates(df_raw_pooled, raw_age_cols, raw_emp_cols, label="pooled_panel")

    if not df_pres19_raw.empty:
        run_pooled_iv_panel(df_pooled_pca, df_pres19_raw, dm_cols_pooled)

    print(f"\n[+] All tasks completed. Full statistical logs exported to -> {LOG_FILE_PATH}")


--- Loading Data for 19th Presidential Election (2017) ---


/tmp/ipykernel_4025/2111490544.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: NO DATA FOUND
  > Employment Data: Loaded successfully
  > Polling Stations: NO DATA FOUND
  > Processed 3537 distinct dong-level observations.

--- Loading Data for 21st General Election (2020) ---


/tmp/ipykernel_4025/2111490544.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_21.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 5519 distinct dong-level observations.

--- Loading Data for 22nd General Election (2024) ---


...

  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_22.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 4513 distinct dong-level observations.

--- Loading Data for 20th Presidential Election (2022) ---


/tmp/ipykernel_4025/2111490544.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_pres20.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 5976 distinct dong-level observations.

--- Loading Data for 21st Presidential Election (2025) ---


/tmp/ipykernel_4025/2111490544.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_pres21.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 6050 distinct dong-level observations.


######################################################################
 PART A: ISOLATED SINGLE-ELECTION PIPELINE
######################################################################
  -> Saved PCA Loadings for isolated_21 to output_data/pca_loadings_isolated_21.csv

  SINGLE ELECTION ANALYSIS: Election_21

--- Sub-Target: turnout ---
[Formula]: turnout ~ pc1_dm + pc2_dm + pc3_dm + pc4_dm + pc5_dm + pc6_dm + pc7_dm + pc8_dm + pc9_dm + pc10_dm + pc11_dm + C(province_tag) + voters_per_station_dm
                            WLS Regression Results                            
Dep. Variable:                turnout   R-squared:                       0.291
Model:                            WLS   Adj. R-squared:                  0.288
Metho

turnout (either total or early) covariates turned on in democratic vote share regressions

NO LJS vote share covariate in democratic vote share regressions

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import glob
from difflib import get_close_matches
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.formula.api as smf

try:
    from linearmodels.iv import IV2SLS
except ImportError:
    print("[!] 'linearmodels' not found. Please run: pip install linearmodels")
    exit(1)

# ==========================================
# CONFIGURATION & GLOBAL SETUP
# ==========================================

# --- GLOBAL TOGGLES ---
EXCLUDE_HONAM_YEONGNAM       = False
INCLUDE_TURNOUT_IN_VOTESHARE = True   # Toggle Turnout IV in Vote Share Regressions
INCLUDE_LJS_IN_VOTESHARE     = False   # Toggle LJS Vote Share IV in relevant Regressions

# --- DYNAMIC FILENAME SUFFIX ---
if INCLUDE_LJS_IN_VOTESHARE and INCLUDE_TURNOUT_IN_VOTESHARE:
    IV_SUFFIX = "_LJS_turnout_IV_cov"
elif INCLUDE_LJS_IN_VOTESHARE:
    IV_SUFFIX = "_LJS_IV_cov"
elif INCLUDE_TURNOUT_IN_VOTESHARE:
    IV_SUFFIX = "_turnout_IV_cov"
else:
    IV_SUFFIX = "_no_IV_cov"

os.makedirs('output_data', exist_ok=True)
LOG_FILE_PATH = f'output_data/comprehensive_regression_logs{IV_SUFFIX}.txt'

PCA_VARIANCE_THRESHOLD = 0.90
DEMEAN_GROUP_COLS      = ['province_tag']
MIN_COVERAGE           = 0.50

ELECTION_CONFIGS = {
    'pres19': {
        'election_type':  'presidential',
        'census_csv':     '19th_presidential_election_census.csv',
        'result_csv':     '19th_presidential_election_result.csv',
        'apt_csv_glob':   None,
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'자유한국당',
        'third_pattern':  r'유승민|바른정당',
        'label':          '19th Presidential Election (2017)',
        'year':           2017,
        'election_month': 5,
    },
    21: {
        'election_type':  'general',
        'census_csv':     '21st_election_census.csv',
        'result_csv':     '21st_election_result.csv',
        'apt_csv_glob':   '*21st_election_*_apt_price.csv',
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'미래통합당|자유한국당',
        'third_pattern':  None,
        'label':          '21st General Election (2020)',
        'year':           2020,
        'election_month': 4,
    },
    22: {
        'election_type':  'general',
        'census_csv':     '22nd_election_census.csv',
        'result_csv':     '22nd_election_result.csv',
        'apt_csv_glob':   '*22nd_election_*_apt_price.csv',
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'국민의힘',
        'third_pattern':  r'개혁신당',
        'label':          '22nd General Election (2024)',
        'year':           2024,
        'election_month': 4,
    },
    'pres20': {
        'election_type':         'presidential',
        'census_csv':            '20th_presidential_election_census.csv',
        'result_csv':            '20th_presidential_election_result.csv',
        'apt_csv_glob':          '*20th_presidential_election_*_apt_prices.csv',
        'dem_pattern':           r'더불어민주당',
        'con_pattern':           r'국민의힘',
        'third_pattern':         None,
        'label':                 '20th Presidential Election (2022)',
        'year':                  2022,
        'election_month':        3,
    },
    'pres21': {
        'election_type':         'presidential',
        'census_csv':            '21st_presidential_election_census.csv',
        'result_csv':            '21st_presidential_election_result.csv',
        'apt_csv_glob':          '*21st_presidential_election_*_apt_prices.csv',
        'dem_pattern':           r'더불어민주당',
        'con_pattern':           r'국민의힘',
        'third_pattern':         r'이준석|개혁신당',
        'label':                 '21st Presidential Election (2025)',
        'year':                  2025,
        'election_month':        6,
    },
}

SPECIAL_DONG_NAMES_GENERAL = {
    '거소·선상투표', '관외사전투표', '국외부재자투표',
    '국외부재자투표(공관)', '잘못 투입·구분된 투표지',
}
SPECIAL_DONG_NAMES_PRESIDENTIAL = {
    '거소·선상투표', '관외사전투표', '재외투표',
    '잘못 투입·구분된 투표지',
}
GWANNAESA_LABEL = '관내사전투표'
META_CANDIDATES = {'선거인수', '투표수', '무효 투표수', '기권자수'}

PROV_FULL_TO_SHORT = {
    '서울특별시': 'Seoul',  '부산광역시': 'Busan',   '대구광역시': 'Daegu',
    '인천광역시': 'Incheon','광주광역시': 'Gwangju', '대전광역시': 'Daejeon',
    '울산광역시': 'Ulsan',  '세종특별자치시': 'Sejong',
    '경기도': 'Gyeonggi',  '강원도': 'Gangwon',     '강원특별자치도': 'Gangwon',
    '충청북도': 'Chungbuk', '충청남도': 'Chungnam',
    '전라북도': 'Jeonbuk',  '전북특별자치도': 'Jeonbuk', '전라남도': 'Jeonnam',
    '경상북도': 'Gyeongbuk','경상남도': 'Gyeongnam', '제주특별자치도': 'Jeju',
    '서울': 'Seoul',   '부산': 'Busan',    '대구': 'Daegu',
    '인천': 'Incheon', '광주': 'Gwangju',  '대전': 'Daejeon',
    '울산': 'Ulsan',
}

SGG_TO_PROVINCE_EMP = {
    '가평군':'Gyeonggi','고양시':'Gyeonggi','과천시':'Gyeonggi','광명시':'Gyeonggi',
    '광주시':'Gyeonggi','구리시':'Gyeonggi','군포시':'Gyeonggi','김포시':'Gyeonggi',
    '남양주시':'Gyeonggi','동두천시':'Gyeonggi','부천시':'Gyeonggi','성남시':'Gyeonggi',
    '수원시':'Gyeonggi','시흥시':'Gyeonggi','안산시':'Gyeonggi','안성시':'Gyeonggi',
    '안양시':'Gyeonggi','양주시':'Gyeonggi','양평군':'Gyeonggi','여주군':'Gyeonggi',
    '여주시':'Gyeonggi','연천군':'Gyeonggi','오산시':'Gyeonggi','용인시':'Gyeonggi',
    '의왕시':'Gyeonggi','의정부시':'Gyeonggi','이천시':'Gyeonggi','파주시':'Gyeonggi',
    '평택시':'Gyeonggi','포천시':'Gyeonggi','하남시':'Gyeonggi','화성시':'Gyeonggi',
    '강릉시':'Gangwon','고성군':'Gangwon','동해시':'Gangwon','삼척시':'Gangwon',
    '속초시':'Gangwon','양구군':'Gangwon','양양군':'Gangwon','영월군':'Gangwon',
    '원주시':'Gangwon','인제군':'Gangwon','정선군':'Gangwon','철원군':'Gangwon',
    '춘천시':'Gangwon','태백시':'Gangwon','평창군':'Gangwon','홍천군':'Gangwon',
    '화천군':'Gangwon','횡성군':'Gangwon','괴산군':'Chungbuk','단양군':'Chungbuk',
    '보은군':'Chungbuk','영동군':'Chungbuk','옥천군':'Chungbuk','음성군':'Chungbuk',
    '제천시':'Chungbuk','증평군':'Chungbuk','진천군':'Chungbuk','청원군':'Chungbuk',
    '청주시':'Chungbuk','충주시':'Chungbuk','계룡시':'Chungnam','공주시':'Chungnam',
    '금산군':'Chungnam','논산시':'Chungnam','당진시':'Chungnam','보령시':'Chungnam',
    '부여군':'Chungnam','서산시':'Chungnam','서천군':'Chungnam','아산시':'Chungnam',
    '연기군':'Chungnam','예산군':'Chungnam','청양군':'Chungnam','천안시':'Chungnam',
    '태안군':'Chungnam','홍성군':'Chungnam','고창군':'Jeonbuk','군산시':'Jeonbuk',
    '김제시':'Jeonbuk','남원시':'Jeonbuk','무주군':'Jeonbuk','부안군':'Jeonbuk',
    '순창군':'Jeonbuk','완주군':'Jeonbuk','익산시':'Jeonbuk','임실군':'Jeonbuk',
    '장수군':'Jeonbuk','전주시':'Jeonbuk','정읍시':'Jeonbuk','진안군':'Jeonbuk',
    '강진군':'Jeonnam','고흥군':'Jeonnam','곡성군':'Jeonnam','구례군':'Jeonnam',
    '나주시':'Jeonnam','담양군':'Jeonnam','목포시':'Jeonnam','무안군':'Jeonnam',
    '보성군':'Jeonnam','순천시':'Jeonnam','신안군':'Jeonnam','여수시':'Jeonnam',
    '영광군':'Jeonnam','영암군':'Jeonnam','완도군':'Jeonnam','장성군':'Jeonnam',
    '장흥군':'Jeonnam','진도군':'Jeonnam','함평군':'Jeonnam','화순군':'Jeonnam',
    '광양시':'Jeonnam','해남군':'Jeonnam','경산시':'Gyeongbuk','경주시':'Gyeongbuk',
    '고령군':'Gyeongbuk','구미시':'Gyeongbuk','김천시':'Gyeongbuk','문경시':'Gyeongbuk',
    '봉화군':'Gyeongbuk','상주시':'Gyeongbuk','성주군':'Gyeongbuk','안동시':'Gyeongbuk',
    '영덕군':'Gyeongbuk','영양군':'Gyeongbuk','영주시':'Gyeongbuk','영천시':'Gyeongbuk',
    '예천군':'Gyeongbuk','울릉군':'Gyeongbuk','울진군':'Gyeongbuk','의성군':'Gyeongbuk',
    '청도군':'Gyeongbuk','청송군':'Gyeongbuk','칠곡군':'Gyeongbuk','포항시':'Gyeongbuk',
    '거제시':'Gyeongnam','거창군':'Gyeongnam','김해시':'Gyeongnam','남해군':'Gyeongnam',
    '밀양시':'Gyeongnam','사천시':'Gyeongnam','산청군':'Gyeongnam','양산시':'Gyeongnam',
    '의령군':'Gyeongnam','진주시':'Gyeongnam','창녕군':'Gyeongnam','창원시':'Gyeongnam',
    '통영시':'Gyeongnam','하동군':'Gyeongnam','함안군':'Gyeongnam','함양군':'Gyeongnam',
    '합천군':'Gyeongnam','서귀포시':'Jeju','제주시':'Jeju',
}

PARTISAN_REGION_PROVINCES = {
    'Honam':   {'Jeonnam', 'Jeonbuk', 'Gwangju', '전남', '전북', '광주'},
    'Yeongnam':{'Gyeongbuk', 'Gyeongnam', 'Daegu', 'Busan', 'Ulsan', '경북', '경남', '대구', '부산', '울산'},
}

AGE_GENDER_COLS = [
    'pct_m_1824', 'pct_m_2529', 'pct_m_3034', 'pct_m_3539', 'pct_m_4044', 'pct_m_4549',
    'pct_m_5054', 'pct_m_5559', 'pct_m_6064', 'pct_m_6569', 'pct_m_70plus',
    'pct_f_1824', 'pct_f_2529', 'pct_f_3034', 'pct_f_3539', 'pct_f_4044', 'pct_f_4549',
    'pct_f_5054', 'pct_f_5559', 'pct_f_6064', 'pct_f_6569', 'pct_f_70plus',
]

EMPLOYMENT_FEATURE_COLS = [
    'emp_men_total', 'emp_men_1529', 'emp_men_3049', 'emp_men_5064', 'emp_men_65plus',
    'emp_wmn_total', 'emp_wmn_1529', 'emp_wmn_3049', 'emp_wmn_5064', 'emp_wmn_65plus',
    'occ_wage_share', 'occ_regular_share', 'occ_nonwage_share',
    'job_professional', 'job_clerical', 'job_service_sales',
    'job_skilled_machine', 'job_simple_labor', 'job_farming',
    'ind_agriculture', 'ind_manufacturing', 'ind_construction',
    'ind_retail_food', 'ind_transport_finance', 'ind_services',
    'nonp_male_share', 'nonp_young_share', 'nonp_middle_share', 'nonp_old_share',
]

# ==========================================
# FILE WRITER HELPER
# ==========================================
def log_to_file(text: str, mode='a'):
    print(text)
    with open(LOG_FILE_PATH, mode, encoding='utf-8') as f:
        f.write(text + "\n")

# ==========================================
# NORMALISATION & IO UTILITIES
# ==========================================
def _read_csv_auto(path: str, **kwargs) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding='utf-8', **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='cp949', **kwargs)

def _to_numeric_safe(series: pd.Series) -> pd.Series:
    return (series.astype(str).str.strip()
            .replace(['-', '.', '', 'nan'], np.nan)
            .pipe(pd.to_numeric, errors='coerce'))

def normalize_dong_name(name: str) -> str:
    if not isinstance(name, str): return ""
    name = re.sub(r'\(.*?\)', '', name).strip().replace('.', '·')
    name = re.sub(r'제(\d)', r'\1', name)
    name = re.sub(r'·\d+', '', name)
    name = re.sub(r'(\d+)(동|읍|면)$', r'\2', name)
    return re.sub(r'\s+', ' ', name)

def split_admin_tokens(name: str) -> list:
    tokens, buf = [], []
    for ch in name:
        buf.append(ch)
        if ch in '시군구' and len(buf) >= 2:
            tokens.append(''.join(buf))
            buf = []
    if buf: tokens.append(''.join(buf))
    return [t for t in tokens if t]

def normalize_sigungu(name: str) -> list:
    if not isinstance(name, str): return []
    name = re.sub(r'\(.*?\)', '', name).strip()
    if not name: return []
    tokens = split_admin_tokens(name)
    if not tokens:
        stripped = re.sub(r'[시군구갑을병정무]$', '', name).strip()
        return [stripped] if stripped else []
    si_gun_count = sum(1 for t in tokens if t[-1] in '시군' and len(t) >= 2)
    gu_count     = sum(1 for t in tokens if t[-1] == '구'  and len(t) >= 2)
    ordered = tokens if (si_gun_count >= 2 or (si_gun_count == 0 and gu_count >= 2)) else list(reversed(tokens))
    candidates = []
    for t in ordered:
        key = re.sub(r'[시군구]$', '', t).strip()
        if key and key not in candidates: candidates.append(key)
    return candidates

def get_election_periods(month: int, year: int) -> dict:
    if month <= 6:
        prev = year - 1
        return {'half_periods': [f'{prev}.1/2', f'{prev}.2/2'], 'grdp_year': prev}
    else:
        prev = year - 1
        return {'half_periods': [f'{prev}.2/2', f'{year}.1/2'], 'grdp_year': prev}

def parse_employment_sgg(region_name: str) -> tuple:
    if not isinstance(region_name, str): return ('', '')
    parts = region_name.strip().split(None, 1)
    if not parts: return ('', '')
    prov = parts[0]
    if len(parts) == 1:
        prov_eng = SGG_TO_PROVINCE_EMP.get(prov, '')
        primary  = re.sub(r'[시군구]$', '', prov).strip()
        return (prov_eng, primary)
    sgg_raw  = parts[1]
    sgg_norm = normalize_sigungu(sgg_raw)
    primary  = sgg_norm[0] if sgg_norm else re.sub(r'[시군구]$', '', sgg_raw).strip()
    prov_eng = PROV_FULL_TO_SHORT.get(prov, prov)
    return (prov_eng, primary)

# ==========================================
# 1. DEMOGRAPHIC, ASSET & STATION DATA LOADERS
# ==========================================
def load_polling_stations(elec_id) -> pd.DataFrame:
    # Look strictly for normal-day station files to proxy physical density friction
    paths_to_try = [
        f"stations_normal_{elec_id}.csv",
        f"polling_stations_{elec_id}.csv",
        f"stations_normal_20240410.csv" if str(elec_id) == '22' else f"stations_normal_{elec_id}.csv"
    ]

    df = pd.DataFrame()
    for path in paths_to_try:
        if os.path.exists(path):
            df = _read_csv_auto(path)
            break

    if df.empty: return df

    if 'province_tag' not in df.columns and 'sdName' in df.columns:
        df['province_tag'] = df['sdName'].map(PROV_FULL_TO_SHORT).fillna(df['sdName'])
        def extract_sgg(x):
            cands = normalize_sigungu(x)
            return cands[0] if cands else re.sub(r'[시군구]$', '', str(x)).strip()
        df['primary_sgg'] = df['wiwName'].apply(extract_sgg)
        df['dong_norm']   = df['emdName'].apply(normalize_dong_name)

        mask_seoul = df['province_tag'] == 'Seoul'
        mask_g3 = df['primary_sgg'].isin(['강남', '서초', '송파'])
        df.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
        df.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    if 'normal_station_count' not in df.columns:
        if 'station_count' in df.columns:
            df = df.rename(columns={'station_count': 'normal_station_count'})
        else:
            df['normal_station_count'] = 1

    return df[['province_tag', 'primary_sgg', 'dong_norm', 'normal_station_count']]


def _detect_year_prefix(df: pd.DataFrame) -> str:
    for col in df.columns:
        m = re.match(r'(\d{4}년\d{2}월)_계_총인구수', col)
        if m: return m.group(1)
    raise ValueError("Cannot detect census year prefix.")

def load_census_csv(csv_path: str) -> pd.DataFrame:
    if not csv_path or not os.path.exists(csv_path): return pd.DataFrame()
    try:
        df = _read_csv_auto(csv_path, low_memory=False)
        prefix = _detect_year_prefix(df)
        voting_age_cols = ([f"{prefix}_계_{a}세" for a in range(18, 100)] + [f"{prefix}_계_100세 이상"])
        cols_4059 = [f"{prefix}_계_{a}세" for a in range(40, 60)]
        all_target_cols = list(voting_age_cols)
        for g in ['남', '여']:
            for a in range(18, 100):
                all_target_cols.append(f"{prefix}_{g}_{a}세")
            all_target_cols.append(f"{prefix}_{g}_100세 이상")

        for col in set(all_target_cols):
            if col in df.columns:
                df[col] = (df[col].astype(str).str.replace(',', '', regex=False).pipe(pd.to_numeric, errors='coerce').fillna(0))

        df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)
        df = df[df['total_voting_pop'] > 0].copy()

        ranges = [(18,25,'1824'),(25,30,'2529'),(30,35,'3034'),(35,40,'3539'),
                  (40,45,'4044'),(45,50,'4549'),(50,55,'5054'),(55,60,'5559'),
                  (60,65,'6064'),(65,70,'6569')]

        for g, g_str in [('남','m'), ('여','f')]:
            for r_start, r_end, r_str in ranges:
                cols = [f"{prefix}_{g}_{a}세" for a in range(r_start, r_end)]
                df[f'pct_{g_str}_{r_str}'] = (df[[c for c in cols if c in df.columns]].sum(axis=1) / df['total_voting_pop'])
            cols_70 = ([f"{prefix}_{g}_{a}세" for a in range(70, 100)] + [f"{prefix}_{g}_100세 이상"])
            df[f'pct_{g_str}_70plus'] = (df[[c for c in cols_70 if c in df.columns]].sum(axis=1) / df['total_voting_pop'])

        df['demographic_propensity'] = (df[[c for c in cols_4059 if c in df.columns]].sum(axis=1) / df['total_voting_pop'])

        def extract_census_keys(admin_name):
            if not isinstance(admin_name, str): return [], ""
            clean = re.sub(r'\(.*?\)', '', admin_name).strip()
            parts = clean.split()
            dong_norm = normalize_dong_name(parts[-1]) if parts else ""
            sgg_cands = []
            for token in reversed(parts[:-1]):
                for c in normalize_sigungu(token):
                    if c not in sgg_cands: sgg_cands.append(c)
            return sgg_cands, dong_norm

        rows = []
        for _, row in df.iterrows():
            sgg_cands, dong_norm = extract_census_keys(row['행정구역'])
            row_dict = {
                'sgg_candidates': sgg_cands,
                'primary_sgg':    sgg_cands[0] if sgg_cands else "",
                'dong_norm':      dong_norm,
                'demographic_propensity': row['demographic_propensity'],
            }
            for col in AGE_GENDER_COLS:
                row_dict[col] = row.get(col, np.nan)
            rows.append(row_dict)

        return pd.DataFrame(rows)
    except Exception as e:
        return pd.DataFrame()

def load_apt_csv(glob_pattern: str, elec_id: str) -> pd.DataFrame:
    if not glob_pattern: return pd.DataFrame()

    processed_path = f"output_data/processed_apt_data_{elec_id}.csv"

    if os.path.exists(processed_path):
        try:
            return _read_csv_auto(processed_path)
        except Exception:
            pass

    file_list = glob.glob(glob_pattern)
    if not file_list: return pd.DataFrame()

    df_list = []
    for file in file_list:
        try:
            df_list.append(_read_csv_auto(file, skiprows=15))
        except Exception: pass
    if not df_list: return pd.DataFrame()

    df = pd.concat(df_list, ignore_index=True)
    try:
        df['거래금액(만원)'] = pd.to_numeric(df['거래금액(만원)'].astype(str).str.replace(',', '').str.strip(), errors='coerce')
        df['전용면적(㎡)']  = pd.to_numeric(df['전용면적(㎡)'], errors='coerce')
        df['price_per_sqm']  = df['거래금액(만원)'] / df['전용면적(㎡)']

        def parse_loc(x):
            parts = str(x).split()
            prov  = PROV_FULL_TO_SHORT.get(parts[0], parts[0]) if parts else ""
            sgg   = normalize_sigungu(parts[1])[0] if len(parts) > 2 and normalize_sigungu(parts[1]) else ""
            dong  = normalize_dong_name(parts[-1]) if parts else ""
            return pd.Series([prov, sgg, dong])

        df[['prov','sgg','dong_norm']] = df['시군구'].apply(parse_loc)

        mask_seoul = df['prov'] == 'Seoul'
        mask_g3 = df['sgg'].isin(['강남', '서초', '송파'])
        df.loc[mask_seoul & mask_g3, 'prov'] = 'Seoul (Gangnam3gu)'
        df.loc[mask_seoul & ~mask_g3, 'prov'] = 'Seoul (Non-Gangnam3gu)'

        apt_agg = (df.groupby(['prov','sgg','dong_norm'])['price_per_sqm'].median().reset_index()
                   .rename(columns={'price_per_sqm': 'median_apt_price_sqm'}))

        apt_agg.to_csv(processed_path, index=False)
        return apt_agg
    except Exception as e:
        return pd.DataFrame()

def _avg_half_periods(df: pd.DataFrame, base_cols: list) -> pd.Series:
    available = [c for c in base_cols if c in df.columns]
    if not available: return pd.Series(np.nan, index=df.index)
    return df[available].apply(pd.to_numeric, errors='coerce').mean(axis=1)

def load_employment_sgg(election_key, **kwargs) -> pd.DataFrame:
    cfg    = ELECTION_CONFIGS[election_key if type(election_key)==str else int(election_key)]
    month  = cfg['election_month']
    year   = cfg['year']
    periods_info = get_election_periods(month, year)
    periods      = periods_info['half_periods']

    def read_emp(path):
        if not os.path.exists(path): return pd.DataFrame()
        df = _read_csv_auto(path, low_memory=False)
        return df[[c for c in df.columns if not str(c).startswith('Unnamed')]]

    age_map = {'total':'계','1529':'15 - 29세','3049':'30 - 49세',
               '5064':'50 - 64세','65plus':'65세이상'}

    def extract_gender_emp(csv_path, gender_prefix):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df_rate = df[df['항목'].str.strip() == '고용률 (%)'].copy()
        records = {}
        for suf, kor_age in age_map.items():
            sub = df_rate[df_rate['연령별'].str.strip() == kor_age].copy()
            if sub.empty: continue
            val_col = f'{gender_prefix}_{suf}'
            sub[val_col] = _avg_half_periods(sub, periods)
            prov_sgg = sub['행정구역별'].apply(parse_employment_sgg)
            sub['province_tag'] = prov_sgg.apply(lambda x: x[0])
            sub['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
            records[val_col] = sub.groupby(['province_tag','primary_sgg'])[val_col].mean()
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_men = extract_gender_emp(kwargs.get('men_csv', 'men_employment_data.csv'), 'emp_men')
    df_wmn = extract_gender_emp(kwargs.get('women_csv', 'women_employment_data.csv'), 'emp_wmn')

    def extract_occ_type(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df_total = df[df['종사상지위별'].str.strip() == '계'].copy()
        df_total['_total'] = _avg_half_periods(df_total, periods)
        prov_sgg = df_total['행정구역별'].apply(parse_employment_sgg)
        df_total['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df_total['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df_total.set_index(['province_tag','primary_sgg'])['_total']
        records = {}
        for col_name, kor_cat in {'occ_wage_share':'임금근로자', 'occ_regular_share':'- 상용근로자', 'occ_nonwage_share':'비임금근로자'}.items():
            sub = df[df['종사상지위별'].str.strip() == kor_cat].copy()
            if sub.empty: continue
            sub['_val'] = _avg_half_periods(sub, periods)
            prov_sgg2 = sub['행정구역별'].apply(parse_employment_sgg)
            sub['province_tag'] = prov_sgg2.apply(lambda x: x[0])
            sub['primary_sgg']  = prov_sgg2.apply(lambda x: x[1])
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_occ_type = extract_occ_type(kwargs.get('occ_type_csv', 'occupation_type_employment_data.csv'))

    def extract_job_shares(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df[df['항목'].str.strip() == '취업자[천명]'].copy()
        df['_val'] = _avg_half_periods(df, periods)
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df[df['직업별'].str.strip() == '계'].set_index(['province_tag','primary_sgg'])['_val']
        records = {}
        for col_name, kor_cat in {'job_professional': '관리자, 전문가 및 관련종사자', 'job_clerical': '사무 종사자', 'job_service_sales': '서비스·판매 종사자', 'job_skilled_machine':'기능·기계조작·조립 종사자', 'job_simple_labor': '단순노무 종사자', 'job_farming': '농림어업 숙련종사자'}.items():
            sub = df[df['직업별'].str.strip() == kor_cat].copy()
            if sub.empty: continue
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_job = extract_job_shares(kwargs.get('occ_csv', 'occupation_employment_data.csv'))

    def extract_ind_shares(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df[df['항목'].str.strip() == '취업자[천명]'].copy()
        df['_val'] = _avg_half_periods(df, periods)
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df[df['산업별'].str.strip() == '계'].set_index(['province_tag','primary_sgg'])['_val']
        records = {}
        for col_name, kor_cat in {'ind_agriculture': '농업, 임업 및 어업 (A)', 'ind_manufacturing': '광·제조업(B,C)', 'ind_construction': '건설업 (F) ', 'ind_retail_food': '도소매·숙박음식업(G,I)', 'ind_transport_finance': '전기·운수·통신·금융(D,H,J,K)', 'ind_services': '사업·개인·공공서비스 및 기타(E,L~U)'}.items():
            sub = df[df['산업별'].str.strip() == kor_cat.strip()].copy()
            if sub.empty:
                match = get_close_matches(kor_cat.strip(), df['산업별'].str.strip().unique().tolist(), n=1, cutoff=0.85)
                if match: sub = df[df['산업별'].str.strip() == match[0]].copy()
            if sub.empty: continue
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_ind = extract_ind_shares(kwargs.get('ind_csv', 'industry_employment_data.csv'))

    def extract_nonpart(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df.iloc[1:].copy()
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        sub_suffixes = {'total':'', 'male':'.1', 'young':'.3', 'middle':'.4'}
        period_vals  = {k: [] for k in sub_suffixes}
        for period in periods:
            for k, suf in sub_suffixes.items():
                if period + suf in df.columns: period_vals[k].append(_to_numeric_safe(df[period + suf]))
        def mean_series(lst): return pd.concat(lst, axis=1).mean(axis=1) if lst else pd.Series(np.nan, index=df.index)
        total  = mean_series(period_vals['total'])
        result = df[['province_tag','primary_sgg']].copy()
        result['nonp_male_share']   = mean_series(period_vals['male'])   / total.replace(0, np.nan)
        result['nonp_young_share']  = mean_series(period_vals['young'])  / total.replace(0, np.nan)
        result['nonp_middle_share'] = mean_series(period_vals['middle']) / total.replace(0, np.nan)
        return (result.groupby(['province_tag','primary_sgg'])[['nonp_male_share','nonp_young_share','nonp_middle_share']].mean().reset_index())

    df_nonp = extract_nonpart(kwargs.get('nonp_csv', 'non_participant_employment_data.csv'))

    merge_key = ['province_tag','primary_sgg']
    frames = [f for f in [df_men, df_wmn, df_occ_type, df_job, df_ind, df_nonp] if not f.empty]
    if not frames: return pd.DataFrame(columns=merge_key + EMPLOYMENT_FEATURE_COLS)

    df_emp = frames[0]
    for frame in frames[1:]: df_emp = df_emp.merge(frame, on=merge_key, how='outer')
    for col in EMPLOYMENT_FEATURE_COLS:
        if col not in df_emp.columns: df_emp[col] = np.nan

    if not df_emp.empty:
        mask_seoul = df_emp['province_tag'] == 'Seoul'
        mask_g3 = df_emp['primary_sgg'].isin(['강남', '서초', '송파'])
        df_emp.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
        df_emp.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    return df_emp

# ==========================================
# 2. ELECTION CSV LOADER & MATCHER
# ==========================================
def load_election_csv(csv_path: str, dem_pattern: str, con_pattern: str,
                      third_pattern: str = None, election_type: str = 'general'):
    try:
        df = _read_csv_auto(csv_path, low_memory=False)
    except Exception: return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if election_type == 'presidential':
        df = df.rename(columns={'구시군명': '선거구명', '읍면동명': '법정읍면동명'})
        special_dong_names = SPECIAL_DONG_NAMES_PRESIDENTIAL
    else:
        special_dong_names = SPECIAL_DONG_NAMES_GENERAL

    df['득표수']    = pd.to_numeric(df['득표수'], errors='coerce').fillna(0).astype(int)
    df['is_dem']   = df['후보자'].str.contains(dem_pattern,   case=False, na=False)

    if election_type == 'general':
        sejong_gap_kim = (df['선거구명'].astype(str).str.replace(' ', '').str.contains('세종.*갑', na=False)) & \
                         (df['후보자'].astype(str).str.contains('김종민', na=False))
        df.loc[sejong_gap_kim, 'is_dem'] = True

    df['is_con']   = df['후보자'].str.contains(con_pattern,   case=False, na=False)
    df['is_third'] = df['후보자'].str.contains(third_pattern, case=False, na=False) if third_pattern else False
    df['is_meta']  = df['후보자'].isin(META_CANDIDATES)
    df['is_early'] = df['투표구명'] == GWANNAESA_LABEL

    dong_key  = ['시도명','선거구명','법정읍면동명']
    const_key = ['시도명','선거구명']

    def sgg_cands(name):
        if not isinstance(name, str): return []
        if '_' in name: return normalize_sigungu(name.split('_', 1)[1])
        return normalize_sigungu(re.sub(r'[갑을병정무]$', '', name).strip())

    df_geo   = df[~df['법정읍면동명'].isin(special_dong_names)].copy()
    df_votes = df_geo[~df_geo['is_meta']].copy()

    gn_dem = df_votes[df_votes['is_dem'] & df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_dem')
    gn_tot = df_votes[df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_total')
    gn_third = df_votes[df_votes['is_third'] & df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_third')

    sd_dem = df_votes[df_votes['is_dem'] & ~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_dem')
    sd_tot = df_votes[~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_total')
    sd_third = df_votes[df_votes['is_third'] & ~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_third')

    sum_people_dong = (df_geo[~df_geo['is_early'] & (df_geo['후보자'] == '선거인수')].groupby(dong_key)['득표수'].sum().reset_index(name='sum_people'))
    sum_vote_geo    = (df_geo[df_geo['후보자'] == '투표수'].groupby(dong_key)['득표수'].sum().reset_index(name='sum_vote_geo'))

    df_dong = gn_dem.copy()
    for frame in (gn_tot, gn_third, sd_dem, sd_tot, sd_third, sum_people_dong, sum_vote_geo):
        df_dong = df_dong.merge(frame, on=dong_key, how='outer')
    df_dong = df_dong.fillna(0)

    gn_ppl_dong = (df_geo[df_geo['is_early'] & (df_geo['후보자'] == '선거인수')].groupby(dong_key)['득표수'].sum().reset_index(name='_gn_ppl'))
    df_dong = df_dong.merge(gn_ppl_dong, on=dong_key, how='left')
    df_dong['sum_people'] = df_dong['sum_people'] + df_dong['_gn_ppl'].fillna(0)
    df_dong.drop(columns=['_gn_ppl'], inplace=True)

    df_dong['sgg_candidates'] = df_dong['선거구명'].apply(sgg_cands)
    df_dong['primary_sgg']    = df_dong['sgg_candidates'].apply(lambda x: x[0] if x else "")
    df_dong['dong_norm']      = df_dong['법정읍면동명'].apply(normalize_dong_name)
    df_dong['province_tag']   = df_dong['시도명'].map(PROV_FULL_TO_SHORT).fillna(df_dong['시도명'])
    df_dong['area2_name']     = df_dong['선거구명']

    mask_seoul = df_dong['province_tag'] == 'Seoul'
    mask_g3 = df_dong['primary_sgg'].isin(['강남', '서초', '송파'])
    df_dong.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
    df_dong.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    df_gw   = df[df['법정읍면동명'].isin(special_dong_names)]
    df_gw_v = df_gw[~df_gw['is_meta']]

    go_dem_c  = df_gw_v[df_gw_v['is_dem']].groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_dem')
    go_tot_c  = df_gw_v.groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_total')
    go_turn_c = (df_gw[df_gw['후보자'] == '투표수'].groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_turnout'))

    df_const = df_dong.groupby(const_key)['sum_people'].sum().reset_index(name='sum_people')
    for frame in (go_dem_c, go_tot_c, go_turn_c):
        df_const = df_const.merge(frame, on=const_key, how='left')
    df_const = df_const.fillna(0)

    return df_dong, df_const, pd.DataFrame()

def merge_dong_with_covariates(df_election: pd.DataFrame, df_census: pd.DataFrame,
                                df_apt: pd.DataFrame, df_emp: pd.DataFrame) -> pd.DataFrame:
    if not df_census.empty:
        census_lookup = {}
        census_by_sgg = {}
        for _, row in df_census.iterrows():
            dnorm = row['dong_norm']
            covs  = {'demographic_propensity': row['demographic_propensity']}
            for c in AGE_GENDER_COLS: covs[c] = row.get(c, np.nan)
            for sgg in row['sgg_candidates']:
                census_lookup[(sgg, dnorm)] = covs
                census_by_sgg.setdefault(sgg, []).append(dnorm)

        results = []
        for _, row in df_election.iterrows():
            covs, dk, sgc = None, row['dong_norm'], row['sgg_candidates'] if isinstance(row['sgg_candidates'], list) else [row['primary_sgg']]
            if (row['primary_sgg'], dk) in census_lookup: covs = census_lookup[(row['primary_sgg'], dk)]
            if covs is None and '·' in dk:
                if (row['primary_sgg'], dk.replace('·', '')) in census_lookup: covs = census_lookup[(row['primary_sgg'], dk.replace('·', ''))]
            if covs is None:
                for sgg in sgc[1:]:
                    if (sgg, dk) in census_lookup: covs = census_lookup[(sgg, dk)]; break
            if covs is None:
                for sgg in sgc:
                    pool = census_by_sgg.get(sgg, [])
                    if pool:
                        m = get_close_matches(dk, pool, n=1, cutoff=0.82)
                        if m and (sgg, m[0]) in census_lookup: covs = census_lookup[(sgg, m[0])]; break
            if covs is None: covs = {k: np.nan for k in ['demographic_propensity'] + AGE_GENDER_COLS}
            rd = row.to_dict(); rd.update(covs)
            results.append(rd)
        df_out = pd.DataFrame(results)
    else:
        df_out = df_election.copy()

    if not df_apt.empty:
        df_out = df_out.merge(df_apt, left_on=['province_tag','primary_sgg','dong_norm'], right_on=['prov','sgg','dong_norm'], how='left')
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out.groupby('primary_sgg')['median_apt_price_sqm'].transform('median'))
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out.groupby('province_tag')['median_apt_price_sqm'].transform('median'))
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out['median_apt_price_sqm'].median())
        df_out['log_apt_price'] = np.log1p(df_out['median_apt_price_sqm'])
    else:
        df_out['log_apt_price'] = np.nan

    if not df_emp.empty:
        df_out = df_out.merge(df_emp[['province_tag','primary_sgg'] + EMPLOYMENT_FEATURE_COLS], on=['province_tag','primary_sgg'], how='left')
        for col in EMPLOYMENT_FEATURE_COLS:
            if col not in df_out.columns: df_out[col] = np.nan
            df_out[col] = df_out[col].fillna(df_out.groupby('province_tag')[col].transform('median')).fillna(df_out[col].median())
    else:
        for col in EMPLOYMENT_FEATURE_COLS: df_out[col] = np.nan

    return df_out[df_out['pct_f_4044'].notna()].copy()

def prep_election_data(df_dong_raw: pd.DataFrame, df_const_raw: pd.DataFrame,
                       df_census: pd.DataFrame, df_apt: pd.DataFrame,
                       df_emp: pd.DataFrame, df_stations: pd.DataFrame, elec_id: str) -> pd.DataFrame:
    dm = merge_dong_with_covariates(df_dong_raw, df_census, df_apt, df_emp)
    dm = dm[(dm['in_precinct_early_total'] > 50) & (dm['same_day_total'] > 50)].copy()

    no_dem = (dm['in_precinct_early_dem'] == 0) & (dm['same_day_dem'] == 0)
    if dm[no_dem].shape[0] > 0:
        dm = dm[~no_dem].copy()

    const_sums = dm.groupby(['시도명', '선거구명'])[['in_precinct_early_total', 'same_day_total']].transform('sum')
    dm['dong_weight'] = (dm['in_precinct_early_total'] + dm['same_day_total']) / (const_sums['in_precinct_early_total'] + const_sums['same_day_total']).replace(0, np.nan)

    df_const_sub = df_const_raw[['시도명', '선거구명', 'out_precinct_early_dem', 'out_precinct_early_total', 'out_precinct_early_turnout']].drop_duplicates()
    dm = dm.merge(df_const_sub, on=['시도명', '선거구명'], how='left')

    dm['out_precinct_alloc_dem']     = dm['out_precinct_early_dem'].fillna(0) * dm['dong_weight'].fillna(0)
    dm['out_precinct_alloc_tot']     = dm['out_precinct_early_total'].fillna(0) * dm['dong_weight'].fillna(0)
    dm['out_precinct_alloc_turnout'] = dm['out_precinct_early_turnout'].fillna(0) * dm['dong_weight'].fillna(0)

    dm['vote_share'] = ((dm['in_precinct_early_dem'] + dm['same_day_dem'] + dm['out_precinct_alloc_dem']) /
                        (dm['in_precinct_early_total'] + dm['same_day_total'] + dm['out_precinct_alloc_tot'])).fillna(0)
    dm['turnout'] = (dm['sum_vote_geo'] + dm['out_precinct_alloc_turnout']) / (dm['sum_people'] + dm['out_precinct_alloc_turnout']).replace(0, np.nan)
    dm['early_turnout'] = (dm['in_precinct_early_total'] + dm['out_precinct_alloc_tot']) / (dm['sum_people'] + dm['out_precinct_alloc_turnout']).replace(0, np.nan)
    dm['early_dem_share'] = (dm['in_precinct_early_dem'] + dm['out_precinct_alloc_dem']) / (dm['in_precinct_early_total'] + dm['out_precinct_alloc_tot']).replace(0, np.nan)
    dm['third_vote_share'] = ((dm['in_precinct_early_third'] + dm['same_day_third']) / (dm['in_precinct_early_total'] + dm['same_day_total'])).fillna(0)

    # ---------------------------------------------------------
    # INSTRUMENT INTEGRATION: voters_per_station
    # ---------------------------------------------------------
    if not df_stations.empty:
        dm = dm.merge(df_stations, on=['province_tag', 'primary_sgg', 'dong_norm'], how='left')
        dm['normal_station_count'] = dm['normal_station_count'].fillna(1)
    else:
        dm['normal_station_count'] = 1

    dm['voters_per_station'] = dm['sum_people'] / dm['normal_station_count']

    # Strict Local Demeaning (Province + Primary SGG Level)
    dm['voters_per_station_dm'] = dm['voters_per_station'] - dm.groupby(['province_tag', 'primary_sgg'])['voters_per_station'].transform('mean')

    if dm['voters_per_station_dm'].std(skipna=True) > 0:
        dm['voters_per_station_dm'] = StandardScaler().fit_transform(dm[['voters_per_station_dm']].fillna(0))
    else:
        dm['voters_per_station_dm'] = 0.0

    dm['election_id'] = str(elec_id)
    return dm

# ==========================================
# 4. PCA COLLINEARITY PIPELINE
# ==========================================
def reduce_pca(df: pd.DataFrame, feature_cols: list, threshold: float, name_label: str) -> tuple:
    available = [c for c in feature_cols if c in df.columns and df[c].notna().mean() >= MIN_COVERAGE]
    if len(available) < 2: return df.copy(), [], None, available

    valid_mask = df[available].notna().all(axis=1)
    if valid_mask.sum() < 30: return df.copy(), [], None, available

    X = df.loc[valid_mask, available].values.astype(float)
    X_scaled = StandardScaler().fit_transform(X)

    pca_probe = PCA().fit(X_scaled)
    cumvar = np.cumsum(pca_probe.explained_variance_ratio_)
    n_components = min(int(np.searchsorted(cumvar, threshold)) + 1, len(available))

    pca_final = PCA(n_components=n_components)
    components = pca_final.fit_transform(X_scaled)

    pc_cols = [f'pc{i+1}' for i in range(n_components)]
    df_out = df.copy()
    for col in pc_cols: df_out[col] = np.nan
    df_out.loc[valid_mask, pc_cols] = components

    loadings = pd.DataFrame(pca_final.components_.T, columns=pc_cols, index=available)
    out_path = f'output_data/pca_loadings_{name_label}.csv'
    loadings.to_csv(out_path)
    log_to_file(f"  -> Saved PCA Loadings for {name_label} to {out_path}")

    return df_out, pc_cols, pca_final, available

def prepare_continuous_covariates(df: pd.DataFrame, raw_age_cols: list, raw_emp_cols: list, label: str) -> tuple:
    combined_raw = list(raw_age_cols) + list(raw_emp_cols)
    if 'log_apt_price' in df.columns: combined_raw.append('log_apt_price')

    df, pc_cols, pca_model, used_features = reduce_pca(df, combined_raw, PCA_VARIANCE_THRESHOLD, label)
    pc_cols = [c for c in pc_cols if df[c].notna().mean() >= MIN_COVERAGE]

    dm_cols = []
    if pc_cols:
        active_groups = [g for g in DEMEAN_GROUP_COLS if g in df.columns]
        for col in pc_cols:
            df[f'{col}_dm'] = df[col] - df.groupby(active_groups)[col].transform('mean')
            dm_cols.append(f'{col}_dm')

    dm_cols = [c for c in dm_cols if df[c].std(skipna=True) > 1e-10]

    if dm_cols:
        valid_mask = df[dm_cols].notna().all(axis=1)
        if valid_mask.sum() > len(dm_cols) + 5:
            df.loc[valid_mask, dm_cols] = StandardScaler().fit_transform(df.loc[valid_mask, dm_cols])

    return df, dm_cols, pca_model, used_features


# ==========================================
# 5. REGRESSIONS (WLS & IV)
# ==========================================
def run_single_analysis(df_single: pd.DataFrame, elec_id, dm_cols: list):
    label = f"Election_{elec_id}"
    header = f"\n{'='*70}\n  SINGLE ELECTION ANALYSIS: {label}\n{'='*70}"
    log_to_file(header)

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share', 'sum_people',
                'area2_name', 'province_tag', 'voters_per_station_dm'] + dm_cols
    df_sub = df_single.dropna(subset=req_cols).copy()

    df_sub.to_csv(f'output_data/regression_data_{label}{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    if df_sub.empty:
        log_to_file(f" [!] Not enough data to run analysis for {label}.")
        return

    exog_vars = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag)"

    # 1. Turnout Regressions
    for target in ['turnout', 'early_turnout']:
        log_to_file(f"\n--- Sub-Target: {target} ---")
        exog_str = exog_vars + " + voters_per_station_dm"

        formula = f"{target} ~ {exog_str}"
        model = smf.wls(formula, data=df_sub, weights=df_sub['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_sub['area2_name']})
        log_to_file(f"[Formula]: {formula}")
        log_to_file(model.summary().as_text())

    # 2. Total Dem Vote Share & Early Dem Share Regressions
    for main_target in ['vote_share', 'early_dem_share']:
        turnout_vars = ['early_turnout', 'turnout'] if INCLUDE_TURNOUT_IN_VOTESHARE else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (Instrumenting {turnout_var})\n{'='*55}")
                formula_vs = f"{main_target} ~ {exog_vars} + [{turnout_var} ~ voters_per_station_dm]"
                model_vs = IV2SLS.from_formula(formula_vs, df_sub, weights=df_sub['sum_people'])
                res_vs = model_vs.fit(cov_type='robust')

                log_to_file(f"\n[Formula]: {formula_vs}")
                if hasattr(res_vs, 'first_stage') and res_vs.first_stage is not None:
                    log_to_file("\n[First Stage F-Stats]")
                    for endog in res_vs.first_stage.diagnostics.index:
                        f_stat = res_vs.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file("\n[Second Stage Summary]")
                log_to_file(res_vs.summary.as_text())
            else:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (No Turnout Covariate)\n{'='*55}")
                formula_vs = f"{main_target} ~ {exog_vars}"
                model_vs = smf.wls(formula_vs, data=df_sub, weights=df_sub['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_sub['area2_name']})
                log_to_file(f"\n[Formula]: {formula_vs}")
                log_to_file(model_vs.summary().as_text())


def run_iv_causal_analysis(df_pres21: pd.DataFrame, df_pres19: pd.DataFrame, dm_cols: list):
    header = f"\n{'='*70}\n  EXACTLY IDENTIFIED IV (2SLS) ANALYSIS (21st Pres)\n{'='*70}"
    log_to_file(header)

    instrument_df = df_pres19[['province_tag', 'primary_sgg', 'dong_norm', 'third_vote_share']].drop_duplicates()
    instrument_df = instrument_df.rename(columns={'third_vote_share': 'yoo_vote_share_19'})

    df_iv = df_pres21.rename(columns={'third_vote_share': 'LJS_vote_share'})
    df_iv = df_iv.merge(instrument_df, on=['province_tag', 'primary_sgg', 'dong_norm'], how='inner')

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share',
                'LJS_vote_share', 'yoo_vote_share_19', 'sum_people', 'voters_per_station_dm', 'area2_name'] + dm_cols
    df_iv = df_iv.dropna(subset=req_cols).copy()

    df_iv.to_csv(f'output_data/regression_data_iv_pres21{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    exog_vars = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag)"

    # 1. Turnout Regressions
    for target in ['turnout', 'early_turnout']:
        log_to_file(f"\n\n{'='*55}\n Target: {target}\n{'='*55}")

        exog_str = exog_vars + " + voters_per_station_dm"

        if INCLUDE_LJS_IN_VOTESHARE:
            formula = f"{target} ~ {exog_str} + [LJS_vote_share ~ yoo_vote_share_19]"
            res = IV2SLS.from_formula(formula, df_iv, weights=df_iv['sum_people']).fit(cov_type='robust')
            log_to_file(f"\n[Formula]: {formula}")

            if hasattr(res, 'first_stage') and res.first_stage is not None:
                log_to_file("\n[First Stage F-Stats]")
                for endog in res.first_stage.diagnostics.index:
                    f_stat = res.first_stage.diagnostics.loc[endog, 'f.stat']
                    log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
            log_to_file("\n[Second Stage Summary]")
            log_to_file(res.summary.as_text())
        else:
            formula = f"{target} ~ {exog_str}"
            res = smf.wls(formula, data=df_iv, weights=df_iv['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_iv['area2_name']})
            log_to_file(f"\n[Formula]: {formula}")
            log_to_file(res.summary().as_text())

    # 2. Vote Share & Early Dem Share Regressions (Dynamic string building based on toggles)
    for main_target in ['vote_share', 'early_dem_share']:
        turnout_vars = ['early_turnout', 'turnout'] if INCLUDE_TURNOUT_IN_VOTESHARE else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (Turnout Covariate: {turnout_var})\n{'='*55}")
            else:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (No Turnout Covariate)\n{'='*55}")

            endogs, instrs = [], []

            if INCLUDE_LJS_IN_VOTESHARE:
                endogs.append('LJS_vote_share')
                instrs.append('yoo_vote_share_19')
            if INCLUDE_TURNOUT_IN_VOTESHARE and turnout_var:
                endogs.append(turnout_var)
                instrs.append('voters_per_station_dm')

            if endogs:
                formula = f"{main_target} ~ {exog_vars} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res = IV2SLS.from_formula(formula, df_iv, weights=df_iv['sum_people']).fit(cov_type='robust')

                log_to_file(f"\n[Formula]: {formula}")
                if hasattr(res, 'first_stage') and res.first_stage is not None:
                    log_to_file("\n[First Stage F-Stats]")
                    for endog in res.first_stage.diagnostics.index:
                        f_stat = res.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file("\n[Second Stage Summary]")
                log_to_file(res.summary.as_text())
            else:
                formula = f"{main_target} ~ {exog_vars}"
                res = smf.wls(formula, data=df_iv, weights=df_iv['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_iv['area2_name']})
                log_to_file(f"\n[Formula]: {formula}")
                log_to_file(res.summary().as_text())

    return df_iv


def run_pooled_iv_panel(df_pooled: pd.DataFrame, df_pres19: pd.DataFrame, dm_cols: list):
    header = f"\n{'='*70}\n  POOLED IV PANEL REGRESSIONS\n{'='*70}"
    log_to_file(header)

    df_panel = df_pooled.copy()

    inst_df = df_pres19[['province_tag', 'primary_sgg', 'dong_norm', 'third_vote_share']].drop_duplicates()
    inst_df = inst_df.rename(columns={'third_vote_share': 'base_yoo_pr'})

    df_panel = df_panel.merge(inst_df, on=['province_tag', 'primary_sgg', 'dong_norm'], how='inner')

    df_panel['LJS_vote_share'] = np.where(df_panel['election_id'] == 'pres21', df_panel['third_vote_share'], 0)
    df_panel['active_yoo_pr'] = np.where(df_panel['election_id'] == 'pres21', df_panel['base_yoo_pr'], 0)
    df_panel['prov_elec'] = df_panel['province_tag'] + "_" + df_panel['election_id'].astype(str)

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share',
                'LJS_vote_share', 'active_yoo_pr', 'sum_people', 'election_id',
                'province_tag', 'prov_elec', 'voters_per_station_dm', 'area2_name'] + dm_cols
    df_panel = df_panel.dropna(subset=req_cols).copy()

    df_panel.to_csv(f'output_data/regression_data_pooled_panel{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    exog_vars_add = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag) + C(election_id)"
    exog_vars_int = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(prov_elec)"

    for target in ['turnout', 'early_turnout', 'early_dem_share', 'vote_share']:
        log_to_file(f"\n\n{'='*55}\n PANEL TARGET: {target}\n{'='*55}")

        exog_add_base = exog_vars_add
        exog_int_base = exog_vars_int

        if target in ['turnout', 'early_turnout']:
            exog_add_base += " + voters_per_station_dm"
            exog_int_base += " + voters_per_station_dm"

        turnout_vars = ['early_turnout', 'turnout'] if (target in ['vote_share', 'early_dem_share'] and INCLUDE_TURNOUT_IN_VOTESHARE) else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n--- Sub-Target for {target} (Turnout Covariate: {turnout_var}) ---")

            endogs, instrs = [], []

            if INCLUDE_LJS_IN_VOTESHARE:
                endogs.append('LJS_vote_share')
                instrs.append('active_yoo_pr')

            if target in ['vote_share', 'early_dem_share'] and INCLUDE_TURNOUT_IN_VOTESHARE and turnout_var:
                endogs.append(turnout_var)
                instrs.append('voters_per_station_dm')

            # Execute Spec 1 (Additive)
            log_to_file(f"\n[Spec 1: Additive Province & Election FEs]")
            if endogs:
                formula_add = f"{target} ~ {exog_add_base} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res_add = IV2SLS.from_formula(formula_add, df_panel, weights=df_panel['sum_people']).fit(cov_type='robust')
                log_to_file(f"[Formula]: {formula_add}")
                if hasattr(res_add, 'first_stage') and res_add.first_stage is not None:
                    log_to_file("[First Stage F-Stats]")
                    for endog in res_add.first_stage.diagnostics.index:
                        f_stat = res_add.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file(res_add.summary.as_text())
            else:
                formula_add = f"{target} ~ {exog_add_base}"
                res_add = smf.wls(formula_add, data=df_panel, weights=df_panel['sum_people']).fit(cov_type='HC3')
                log_to_file(f"[Formula]: {formula_add}")
                log_to_file(res_add.summary().as_text())

            # Execute Spec 2 (Interacted)
            log_to_file(f"\n[Spec 2: Interacted Province x Election FEs]")
            if endogs:
                formula_int = f"{target} ~ {exog_int_base} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res_int = IV2SLS.from_formula(formula_int, df_panel, weights=df_panel['sum_people']).fit(cov_type='robust')
                log_to_file(f"[Formula]: {formula_int}")
                if hasattr(res_int, 'first_stage') and res_int.first_stage is not None:
                    log_to_file("[First Stage F-Stats]")
                    for endog in res_int.first_stage.diagnostics.index:
                        f_stat = res_int.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file(res_int.summary.as_text())
            else:
                formula_int = f"{target} ~ {exog_int_base}"
                res_int = smf.wls(formula_int, data=df_panel, weights=df_panel['sum_people']).fit(cov_type='HC3')
                log_to_file(f"[Formula]: {formula_int}")
                log_to_file(res_int.summary().as_text())


# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    with open(LOG_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write("COMPREHENSIVE REGRESSION & DATA LOG\n")
        f.write("="*60 + "\n")

    all_dm = []

    for elec_id in ['pres19', 21, 22, 'pres20', 'pres21']:
        cfg = ELECTION_CONFIGS[elec_id]
        log_to_file(f"\n--- Loading Data for {cfg['label']} ---")

        df_census = load_census_csv(cfg.get('census_csv'))
        df_apt = load_apt_csv(cfg.get('apt_csv_glob'), elec_id)
        df_emp = load_employment_sgg(election_key=elec_id)
        df_stations = load_polling_stations(elec_id)

        log_to_file(f"  > Census Data: {'Loaded successfully' if not df_census.empty else 'NO DATA FOUND'}")

        apt_cache_path = f"output_data/processed_apt_data_{elec_id}.csv"
        if not df_apt.empty:
            if os.path.exists(apt_cache_path) and os.path.getmtime(apt_cache_path) > 0:
                log_to_file(f"  > APT Price Data: Loaded from Cache ({apt_cache_path})")
            else:
                log_to_file(f"  > APT Price Data: Parsed from Raw and Saved to Cache")
        else:
            log_to_file(f"  > APT Price Data: NO DATA FOUND")

        log_to_file(f"  > Employment Data: {'Loaded successfully' if not df_emp.empty else 'NO DATA FOUND'}")
        log_to_file(f"  > Polling Stations: {'Loaded successfully' if not df_stations.empty else 'NO DATA FOUND'}")

        df_dong, df_const, _ = load_election_csv(
            cfg['result_csv'],
            dem_pattern=cfg['dem_pattern'],
            con_pattern=cfg['con_pattern'],
            third_pattern=cfg.get('third_pattern'),
            election_type=cfg['election_type'],
        )

        if EXCLUDE_HONAM_YEONGNAM:
            exclude_provs = PARTISAN_REGION_PROVINCES['Honam'] | PARTISAN_REGION_PROVINCES['Yeongnam']
            short_exclude = {PROV_FULL_TO_SHORT.get(p, p) for p in exclude_provs} | exclude_provs
            df_dong  = df_dong[~df_dong['province_tag'].isin(short_exclude)]
            df_const = df_const[~df_const['province_tag'].isin(short_exclude)]

        if not df_dong.empty and not df_const.empty:
            dm = prep_election_data(df_dong, df_const, df_census, df_apt, df_emp, df_stations, elec_id)
            all_dm.append(dm)
            log_to_file(f"  > Processed {len(dm)} distinct dong-level observations.")

    df_raw_pooled = pd.concat(all_dm, ignore_index=True)
    raw_age_cols = [c for c in AGE_GENDER_COLS if c in df_raw_pooled.columns]
    raw_emp_cols = [c for c in EMPLOYMENT_FEATURE_COLS if c in df_raw_pooled.columns]

    # ---------------------------------------------------------
    # PART A: ISOLATED SINGLE-ELECTION PIPELINE
    # ---------------------------------------------------------
    log_to_file("\n\n" + "#"*70 + "\n PART A: ISOLATED SINGLE-ELECTION PIPELINE\n" + "#"*70)

    for elec in [21, 22, 'pres20']:
        df_single_raw = df_raw_pooled[df_raw_pooled['election_id'] == str(elec)].copy()
        if not df_single_raw.empty:
            df_single_pca, dm_cols_single, _, _ = prepare_continuous_covariates(df_single_raw, raw_age_cols, raw_emp_cols, label=f"isolated_{elec}")
            run_single_analysis(df_single_pca, elec, dm_cols_single)

    df_pres21_raw = df_raw_pooled[df_raw_pooled['election_id'] == 'pres21'].copy()
    df_pres19_raw = df_raw_pooled[df_raw_pooled['election_id'] == 'pres19'].copy()

    if not df_pres21_raw.empty and not df_pres19_raw.empty:
        df_pres21_pca, dm_cols_iv, _, _ = prepare_continuous_covariates(df_pres21_raw, raw_age_cols, raw_emp_cols, label="isolated_pres21")
        run_iv_causal_analysis(df_pres21_pca, df_pres19_raw, dm_cols_iv)

    # ---------------------------------------------------------
    # PART B: POOLED ELECTION ANALYSIS
    # ---------------------------------------------------------
    log_to_file("\n\n" + "#"*70 + "\n PART B: POOLED PANEL ELECTION PIPELINE\n" + "#"*70)

    df_pooled_pca, dm_cols_pooled, _, _ = prepare_continuous_covariates(df_raw_pooled, raw_age_cols, raw_emp_cols, label="pooled_panel")

    if not df_pres19_raw.empty:
        run_pooled_iv_panel(df_pooled_pca, df_pres19_raw, dm_cols_pooled)

    print(f"\n[+] All tasks completed. Full statistical logs exported to -> {LOG_FILE_PATH}")


--- Loading Data for 19th Presidential Election (2017) ---


/tmp/ipykernel_4025/3691552743.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: NO DATA FOUND
  > Employment Data: Loaded successfully
  > Polling Stations: NO DATA FOUND
  > Processed 3537 distinct dong-level observations.

--- Loading Data for 21st General Election (2020) ---


/tmp/ipykernel_4025/3691552743.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_21.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 5519 distinct dong-level observations.

--- Loading Data for 22nd General Election (2024) ---


...

  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_pres21.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 6050 distinct dong-level observations.


######################################################################
 PART A: ISOLATED SINGLE-ELECTION PIPELINE
######################################################################
  -> Saved PCA Loadings for isolated_21 to output_data/pca_loadings_isolated_21.csv

  SINGLE ELECTION ANALYSIS: Election_21

--- Sub-Target: turnout ---
[Formula]: turnout ~ pc1_dm + pc2_dm + pc3_dm + pc4_dm + pc5_dm + pc6_dm + pc7_dm + pc8_dm + pc9_dm + pc10_dm + pc11_dm + C(province_tag) + voters_per_station_dm
                            WLS Regression Results                            
Dep. Variable:                turnout   R-squared:                       0.291
Model:                            WLS   Adj. R-squared:                  0.288
Metho

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 84, but rank is 81
  warnings.warn('covariance of constraints does not have full '


[Formula]: early_turnout ~ pc1_dm + pc2_dm + pc3_dm + pc4_dm + pc5_dm + pc6_dm + pc7_dm + pc8_dm + pc9_dm + pc10_dm + pc11_dm + pc12_dm + C(province_tag) + C(election_id) + voters_per_station_dm
                            WLS Regression Results                            
Dep. Variable:          early_turnout   R-squared:                       0.514
Model:                            WLS   Adj. R-squared:                  0.514
Method:                 Least Squares   F-statistic:                     89.79
Date:                Fri, 27 Mar 2026   Prob (F-statistic):          4.35e-166
Time:                        18:27:10   Log-Likelihood:                 83639.
No. Observations:               55517   AIC:                        -1.672e+05
Df Residuals:                   55483   BIC:                        -1.669e+05
Df Model:                          33                                         
Covariance Type:              cluster                                         
               

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 84, but rank is 81
  warnings.warn('covariance of constraints does not have full '


[Formula]: early_dem_share ~ pc1_dm + pc2_dm + pc3_dm + pc4_dm + pc5_dm + pc6_dm + pc7_dm + pc8_dm + pc9_dm + pc10_dm + pc11_dm + pc12_dm + C(province_tag) + C(election_id) + [early_turnout ~ voters_per_station_dm]
[First Stage F-Stats]
  early_turnout ~ Instruments F-Stat: 233.56
                          IV-2SLS Estimation Summary                          
Dep. Variable:        early_dem_share   R-squared:                      0.8543
Estimator:                    IV-2SLS   Adj. R-squared:                 0.8542
No. Observations:               55517   F-statistic:                 1.256e+07
Date:                Fri, Mar 27 2026   P-value (F-stat)                0.0000
Time:                        18:27:14   Distribution:                 chi2(34)
Cov. Estimator:                robust                                         
                                                                              
                                            Parameter Estimates                       

NONE of LJS vote share or turnout (either total or early) covariates turned on in democratic vote share regressions

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import glob
from difflib import get_close_matches
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.formula.api as smf

try:
    from linearmodels.iv import IV2SLS
except ImportError:
    print("[!] 'linearmodels' not found. Please run: pip install linearmodels")
    exit(1)

# ==========================================
# CONFIGURATION & GLOBAL SETUP
# ==========================================

# --- GLOBAL TOGGLES ---
EXCLUDE_HONAM_YEONGNAM       = False
INCLUDE_TURNOUT_IN_VOTESHARE = False   # Toggle Turnout IV in Vote Share Regressions
INCLUDE_LJS_IN_VOTESHARE     = False   # Toggle LJS Vote Share IV in relevant Regressions

# --- DYNAMIC FILENAME SUFFIX ---
if INCLUDE_LJS_IN_VOTESHARE and INCLUDE_TURNOUT_IN_VOTESHARE:
    IV_SUFFIX = "_LJS_turnout_IV_cov"
elif INCLUDE_LJS_IN_VOTESHARE:
    IV_SUFFIX = "_LJS_IV_cov"
elif INCLUDE_TURNOUT_IN_VOTESHARE:
    IV_SUFFIX = "_turnout_IV_cov"
else:
    IV_SUFFIX = "_no_IV_cov"

os.makedirs('output_data', exist_ok=True)
LOG_FILE_PATH = f'output_data/comprehensive_regression_logs{IV_SUFFIX}.txt'

PCA_VARIANCE_THRESHOLD = 0.90
DEMEAN_GROUP_COLS      = ['province_tag']
MIN_COVERAGE           = 0.50

ELECTION_CONFIGS = {
    'pres19': {
        'election_type':  'presidential',
        'census_csv':     '19th_presidential_election_census.csv',
        'result_csv':     '19th_presidential_election_result.csv',
        'apt_csv_glob':   None,
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'자유한국당',
        'third_pattern':  r'유승민|바른정당',
        'label':          '19th Presidential Election (2017)',
        'year':           2017,
        'election_month': 5,
    },
    21: {
        'election_type':  'general',
        'census_csv':     '21st_election_census.csv',
        'result_csv':     '21st_election_result.csv',
        'apt_csv_glob':   '*21st_election_*_apt_price.csv',
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'미래통합당|자유한국당',
        'third_pattern':  None,
        'label':          '21st General Election (2020)',
        'year':           2020,
        'election_month': 4,
    },
    22: {
        'election_type':  'general',
        'census_csv':     '22nd_election_census.csv',
        'result_csv':     '22nd_election_result.csv',
        'apt_csv_glob':   '*22nd_election_*_apt_price.csv',
        'dem_pattern':    r'더불어민주당',
        'con_pattern':    r'국민의힘',
        'third_pattern':  r'개혁신당',
        'label':          '22nd General Election (2024)',
        'year':           2024,
        'election_month': 4,
    },
    'pres20': {
        'election_type':         'presidential',
        'census_csv':            '20th_presidential_election_census.csv',
        'result_csv':            '20th_presidential_election_result.csv',
        'apt_csv_glob':          '*20th_presidential_election_*_apt_prices.csv',
        'dem_pattern':           r'더불어민주당',
        'con_pattern':           r'국민의힘',
        'third_pattern':         None,
        'label':                 '20th Presidential Election (2022)',
        'year':                  2022,
        'election_month':        3,
    },
    'pres21': {
        'election_type':         'presidential',
        'census_csv':            '21st_presidential_election_census.csv',
        'result_csv':            '21st_presidential_election_result.csv',
        'apt_csv_glob':          '*21st_presidential_election_*_apt_prices.csv',
        'dem_pattern':           r'더불어민주당',
        'con_pattern':           r'국민의힘',
        'third_pattern':         r'이준석|개혁신당',
        'label':                 '21st Presidential Election (2025)',
        'year':                  2025,
        'election_month':        6,
    },
}

SPECIAL_DONG_NAMES_GENERAL = {
    '거소·선상투표', '관외사전투표', '국외부재자투표',
    '국외부재자투표(공관)', '잘못 투입·구분된 투표지',
}
SPECIAL_DONG_NAMES_PRESIDENTIAL = {
    '거소·선상투표', '관외사전투표', '재외투표',
    '잘못 투입·구분된 투표지',
}
GWANNAESA_LABEL = '관내사전투표'
META_CANDIDATES = {'선거인수', '투표수', '무효 투표수', '기권자수'}

PROV_FULL_TO_SHORT = {
    '서울특별시': 'Seoul',  '부산광역시': 'Busan',   '대구광역시': 'Daegu',
    '인천광역시': 'Incheon','광주광역시': 'Gwangju', '대전광역시': 'Daejeon',
    '울산광역시': 'Ulsan',  '세종특별자치시': 'Sejong',
    '경기도': 'Gyeonggi',  '강원도': 'Gangwon',     '강원특별자치도': 'Gangwon',
    '충청북도': 'Chungbuk', '충청남도': 'Chungnam',
    '전라북도': 'Jeonbuk',  '전북특별자치도': 'Jeonbuk', '전라남도': 'Jeonnam',
    '경상북도': 'Gyeongbuk','경상남도': 'Gyeongnam', '제주특별자치도': 'Jeju',
    '서울': 'Seoul',   '부산': 'Busan',    '대구': 'Daegu',
    '인천': 'Incheon', '광주': 'Gwangju',  '대전': 'Daejeon',
    '울산': 'Ulsan',
}

SGG_TO_PROVINCE_EMP = {
    '가평군':'Gyeonggi','고양시':'Gyeonggi','과천시':'Gyeonggi','광명시':'Gyeonggi',
    '광주시':'Gyeonggi','구리시':'Gyeonggi','군포시':'Gyeonggi','김포시':'Gyeonggi',
    '남양주시':'Gyeonggi','동두천시':'Gyeonggi','부천시':'Gyeonggi','성남시':'Gyeonggi',
    '수원시':'Gyeonggi','시흥시':'Gyeonggi','안산시':'Gyeonggi','안성시':'Gyeonggi',
    '안양시':'Gyeonggi','양주시':'Gyeonggi','양평군':'Gyeonggi','여주군':'Gyeonggi',
    '여주시':'Gyeonggi','연천군':'Gyeonggi','오산시':'Gyeonggi','용인시':'Gyeonggi',
    '의왕시':'Gyeonggi','의정부시':'Gyeonggi','이천시':'Gyeonggi','파주시':'Gyeonggi',
    '평택시':'Gyeonggi','포천시':'Gyeonggi','하남시':'Gyeonggi','화성시':'Gyeonggi',
    '강릉시':'Gangwon','고성군':'Gangwon','동해시':'Gangwon','삼척시':'Gangwon',
    '속초시':'Gangwon','양구군':'Gangwon','양양군':'Gangwon','영월군':'Gangwon',
    '원주시':'Gangwon','인제군':'Gangwon','정선군':'Gangwon','철원군':'Gangwon',
    '춘천시':'Gangwon','태백시':'Gangwon','평창군':'Gangwon','홍천군':'Gangwon',
    '화천군':'Gangwon','횡성군':'Gangwon','괴산군':'Chungbuk','단양군':'Chungbuk',
    '보은군':'Chungbuk','영동군':'Chungbuk','옥천군':'Chungbuk','음성군':'Chungbuk',
    '제천시':'Chungbuk','증평군':'Chungbuk','진천군':'Chungbuk','청원군':'Chungbuk',
    '청주시':'Chungbuk','충주시':'Chungbuk','계룡시':'Chungnam','공주시':'Chungnam',
    '금산군':'Chungnam','논산시':'Chungnam','당진시':'Chungnam','보령시':'Chungnam',
    '부여군':'Chungnam','서산시':'Chungnam','서천군':'Chungnam','아산시':'Chungnam',
    '연기군':'Chungnam','예산군':'Chungnam','청양군':'Chungnam','천안시':'Chungnam',
    '태안군':'Chungnam','홍성군':'Chungnam','고창군':'Jeonbuk','군산시':'Jeonbuk',
    '김제시':'Jeonbuk','남원시':'Jeonbuk','무주군':'Jeonbuk','부안군':'Jeonbuk',
    '순창군':'Jeonbuk','완주군':'Jeonbuk','익산시':'Jeonbuk','임실군':'Jeonbuk',
    '장수군':'Jeonbuk','전주시':'Jeonbuk','정읍시':'Jeonbuk','진안군':'Jeonbuk',
    '강진군':'Jeonnam','고흥군':'Jeonnam','곡성군':'Jeonnam','구례군':'Jeonnam',
    '나주시':'Jeonnam','담양군':'Jeonnam','목포시':'Jeonnam','무안군':'Jeonnam',
    '보성군':'Jeonnam','순천시':'Jeonnam','신안군':'Jeonnam','여수시':'Jeonnam',
    '영광군':'Jeonnam','영암군':'Jeonnam','완도군':'Jeonnam','장성군':'Jeonnam',
    '장흥군':'Jeonnam','진도군':'Jeonnam','함평군':'Jeonnam','화순군':'Jeonnam',
    '광양시':'Jeonnam','해남군':'Jeonnam','경산시':'Gyeongbuk','경주시':'Gyeongbuk',
    '고령군':'Gyeongbuk','구미시':'Gyeongbuk','김천시':'Gyeongbuk','문경시':'Gyeongbuk',
    '봉화군':'Gyeongbuk','상주시':'Gyeongbuk','성주군':'Gyeongbuk','안동시':'Gyeongbuk',
    '영덕군':'Gyeongbuk','영양군':'Gyeongbuk','영주시':'Gyeongbuk','영천시':'Gyeongbuk',
    '예천군':'Gyeongbuk','울릉군':'Gyeongbuk','울진군':'Gyeongbuk','의성군':'Gyeongbuk',
    '청도군':'Gyeongbuk','청송군':'Gyeongbuk','칠곡군':'Gyeongbuk','포항시':'Gyeongbuk',
    '거제시':'Gyeongnam','거창군':'Gyeongnam','김해시':'Gyeongnam','남해군':'Gyeongnam',
    '밀양시':'Gyeongnam','사천시':'Gyeongnam','산청군':'Gyeongnam','양산시':'Gyeongnam',
    '의령군':'Gyeongnam','진주시':'Gyeongnam','창녕군':'Gyeongnam','창원시':'Gyeongnam',
    '통영시':'Gyeongnam','하동군':'Gyeongnam','함안군':'Gyeongnam','함양군':'Gyeongnam',
    '합천군':'Gyeongnam','서귀포시':'Jeju','제주시':'Jeju',
}

PARTISAN_REGION_PROVINCES = {
    'Honam':   {'Jeonnam', 'Jeonbuk', 'Gwangju', '전남', '전북', '광주'},
    'Yeongnam':{'Gyeongbuk', 'Gyeongnam', 'Daegu', 'Busan', 'Ulsan', '경북', '경남', '대구', '부산', '울산'},
}

AGE_GENDER_COLS = [
    'pct_m_1824', 'pct_m_2529', 'pct_m_3034', 'pct_m_3539', 'pct_m_4044', 'pct_m_4549',
    'pct_m_5054', 'pct_m_5559', 'pct_m_6064', 'pct_m_6569', 'pct_m_70plus',
    'pct_f_1824', 'pct_f_2529', 'pct_f_3034', 'pct_f_3539', 'pct_f_4044', 'pct_f_4549',
    'pct_f_5054', 'pct_f_5559', 'pct_f_6064', 'pct_f_6569', 'pct_f_70plus',
]

EMPLOYMENT_FEATURE_COLS = [
    'emp_men_total', 'emp_men_1529', 'emp_men_3049', 'emp_men_5064', 'emp_men_65plus',
    'emp_wmn_total', 'emp_wmn_1529', 'emp_wmn_3049', 'emp_wmn_5064', 'emp_wmn_65plus',
    'occ_wage_share', 'occ_regular_share', 'occ_nonwage_share',
    'job_professional', 'job_clerical', 'job_service_sales',
    'job_skilled_machine', 'job_simple_labor', 'job_farming',
    'ind_agriculture', 'ind_manufacturing', 'ind_construction',
    'ind_retail_food', 'ind_transport_finance', 'ind_services',
    'nonp_male_share', 'nonp_young_share', 'nonp_middle_share', 'nonp_old_share',
]

# ==========================================
# FILE WRITER HELPER
# ==========================================
def log_to_file(text: str, mode='a'):
    print(text)
    with open(LOG_FILE_PATH, mode, encoding='utf-8') as f:
        f.write(text + "\n")

# ==========================================
# NORMALISATION & IO UTILITIES
# ==========================================
def _read_csv_auto(path: str, **kwargs) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding='utf-8', **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='cp949', **kwargs)

def _to_numeric_safe(series: pd.Series) -> pd.Series:
    return (series.astype(str).str.strip()
            .replace(['-', '.', '', 'nan'], np.nan)
            .pipe(pd.to_numeric, errors='coerce'))

def normalize_dong_name(name: str) -> str:
    if not isinstance(name, str): return ""
    name = re.sub(r'\(.*?\)', '', name).strip().replace('.', '·')
    name = re.sub(r'제(\d)', r'\1', name)
    name = re.sub(r'·\d+', '', name)
    name = re.sub(r'(\d+)(동|읍|면)$', r'\2', name)
    return re.sub(r'\s+', ' ', name)

def split_admin_tokens(name: str) -> list:
    tokens, buf = [], []
    for ch in name:
        buf.append(ch)
        if ch in '시군구' and len(buf) >= 2:
            tokens.append(''.join(buf))
            buf = []
    if buf: tokens.append(''.join(buf))
    return [t for t in tokens if t]

def normalize_sigungu(name: str) -> list:
    if not isinstance(name, str): return []
    name = re.sub(r'\(.*?\)', '', name).strip()
    if not name: return []
    tokens = split_admin_tokens(name)
    if not tokens:
        stripped = re.sub(r'[시군구갑을병정무]$', '', name).strip()
        return [stripped] if stripped else []
    si_gun_count = sum(1 for t in tokens if t[-1] in '시군' and len(t) >= 2)
    gu_count     = sum(1 for t in tokens if t[-1] == '구'  and len(t) >= 2)
    ordered = tokens if (si_gun_count >= 2 or (si_gun_count == 0 and gu_count >= 2)) else list(reversed(tokens))
    candidates = []
    for t in ordered:
        key = re.sub(r'[시군구]$', '', t).strip()
        if key and key not in candidates: candidates.append(key)
    return candidates

def get_election_periods(month: int, year: int) -> dict:
    if month <= 6:
        prev = year - 1
        return {'half_periods': [f'{prev}.1/2', f'{prev}.2/2'], 'grdp_year': prev}
    else:
        prev = year - 1
        return {'half_periods': [f'{prev}.2/2', f'{year}.1/2'], 'grdp_year': prev}

def parse_employment_sgg(region_name: str) -> tuple:
    if not isinstance(region_name, str): return ('', '')
    parts = region_name.strip().split(None, 1)
    if not parts: return ('', '')
    prov = parts[0]
    if len(parts) == 1:
        prov_eng = SGG_TO_PROVINCE_EMP.get(prov, '')
        primary  = re.sub(r'[시군구]$', '', prov).strip()
        return (prov_eng, primary)
    sgg_raw  = parts[1]
    sgg_norm = normalize_sigungu(sgg_raw)
    primary  = sgg_norm[0] if sgg_norm else re.sub(r'[시군구]$', '', sgg_raw).strip()
    prov_eng = PROV_FULL_TO_SHORT.get(prov, prov)
    return (prov_eng, primary)

# ==========================================
# 1. DEMOGRAPHIC, ASSET & STATION DATA LOADERS
# ==========================================
def load_polling_stations(elec_id) -> pd.DataFrame:
    # Look strictly for normal-day station files to proxy physical density friction
    paths_to_try = [
        f"stations_normal_{elec_id}.csv",
        f"polling_stations_{elec_id}.csv",
        f"stations_normal_20240410.csv" if str(elec_id) == '22' else f"stations_normal_{elec_id}.csv"
    ]

    df = pd.DataFrame()
    for path in paths_to_try:
        if os.path.exists(path):
            df = _read_csv_auto(path)
            break

    if df.empty: return df

    if 'province_tag' not in df.columns and 'sdName' in df.columns:
        df['province_tag'] = df['sdName'].map(PROV_FULL_TO_SHORT).fillna(df['sdName'])
        def extract_sgg(x):
            cands = normalize_sigungu(x)
            return cands[0] if cands else re.sub(r'[시군구]$', '', str(x)).strip()
        df['primary_sgg'] = df['wiwName'].apply(extract_sgg)
        df['dong_norm']   = df['emdName'].apply(normalize_dong_name)

        mask_seoul = df['province_tag'] == 'Seoul'
        mask_g3 = df['primary_sgg'].isin(['강남', '서초', '송파'])
        df.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
        df.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    if 'normal_station_count' not in df.columns:
        if 'station_count' in df.columns:
            df = df.rename(columns={'station_count': 'normal_station_count'})
        else:
            df['normal_station_count'] = 1

    return df[['province_tag', 'primary_sgg', 'dong_norm', 'normal_station_count']]


def _detect_year_prefix(df: pd.DataFrame) -> str:
    for col in df.columns:
        m = re.match(r'(\d{4}년\d{2}월)_계_총인구수', col)
        if m: return m.group(1)
    raise ValueError("Cannot detect census year prefix.")

def load_census_csv(csv_path: str) -> pd.DataFrame:
    if not csv_path or not os.path.exists(csv_path): return pd.DataFrame()
    try:
        df = _read_csv_auto(csv_path, low_memory=False)
        prefix = _detect_year_prefix(df)
        voting_age_cols = ([f"{prefix}_계_{a}세" for a in range(18, 100)] + [f"{prefix}_계_100세 이상"])
        cols_4059 = [f"{prefix}_계_{a}세" for a in range(40, 60)]
        all_target_cols = list(voting_age_cols)
        for g in ['남', '여']:
            for a in range(18, 100):
                all_target_cols.append(f"{prefix}_{g}_{a}세")
            all_target_cols.append(f"{prefix}_{g}_100세 이상")

        for col in set(all_target_cols):
            if col in df.columns:
                df[col] = (df[col].astype(str).str.replace(',', '', regex=False).pipe(pd.to_numeric, errors='coerce').fillna(0))

        df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)
        df = df[df['total_voting_pop'] > 0].copy()

        ranges = [(18,25,'1824'),(25,30,'2529'),(30,35,'3034'),(35,40,'3539'),
                  (40,45,'4044'),(45,50,'4549'),(50,55,'5054'),(55,60,'5559'),
                  (60,65,'6064'),(65,70,'6569')]

        for g, g_str in [('남','m'), ('여','f')]:
            for r_start, r_end, r_str in ranges:
                cols = [f"{prefix}_{g}_{a}세" for a in range(r_start, r_end)]
                df[f'pct_{g_str}_{r_str}'] = (df[[c for c in cols if c in df.columns]].sum(axis=1) / df['total_voting_pop'])
            cols_70 = ([f"{prefix}_{g}_{a}세" for a in range(70, 100)] + [f"{prefix}_{g}_100세 이상"])
            df[f'pct_{g_str}_70plus'] = (df[[c for c in cols_70 if c in df.columns]].sum(axis=1) / df['total_voting_pop'])

        df['demographic_propensity'] = (df[[c for c in cols_4059 if c in df.columns]].sum(axis=1) / df['total_voting_pop'])

        def extract_census_keys(admin_name):
            if not isinstance(admin_name, str): return [], ""
            clean = re.sub(r'\(.*?\)', '', admin_name).strip()
            parts = clean.split()
            dong_norm = normalize_dong_name(parts[-1]) if parts else ""
            sgg_cands = []
            for token in reversed(parts[:-1]):
                for c in normalize_sigungu(token):
                    if c not in sgg_cands: sgg_cands.append(c)
            return sgg_cands, dong_norm

        rows = []
        for _, row in df.iterrows():
            sgg_cands, dong_norm = extract_census_keys(row['행정구역'])
            row_dict = {
                'sgg_candidates': sgg_cands,
                'primary_sgg':    sgg_cands[0] if sgg_cands else "",
                'dong_norm':      dong_norm,
                'demographic_propensity': row['demographic_propensity'],
            }
            for col in AGE_GENDER_COLS:
                row_dict[col] = row.get(col, np.nan)
            rows.append(row_dict)

        return pd.DataFrame(rows)
    except Exception as e:
        return pd.DataFrame()

def load_apt_csv(glob_pattern: str, elec_id: str) -> pd.DataFrame:
    if not glob_pattern: return pd.DataFrame()

    processed_path = f"output_data/processed_apt_data_{elec_id}.csv"

    if os.path.exists(processed_path):
        try:
            return _read_csv_auto(processed_path)
        except Exception:
            pass

    file_list = glob.glob(glob_pattern)
    if not file_list: return pd.DataFrame()

    df_list = []
    for file in file_list:
        try:
            df_list.append(_read_csv_auto(file, skiprows=15))
        except Exception: pass
    if not df_list: return pd.DataFrame()

    df = pd.concat(df_list, ignore_index=True)
    try:
        df['거래금액(만원)'] = pd.to_numeric(df['거래금액(만원)'].astype(str).str.replace(',', '').str.strip(), errors='coerce')
        df['전용면적(㎡)']  = pd.to_numeric(df['전용면적(㎡)'], errors='coerce')
        df['price_per_sqm']  = df['거래금액(만원)'] / df['전용면적(㎡)']

        def parse_loc(x):
            parts = str(x).split()
            prov  = PROV_FULL_TO_SHORT.get(parts[0], parts[0]) if parts else ""
            sgg   = normalize_sigungu(parts[1])[0] if len(parts) > 2 and normalize_sigungu(parts[1]) else ""
            dong  = normalize_dong_name(parts[-1]) if parts else ""
            return pd.Series([prov, sgg, dong])

        df[['prov','sgg','dong_norm']] = df['시군구'].apply(parse_loc)

        mask_seoul = df['prov'] == 'Seoul'
        mask_g3 = df['sgg'].isin(['강남', '서초', '송파'])
        df.loc[mask_seoul & mask_g3, 'prov'] = 'Seoul (Gangnam3gu)'
        df.loc[mask_seoul & ~mask_g3, 'prov'] = 'Seoul (Non-Gangnam3gu)'

        apt_agg = (df.groupby(['prov','sgg','dong_norm'])['price_per_sqm'].median().reset_index()
                   .rename(columns={'price_per_sqm': 'median_apt_price_sqm'}))

        apt_agg.to_csv(processed_path, index=False)
        return apt_agg
    except Exception as e:
        return pd.DataFrame()

def _avg_half_periods(df: pd.DataFrame, base_cols: list) -> pd.Series:
    available = [c for c in base_cols if c in df.columns]
    if not available: return pd.Series(np.nan, index=df.index)
    return df[available].apply(pd.to_numeric, errors='coerce').mean(axis=1)

def load_employment_sgg(election_key, **kwargs) -> pd.DataFrame:
    cfg    = ELECTION_CONFIGS[election_key if type(election_key)==str else int(election_key)]
    month  = cfg['election_month']
    year   = cfg['year']
    periods_info = get_election_periods(month, year)
    periods      = periods_info['half_periods']

    def read_emp(path):
        if not os.path.exists(path): return pd.DataFrame()
        df = _read_csv_auto(path, low_memory=False)
        return df[[c for c in df.columns if not str(c).startswith('Unnamed')]]

    age_map = {'total':'계','1529':'15 - 29세','3049':'30 - 49세',
               '5064':'50 - 64세','65plus':'65세이상'}

    def extract_gender_emp(csv_path, gender_prefix):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df_rate = df[df['항목'].str.strip() == '고용률 (%)'].copy()
        records = {}
        for suf, kor_age in age_map.items():
            sub = df_rate[df_rate['연령별'].str.strip() == kor_age].copy()
            if sub.empty: continue
            val_col = f'{gender_prefix}_{suf}'
            sub[val_col] = _avg_half_periods(sub, periods)
            prov_sgg = sub['행정구역별'].apply(parse_employment_sgg)
            sub['province_tag'] = prov_sgg.apply(lambda x: x[0])
            sub['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
            records[val_col] = sub.groupby(['province_tag','primary_sgg'])[val_col].mean()
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_men = extract_gender_emp(kwargs.get('men_csv', 'men_employment_data.csv'), 'emp_men')
    df_wmn = extract_gender_emp(kwargs.get('women_csv', 'women_employment_data.csv'), 'emp_wmn')

    def extract_occ_type(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df_total = df[df['종사상지위별'].str.strip() == '계'].copy()
        df_total['_total'] = _avg_half_periods(df_total, periods)
        prov_sgg = df_total['행정구역별'].apply(parse_employment_sgg)
        df_total['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df_total['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df_total.set_index(['province_tag','primary_sgg'])['_total']
        records = {}
        for col_name, kor_cat in {'occ_wage_share':'임금근로자', 'occ_regular_share':'- 상용근로자', 'occ_nonwage_share':'비임금근로자'}.items():
            sub = df[df['종사상지위별'].str.strip() == kor_cat].copy()
            if sub.empty: continue
            sub['_val'] = _avg_half_periods(sub, periods)
            prov_sgg2 = sub['행정구역별'].apply(parse_employment_sgg)
            sub['province_tag'] = prov_sgg2.apply(lambda x: x[0])
            sub['primary_sgg']  = prov_sgg2.apply(lambda x: x[1])
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_occ_type = extract_occ_type(kwargs.get('occ_type_csv', 'occupation_type_employment_data.csv'))

    def extract_job_shares(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df[df['항목'].str.strip() == '취업자[천명]'].copy()
        df['_val'] = _avg_half_periods(df, periods)
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df[df['직업별'].str.strip() == '계'].set_index(['province_tag','primary_sgg'])['_val']
        records = {}
        for col_name, kor_cat in {'job_professional': '관리자, 전문가 및 관련종사자', 'job_clerical': '사무 종사자', 'job_service_sales': '서비스·판매 종사자', 'job_skilled_machine':'기능·기계조작·조립 종사자', 'job_simple_labor': '단순노무 종사자', 'job_farming': '농림어업 숙련종사자'}.items():
            sub = df[df['직업별'].str.strip() == kor_cat].copy()
            if sub.empty: continue
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_job = extract_job_shares(kwargs.get('occ_csv', 'occupation_employment_data.csv'))

    def extract_ind_shares(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df[df['항목'].str.strip() == '취업자[천명]'].copy()
        df['_val'] = _avg_half_periods(df, periods)
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        total_map = df[df['산업별'].str.strip() == '계'].set_index(['province_tag','primary_sgg'])['_val']
        records = {}
        for col_name, kor_cat in {'ind_agriculture': '농업, 임업 및 어업 (A)', 'ind_manufacturing': '광·제조업(B,C)', 'ind_construction': '건설업 (F) ', 'ind_retail_food': '도소매·숙박음식업(G,I)', 'ind_transport_finance': '전기·운수·통신·금융(D,H,J,K)', 'ind_services': '사업·개인·공공서비스 및 기타(E,L~U)'}.items():
            sub = df[df['산업별'].str.strip() == kor_cat.strip()].copy()
            if sub.empty:
                match = get_close_matches(kor_cat.strip(), df['산업별'].str.strip().unique().tolist(), n=1, cutoff=0.85)
                if match: sub = df[df['산업별'].str.strip() == match[0]].copy()
            if sub.empty: continue
            records[col_name] = sub.set_index(['province_tag','primary_sgg'])['_val'] / total_map
        if not records: return pd.DataFrame()
        return pd.concat(records, axis=1).reset_index()

    df_ind = extract_ind_shares(kwargs.get('ind_csv', 'industry_employment_data.csv'))

    def extract_nonpart(csv_path):
        df = read_emp(csv_path)
        if df.empty: return pd.DataFrame()
        df = df.iloc[1:].copy()
        prov_sgg = df['행정구역별'].apply(parse_employment_sgg)
        df['province_tag'] = prov_sgg.apply(lambda x: x[0])
        df['primary_sgg']  = prov_sgg.apply(lambda x: x[1])
        sub_suffixes = {'total':'', 'male':'.1', 'young':'.3', 'middle':'.4'}
        period_vals  = {k: [] for k in sub_suffixes}
        for period in periods:
            for k, suf in sub_suffixes.items():
                if period + suf in df.columns: period_vals[k].append(_to_numeric_safe(df[period + suf]))
        def mean_series(lst): return pd.concat(lst, axis=1).mean(axis=1) if lst else pd.Series(np.nan, index=df.index)
        total  = mean_series(period_vals['total'])
        result = df[['province_tag','primary_sgg']].copy()
        result['nonp_male_share']   = mean_series(period_vals['male'])   / total.replace(0, np.nan)
        result['nonp_young_share']  = mean_series(period_vals['young'])  / total.replace(0, np.nan)
        result['nonp_middle_share'] = mean_series(period_vals['middle']) / total.replace(0, np.nan)
        return (result.groupby(['province_tag','primary_sgg'])[['nonp_male_share','nonp_young_share','nonp_middle_share']].mean().reset_index())

    df_nonp = extract_nonpart(kwargs.get('nonp_csv', 'non_participant_employment_data.csv'))

    merge_key = ['province_tag','primary_sgg']
    frames = [f for f in [df_men, df_wmn, df_occ_type, df_job, df_ind, df_nonp] if not f.empty]
    if not frames: return pd.DataFrame(columns=merge_key + EMPLOYMENT_FEATURE_COLS)

    df_emp = frames[0]
    for frame in frames[1:]: df_emp = df_emp.merge(frame, on=merge_key, how='outer')
    for col in EMPLOYMENT_FEATURE_COLS:
        if col not in df_emp.columns: df_emp[col] = np.nan

    if not df_emp.empty:
        mask_seoul = df_emp['province_tag'] == 'Seoul'
        mask_g3 = df_emp['primary_sgg'].isin(['강남', '서초', '송파'])
        df_emp.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
        df_emp.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    return df_emp

# ==========================================
# 2. ELECTION CSV LOADER & MATCHER
# ==========================================
def load_election_csv(csv_path: str, dem_pattern: str, con_pattern: str,
                      third_pattern: str = None, election_type: str = 'general'):
    try:
        df = _read_csv_auto(csv_path, low_memory=False)
    except Exception: return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if election_type == 'presidential':
        df = df.rename(columns={'구시군명': '선거구명', '읍면동명': '법정읍면동명'})
        special_dong_names = SPECIAL_DONG_NAMES_PRESIDENTIAL
    else:
        special_dong_names = SPECIAL_DONG_NAMES_GENERAL

    df['득표수']    = pd.to_numeric(df['득표수'], errors='coerce').fillna(0).astype(int)
    df['is_dem']   = df['후보자'].str.contains(dem_pattern,   case=False, na=False)

    if election_type == 'general':
        sejong_gap_kim = (df['선거구명'].astype(str).str.replace(' ', '').str.contains('세종.*갑', na=False)) & \
                         (df['후보자'].astype(str).str.contains('김종민', na=False))
        df.loc[sejong_gap_kim, 'is_dem'] = True

    df['is_con']   = df['후보자'].str.contains(con_pattern,   case=False, na=False)
    df['is_third'] = df['후보자'].str.contains(third_pattern, case=False, na=False) if third_pattern else False
    df['is_meta']  = df['후보자'].isin(META_CANDIDATES)
    df['is_early'] = df['투표구명'] == GWANNAESA_LABEL

    dong_key  = ['시도명','선거구명','법정읍면동명']
    const_key = ['시도명','선거구명']

    def sgg_cands(name):
        if not isinstance(name, str): return []
        if '_' in name: return normalize_sigungu(name.split('_', 1)[1])
        return normalize_sigungu(re.sub(r'[갑을병정무]$', '', name).strip())

    df_geo   = df[~df['법정읍면동명'].isin(special_dong_names)].copy()
    df_votes = df_geo[~df_geo['is_meta']].copy()

    gn_dem = df_votes[df_votes['is_dem'] & df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_dem')
    gn_tot = df_votes[df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_total')
    gn_third = df_votes[df_votes['is_third'] & df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='in_precinct_early_third')

    sd_dem = df_votes[df_votes['is_dem'] & ~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_dem')
    sd_tot = df_votes[~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_total')
    sd_third = df_votes[df_votes['is_third'] & ~df_votes['is_early']].groupby(dong_key)['득표수'].sum().reset_index(name='same_day_third')

    sum_people_dong = (df_geo[~df_geo['is_early'] & (df_geo['후보자'] == '선거인수')].groupby(dong_key)['득표수'].sum().reset_index(name='sum_people'))
    sum_vote_geo    = (df_geo[df_geo['후보자'] == '투표수'].groupby(dong_key)['득표수'].sum().reset_index(name='sum_vote_geo'))

    df_dong = gn_dem.copy()
    for frame in (gn_tot, gn_third, sd_dem, sd_tot, sd_third, sum_people_dong, sum_vote_geo):
        df_dong = df_dong.merge(frame, on=dong_key, how='outer')
    df_dong = df_dong.fillna(0)

    gn_ppl_dong = (df_geo[df_geo['is_early'] & (df_geo['후보자'] == '선거인수')].groupby(dong_key)['득표수'].sum().reset_index(name='_gn_ppl'))
    df_dong = df_dong.merge(gn_ppl_dong, on=dong_key, how='left')
    df_dong['sum_people'] = df_dong['sum_people'] + df_dong['_gn_ppl'].fillna(0)
    df_dong.drop(columns=['_gn_ppl'], inplace=True)

    df_dong['sgg_candidates'] = df_dong['선거구명'].apply(sgg_cands)
    df_dong['primary_sgg']    = df_dong['sgg_candidates'].apply(lambda x: x[0] if x else "")
    df_dong['dong_norm']      = df_dong['법정읍면동명'].apply(normalize_dong_name)
    df_dong['province_tag']   = df_dong['시도명'].map(PROV_FULL_TO_SHORT).fillna(df_dong['시도명'])
    df_dong['area2_name']     = df_dong['선거구명']

    mask_seoul = df_dong['province_tag'] == 'Seoul'
    mask_g3 = df_dong['primary_sgg'].isin(['강남', '서초', '송파'])
    df_dong.loc[mask_seoul & mask_g3, 'province_tag'] = 'Seoul (Gangnam3gu)'
    df_dong.loc[mask_seoul & ~mask_g3, 'province_tag'] = 'Seoul (Non-Gangnam3gu)'

    df_gw   = df[df['법정읍면동명'].isin(special_dong_names)]
    df_gw_v = df_gw[~df_gw['is_meta']]

    go_dem_c  = df_gw_v[df_gw_v['is_dem']].groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_dem')
    go_tot_c  = df_gw_v.groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_total')
    go_turn_c = (df_gw[df_gw['후보자'] == '투표수'].groupby(const_key)['득표수'].sum().reset_index(name='out_precinct_early_turnout'))

    df_const = df_dong.groupby(const_key)['sum_people'].sum().reset_index(name='sum_people')
    for frame in (go_dem_c, go_tot_c, go_turn_c):
        df_const = df_const.merge(frame, on=const_key, how='left')
    df_const = df_const.fillna(0)

    return df_dong, df_const, pd.DataFrame()

def merge_dong_with_covariates(df_election: pd.DataFrame, df_census: pd.DataFrame,
                                df_apt: pd.DataFrame, df_emp: pd.DataFrame) -> pd.DataFrame:
    if not df_census.empty:
        census_lookup = {}
        census_by_sgg = {}
        for _, row in df_census.iterrows():
            dnorm = row['dong_norm']
            covs  = {'demographic_propensity': row['demographic_propensity']}
            for c in AGE_GENDER_COLS: covs[c] = row.get(c, np.nan)
            for sgg in row['sgg_candidates']:
                census_lookup[(sgg, dnorm)] = covs
                census_by_sgg.setdefault(sgg, []).append(dnorm)

        results = []
        for _, row in df_election.iterrows():
            covs, dk, sgc = None, row['dong_norm'], row['sgg_candidates'] if isinstance(row['sgg_candidates'], list) else [row['primary_sgg']]
            if (row['primary_sgg'], dk) in census_lookup: covs = census_lookup[(row['primary_sgg'], dk)]
            if covs is None and '·' in dk:
                if (row['primary_sgg'], dk.replace('·', '')) in census_lookup: covs = census_lookup[(row['primary_sgg'], dk.replace('·', ''))]
            if covs is None:
                for sgg in sgc[1:]:
                    if (sgg, dk) in census_lookup: covs = census_lookup[(sgg, dk)]; break
            if covs is None:
                for sgg in sgc:
                    pool = census_by_sgg.get(sgg, [])
                    if pool:
                        m = get_close_matches(dk, pool, n=1, cutoff=0.82)
                        if m and (sgg, m[0]) in census_lookup: covs = census_lookup[(sgg, m[0])]; break
            if covs is None: covs = {k: np.nan for k in ['demographic_propensity'] + AGE_GENDER_COLS}
            rd = row.to_dict(); rd.update(covs)
            results.append(rd)
        df_out = pd.DataFrame(results)
    else:
        df_out = df_election.copy()

    if not df_apt.empty:
        df_out = df_out.merge(df_apt, left_on=['province_tag','primary_sgg','dong_norm'], right_on=['prov','sgg','dong_norm'], how='left')
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out.groupby('primary_sgg')['median_apt_price_sqm'].transform('median'))
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out.groupby('province_tag')['median_apt_price_sqm'].transform('median'))
        df_out['median_apt_price_sqm'] = df_out['median_apt_price_sqm'].fillna(df_out['median_apt_price_sqm'].median())
        df_out['log_apt_price'] = np.log1p(df_out['median_apt_price_sqm'])
    else:
        df_out['log_apt_price'] = np.nan

    if not df_emp.empty:
        df_out = df_out.merge(df_emp[['province_tag','primary_sgg'] + EMPLOYMENT_FEATURE_COLS], on=['province_tag','primary_sgg'], how='left')
        for col in EMPLOYMENT_FEATURE_COLS:
            if col not in df_out.columns: df_out[col] = np.nan
            df_out[col] = df_out[col].fillna(df_out.groupby('province_tag')[col].transform('median')).fillna(df_out[col].median())
    else:
        for col in EMPLOYMENT_FEATURE_COLS: df_out[col] = np.nan

    return df_out[df_out['pct_f_4044'].notna()].copy()

def prep_election_data(df_dong_raw: pd.DataFrame, df_const_raw: pd.DataFrame,
                       df_census: pd.DataFrame, df_apt: pd.DataFrame,
                       df_emp: pd.DataFrame, df_stations: pd.DataFrame, elec_id: str) -> pd.DataFrame:
    dm = merge_dong_with_covariates(df_dong_raw, df_census, df_apt, df_emp)
    dm = dm[(dm['in_precinct_early_total'] > 50) & (dm['same_day_total'] > 50)].copy()

    no_dem = (dm['in_precinct_early_dem'] == 0) & (dm['same_day_dem'] == 0)
    if dm[no_dem].shape[0] > 0:
        dm = dm[~no_dem].copy()

    const_sums = dm.groupby(['시도명', '선거구명'])[['in_precinct_early_total', 'same_day_total']].transform('sum')
    dm['dong_weight'] = (dm['in_precinct_early_total'] + dm['same_day_total']) / (const_sums['in_precinct_early_total'] + const_sums['same_day_total']).replace(0, np.nan)

    df_const_sub = df_const_raw[['시도명', '선거구명', 'out_precinct_early_dem', 'out_precinct_early_total', 'out_precinct_early_turnout']].drop_duplicates()
    dm = dm.merge(df_const_sub, on=['시도명', '선거구명'], how='left')

    dm['out_precinct_alloc_dem']     = dm['out_precinct_early_dem'].fillna(0) * dm['dong_weight'].fillna(0)
    dm['out_precinct_alloc_tot']     = dm['out_precinct_early_total'].fillna(0) * dm['dong_weight'].fillna(0)
    dm['out_precinct_alloc_turnout'] = dm['out_precinct_early_turnout'].fillna(0) * dm['dong_weight'].fillna(0)

    dm['vote_share'] = ((dm['in_precinct_early_dem'] + dm['same_day_dem'] + dm['out_precinct_alloc_dem']) /
                        (dm['in_precinct_early_total'] + dm['same_day_total'] + dm['out_precinct_alloc_tot'])).fillna(0)
    dm['turnout'] = (dm['sum_vote_geo'] + dm['out_precinct_alloc_turnout']) / (dm['sum_people'] + dm['out_precinct_alloc_turnout']).replace(0, np.nan)
    dm['early_turnout'] = (dm['in_precinct_early_total'] + dm['out_precinct_alloc_tot']) / (dm['sum_people'] + dm['out_precinct_alloc_turnout']).replace(0, np.nan)
    dm['early_dem_share'] = (dm['in_precinct_early_dem'] + dm['out_precinct_alloc_dem']) / (dm['in_precinct_early_total'] + dm['out_precinct_alloc_tot']).replace(0, np.nan)
    dm['third_vote_share'] = ((dm['in_precinct_early_third'] + dm['same_day_third']) / (dm['in_precinct_early_total'] + dm['same_day_total'])).fillna(0)

    # ---------------------------------------------------------
    # INSTRUMENT INTEGRATION: voters_per_station
    # ---------------------------------------------------------
    if not df_stations.empty:
        dm = dm.merge(df_stations, on=['province_tag', 'primary_sgg', 'dong_norm'], how='left')
        dm['normal_station_count'] = dm['normal_station_count'].fillna(1)
    else:
        dm['normal_station_count'] = 1

    dm['voters_per_station'] = dm['sum_people'] / dm['normal_station_count']

    # Strict Local Demeaning (Province + Primary SGG Level)
    dm['voters_per_station_dm'] = dm['voters_per_station'] - dm.groupby(['province_tag', 'primary_sgg'])['voters_per_station'].transform('mean')

    if dm['voters_per_station_dm'].std(skipna=True) > 0:
        dm['voters_per_station_dm'] = StandardScaler().fit_transform(dm[['voters_per_station_dm']].fillna(0))
    else:
        dm['voters_per_station_dm'] = 0.0

    dm['election_id'] = str(elec_id)
    return dm

# ==========================================
# 4. PCA COLLINEARITY PIPELINE
# ==========================================
def reduce_pca(df: pd.DataFrame, feature_cols: list, threshold: float, name_label: str) -> tuple:
    available = [c for c in feature_cols if c in df.columns and df[c].notna().mean() >= MIN_COVERAGE]
    if len(available) < 2: return df.copy(), [], None, available

    valid_mask = df[available].notna().all(axis=1)
    if valid_mask.sum() < 30: return df.copy(), [], None, available

    X = df.loc[valid_mask, available].values.astype(float)
    X_scaled = StandardScaler().fit_transform(X)

    pca_probe = PCA().fit(X_scaled)
    cumvar = np.cumsum(pca_probe.explained_variance_ratio_)
    n_components = min(int(np.searchsorted(cumvar, threshold)) + 1, len(available))

    pca_final = PCA(n_components=n_components)
    components = pca_final.fit_transform(X_scaled)

    pc_cols = [f'pc{i+1}' for i in range(n_components)]
    df_out = df.copy()
    for col in pc_cols: df_out[col] = np.nan
    df_out.loc[valid_mask, pc_cols] = components

    loadings = pd.DataFrame(pca_final.components_.T, columns=pc_cols, index=available)
    out_path = f'output_data/pca_loadings_{name_label}.csv'
    loadings.to_csv(out_path)
    log_to_file(f"  -> Saved PCA Loadings for {name_label} to {out_path}")

    return df_out, pc_cols, pca_final, available

def prepare_continuous_covariates(df: pd.DataFrame, raw_age_cols: list, raw_emp_cols: list, label: str) -> tuple:
    combined_raw = list(raw_age_cols) + list(raw_emp_cols)
    if 'log_apt_price' in df.columns: combined_raw.append('log_apt_price')

    df, pc_cols, pca_model, used_features = reduce_pca(df, combined_raw, PCA_VARIANCE_THRESHOLD, label)
    pc_cols = [c for c in pc_cols if df[c].notna().mean() >= MIN_COVERAGE]

    dm_cols = []
    if pc_cols:
        active_groups = [g for g in DEMEAN_GROUP_COLS if g in df.columns]
        for col in pc_cols:
            df[f'{col}_dm'] = df[col] - df.groupby(active_groups)[col].transform('mean')
            dm_cols.append(f'{col}_dm')

    dm_cols = [c for c in dm_cols if df[c].std(skipna=True) > 1e-10]

    if dm_cols:
        valid_mask = df[dm_cols].notna().all(axis=1)
        if valid_mask.sum() > len(dm_cols) + 5:
            df.loc[valid_mask, dm_cols] = StandardScaler().fit_transform(df.loc[valid_mask, dm_cols])

    return df, dm_cols, pca_model, used_features


# ==========================================
# 5. REGRESSIONS (WLS & IV)
# ==========================================
def run_single_analysis(df_single: pd.DataFrame, elec_id, dm_cols: list):
    label = f"Election_{elec_id}"
    header = f"\n{'='*70}\n  SINGLE ELECTION ANALYSIS: {label}\n{'='*70}"
    log_to_file(header)

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share', 'sum_people',
                'area2_name', 'province_tag', 'voters_per_station_dm'] + dm_cols
    df_sub = df_single.dropna(subset=req_cols).copy()

    df_sub.to_csv(f'output_data/regression_data_{label}{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    if df_sub.empty:
        log_to_file(f" [!] Not enough data to run analysis for {label}.")
        return

    exog_vars = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag)"

    # 1. Turnout Regressions
    for target in ['turnout', 'early_turnout']:
        log_to_file(f"\n--- Sub-Target: {target} ---")
        exog_str = exog_vars + " + voters_per_station_dm"

        formula = f"{target} ~ {exog_str}"
        model = smf.wls(formula, data=df_sub, weights=df_sub['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_sub['area2_name']})
        log_to_file(f"[Formula]: {formula}")
        log_to_file(model.summary().as_text())

    # 2. Total Dem Vote Share & Early Dem Share Regressions
    for main_target in ['vote_share', 'early_dem_share']:
        turnout_vars = ['early_turnout', 'turnout'] if INCLUDE_TURNOUT_IN_VOTESHARE else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (Instrumenting {turnout_var})\n{'='*55}")
                formula_vs = f"{main_target} ~ {exog_vars} + [{turnout_var} ~ voters_per_station_dm]"
                model_vs = IV2SLS.from_formula(formula_vs, df_sub, weights=df_sub['sum_people'])
                res_vs = model_vs.fit(cov_type='robust')

                log_to_file(f"\n[Formula]: {formula_vs}")
                if hasattr(res_vs, 'first_stage') and res_vs.first_stage is not None:
                    log_to_file("\n[First Stage F-Stats]")
                    for endog in res_vs.first_stage.diagnostics.index:
                        f_stat = res_vs.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file("\n[Second Stage Summary]")
                log_to_file(res_vs.summary.as_text())
            else:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (No Turnout Covariate)\n{'='*55}")
                formula_vs = f"{main_target} ~ {exog_vars}"
                model_vs = smf.wls(formula_vs, data=df_sub, weights=df_sub['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_sub['area2_name']})
                log_to_file(f"\n[Formula]: {formula_vs}")
                log_to_file(model_vs.summary().as_text())


def run_iv_causal_analysis(df_pres21: pd.DataFrame, df_pres19: pd.DataFrame, dm_cols: list):
    header = f"\n{'='*70}\n  EXACTLY IDENTIFIED IV (2SLS) ANALYSIS (21st Pres)\n{'='*70}"
    log_to_file(header)

    instrument_df = df_pres19[['province_tag', 'primary_sgg', 'dong_norm', 'third_vote_share']].drop_duplicates()
    instrument_df = instrument_df.rename(columns={'third_vote_share': 'yoo_vote_share_19'})

    df_iv = df_pres21.rename(columns={'third_vote_share': 'LJS_vote_share'})
    df_iv = df_iv.merge(instrument_df, on=['province_tag', 'primary_sgg', 'dong_norm'], how='inner')

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share',
                'LJS_vote_share', 'yoo_vote_share_19', 'sum_people', 'voters_per_station_dm', 'area2_name'] + dm_cols
    df_iv = df_iv.dropna(subset=req_cols).copy()

    df_iv.to_csv(f'output_data/regression_data_iv_pres21{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    exog_vars = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag)"

    # 1. Turnout Regressions
    for target in ['turnout', 'early_turnout']:
        log_to_file(f"\n\n{'='*55}\n Target: {target}\n{'='*55}")

        exog_str = exog_vars + " + voters_per_station_dm"

        if INCLUDE_LJS_IN_VOTESHARE:
            formula = f"{target} ~ {exog_str} + [LJS_vote_share ~ yoo_vote_share_19]"
            res = IV2SLS.from_formula(formula, df_iv, weights=df_iv['sum_people']).fit(cov_type='robust')
            log_to_file(f"\n[Formula]: {formula}")

            if hasattr(res, 'first_stage') and res.first_stage is not None:
                log_to_file("\n[First Stage F-Stats]")
                for endog in res.first_stage.diagnostics.index:
                    f_stat = res.first_stage.diagnostics.loc[endog, 'f.stat']
                    log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
            log_to_file("\n[Second Stage Summary]")
            log_to_file(res.summary.as_text())
        else:
            formula = f"{target} ~ {exog_str}"
            res = smf.wls(formula, data=df_iv, weights=df_iv['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_iv['area2_name']})
            log_to_file(f"\n[Formula]: {formula}")
            log_to_file(res.summary().as_text())

    # 2. Vote Share & Early Dem Share Regressions (Dynamic string building based on toggles)
    for main_target in ['vote_share', 'early_dem_share']:
        turnout_vars = ['early_turnout', 'turnout'] if INCLUDE_TURNOUT_IN_VOTESHARE else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (Turnout Covariate: {turnout_var})\n{'='*55}")
            else:
                log_to_file(f"\n\n{'='*55}\n Target: {main_target} (No Turnout Covariate)\n{'='*55}")

            endogs, instrs = [], []

            if INCLUDE_LJS_IN_VOTESHARE:
                endogs.append('LJS_vote_share')
                instrs.append('yoo_vote_share_19')
            if INCLUDE_TURNOUT_IN_VOTESHARE and turnout_var:
                endogs.append(turnout_var)
                instrs.append('voters_per_station_dm')

            if endogs:
                formula = f"{main_target} ~ {exog_vars} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res = IV2SLS.from_formula(formula, df_iv, weights=df_iv['sum_people']).fit(cov_type='robust')

                log_to_file(f"\n[Formula]: {formula}")
                if hasattr(res, 'first_stage') and res.first_stage is not None:
                    log_to_file("\n[First Stage F-Stats]")
                    for endog in res.first_stage.diagnostics.index:
                        f_stat = res.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file("\n[Second Stage Summary]")
                log_to_file(res.summary.as_text())
            else:
                formula = f"{main_target} ~ {exog_vars}"
                res = smf.wls(formula, data=df_iv, weights=df_iv['sum_people']).fit(cov_type='cluster', cov_kwds={'groups': df_iv['area2_name']})
                log_to_file(f"\n[Formula]: {formula}")
                log_to_file(res.summary().as_text())

    return df_iv


def run_pooled_iv_panel(df_pooled: pd.DataFrame, df_pres19: pd.DataFrame, dm_cols: list):
    header = f"\n{'='*70}\n  POOLED IV PANEL REGRESSIONS\n{'='*70}"
    log_to_file(header)

    df_panel = df_pooled.copy()

    inst_df = df_pres19[['province_tag', 'primary_sgg', 'dong_norm', 'third_vote_share']].drop_duplicates()
    inst_df = inst_df.rename(columns={'third_vote_share': 'base_yoo_pr'})

    df_panel = df_panel.merge(inst_df, on=['province_tag', 'primary_sgg', 'dong_norm'], how='inner')

    df_panel['LJS_vote_share'] = np.where(df_panel['election_id'] == 'pres21', df_panel['third_vote_share'], 0)
    df_panel['active_yoo_pr'] = np.where(df_panel['election_id'] == 'pres21', df_panel['base_yoo_pr'], 0)
    df_panel['prov_elec'] = df_panel['province_tag'] + "_" + df_panel['election_id'].astype(str)

    req_cols = ['vote_share', 'turnout', 'early_turnout', 'early_dem_share',
                'LJS_vote_share', 'active_yoo_pr', 'sum_people', 'election_id',
                'province_tag', 'prov_elec', 'voters_per_station_dm', 'area2_name'] + dm_cols
    df_panel = df_panel.dropna(subset=req_cols).copy()

    df_panel.to_csv(f'output_data/regression_data_pooled_panel{IV_SUFFIX}.csv', index=False, encoding='utf-8-sig')

    exog_vars_add = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(province_tag) + C(election_id)"
    exog_vars_int = f"{' + '.join(dm_cols) if dm_cols else '1'} + C(prov_elec)"

    for target in ['turnout', 'early_turnout', 'early_dem_share', 'vote_share']:
        log_to_file(f"\n\n{'='*55}\n PANEL TARGET: {target}\n{'='*55}")

        exog_add_base = exog_vars_add
        exog_int_base = exog_vars_int

        if target in ['turnout', 'early_turnout']:
            exog_add_base += " + voters_per_station_dm"
            exog_int_base += " + voters_per_station_dm"

        turnout_vars = ['early_turnout', 'turnout'] if (target in ['vote_share', 'early_dem_share'] and INCLUDE_TURNOUT_IN_VOTESHARE) else [None]

        for turnout_var in turnout_vars:
            if turnout_var:
                log_to_file(f"\n--- Sub-Target for {target} (Turnout Covariate: {turnout_var}) ---")

            endogs, instrs = [], []

            if INCLUDE_LJS_IN_VOTESHARE:
                endogs.append('LJS_vote_share')
                instrs.append('active_yoo_pr')

            if target in ['vote_share', 'early_dem_share'] and INCLUDE_TURNOUT_IN_VOTESHARE and turnout_var:
                endogs.append(turnout_var)
                instrs.append('voters_per_station_dm')

            # Execute Spec 1 (Additive)
            log_to_file(f"\n[Spec 1: Additive Province & Election FEs]")
            if endogs:
                formula_add = f"{target} ~ {exog_add_base} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res_add = IV2SLS.from_formula(formula_add, df_panel, weights=df_panel['sum_people']).fit(cov_type='robust')
                log_to_file(f"[Formula]: {formula_add}")
                if hasattr(res_add, 'first_stage') and res_add.first_stage is not None:
                    log_to_file("[First Stage F-Stats]")
                    for endog in res_add.first_stage.diagnostics.index:
                        f_stat = res_add.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file(res_add.summary.as_text())
            else:
                formula_add = f"{target} ~ {exog_add_base}"
                res_add = smf.wls(formula_add, data=df_panel, weights=df_panel['sum_people']).fit(cov_type='HC3')
                log_to_file(f"[Formula]: {formula_add}")
                log_to_file(res_add.summary().as_text())

            # Execute Spec 2 (Interacted)
            log_to_file(f"\n[Spec 2: Interacted Province x Election FEs]")
            if endogs:
                formula_int = f"{target} ~ {exog_int_base} + [{' + '.join(endogs)} ~ {' + '.join(instrs)}]"
                res_int = IV2SLS.from_formula(formula_int, df_panel, weights=df_panel['sum_people']).fit(cov_type='robust')
                log_to_file(f"[Formula]: {formula_int}")
                if hasattr(res_int, 'first_stage') and res_int.first_stage is not None:
                    log_to_file("[First Stage F-Stats]")
                    for endog in res_int.first_stage.diagnostics.index:
                        f_stat = res_int.first_stage.diagnostics.loc[endog, 'f.stat']
                        log_to_file(f"  {endog} ~ Instruments F-Stat: {f_stat:.2f}")
                log_to_file(res_int.summary.as_text())
            else:
                formula_int = f"{target} ~ {exog_int_base}"
                res_int = smf.wls(formula_int, data=df_panel, weights=df_panel['sum_people']).fit(cov_type='HC3')
                log_to_file(f"[Formula]: {formula_int}")
                log_to_file(res_int.summary().as_text())


# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    with open(LOG_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write("COMPREHENSIVE REGRESSION & DATA LOG\n")
        f.write("="*60 + "\n")

    all_dm = []

    for elec_id in ['pres19', 21, 22, 'pres20', 'pres21']:
        cfg = ELECTION_CONFIGS[elec_id]
        log_to_file(f"\n--- Loading Data for {cfg['label']} ---")

        df_census = load_census_csv(cfg.get('census_csv'))
        df_apt = load_apt_csv(cfg.get('apt_csv_glob'), elec_id)
        df_emp = load_employment_sgg(election_key=elec_id)
        df_stations = load_polling_stations(elec_id)

        log_to_file(f"  > Census Data: {'Loaded successfully' if not df_census.empty else 'NO DATA FOUND'}")

        apt_cache_path = f"output_data/processed_apt_data_{elec_id}.csv"
        if not df_apt.empty:
            if os.path.exists(apt_cache_path) and os.path.getmtime(apt_cache_path) > 0:
                log_to_file(f"  > APT Price Data: Loaded from Cache ({apt_cache_path})")
            else:
                log_to_file(f"  > APT Price Data: Parsed from Raw and Saved to Cache")
        else:
            log_to_file(f"  > APT Price Data: NO DATA FOUND")

        log_to_file(f"  > Employment Data: {'Loaded successfully' if not df_emp.empty else 'NO DATA FOUND'}")
        log_to_file(f"  > Polling Stations: {'Loaded successfully' if not df_stations.empty else 'NO DATA FOUND'}")

        df_dong, df_const, _ = load_election_csv(
            cfg['result_csv'],
            dem_pattern=cfg['dem_pattern'],
            con_pattern=cfg['con_pattern'],
            third_pattern=cfg.get('third_pattern'),
            election_type=cfg['election_type'],
        )

        if EXCLUDE_HONAM_YEONGNAM:
            exclude_provs = PARTISAN_REGION_PROVINCES['Honam'] | PARTISAN_REGION_PROVINCES['Yeongnam']
            short_exclude = {PROV_FULL_TO_SHORT.get(p, p) for p in exclude_provs} | exclude_provs
            df_dong  = df_dong[~df_dong['province_tag'].isin(short_exclude)]
            df_const = df_const[~df_const['province_tag'].isin(short_exclude)]

        if not df_dong.empty and not df_const.empty:
            dm = prep_election_data(df_dong, df_const, df_census, df_apt, df_emp, df_stations, elec_id)
            all_dm.append(dm)
            log_to_file(f"  > Processed {len(dm)} distinct dong-level observations.")

    df_raw_pooled = pd.concat(all_dm, ignore_index=True)
    raw_age_cols = [c for c in AGE_GENDER_COLS if c in df_raw_pooled.columns]
    raw_emp_cols = [c for c in EMPLOYMENT_FEATURE_COLS if c in df_raw_pooled.columns]

    # ---------------------------------------------------------
    # PART A: ISOLATED SINGLE-ELECTION PIPELINE
    # ---------------------------------------------------------
    log_to_file("\n\n" + "#"*70 + "\n PART A: ISOLATED SINGLE-ELECTION PIPELINE\n" + "#"*70)

    for elec in [21, 22, 'pres20']:
        df_single_raw = df_raw_pooled[df_raw_pooled['election_id'] == str(elec)].copy()
        if not df_single_raw.empty:
            df_single_pca, dm_cols_single, _, _ = prepare_continuous_covariates(df_single_raw, raw_age_cols, raw_emp_cols, label=f"isolated_{elec}")
            run_single_analysis(df_single_pca, elec, dm_cols_single)

    df_pres21_raw = df_raw_pooled[df_raw_pooled['election_id'] == 'pres21'].copy()
    df_pres19_raw = df_raw_pooled[df_raw_pooled['election_id'] == 'pres19'].copy()

    if not df_pres21_raw.empty and not df_pres19_raw.empty:
        df_pres21_pca, dm_cols_iv, _, _ = prepare_continuous_covariates(df_pres21_raw, raw_age_cols, raw_emp_cols, label="isolated_pres21")
        run_iv_causal_analysis(df_pres21_pca, df_pres19_raw, dm_cols_iv)

    # ---------------------------------------------------------
    # PART B: POOLED ELECTION ANALYSIS
    # ---------------------------------------------------------
    log_to_file("\n\n" + "#"*70 + "\n PART B: POOLED PANEL ELECTION PIPELINE\n" + "#"*70)

    df_pooled_pca, dm_cols_pooled, _, _ = prepare_continuous_covariates(df_raw_pooled, raw_age_cols, raw_emp_cols, label="pooled_panel")

    if not df_pres19_raw.empty:
        run_pooled_iv_panel(df_pooled_pca, df_pres19_raw, dm_cols_pooled)

    print(f"\n[+] All tasks completed. Full statistical logs exported to -> {LOG_FILE_PATH}")


--- Loading Data for 19th Presidential Election (2017) ---


/tmp/ipykernel_4025/3539887745.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: NO DATA FOUND
  > Employment Data: Loaded successfully
  > Polling Stations: NO DATA FOUND
  > Processed 3537 distinct dong-level observations.

--- Loading Data for 21st General Election (2020) ---


/tmp/ipykernel_4025/3539887745.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_21.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 5519 distinct dong-level observations.

--- Loading Data for 22nd General Election (2024) ---


...

  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_22.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 4513 distinct dong-level observations.

--- Loading Data for 20th Presidential Election (2022) ---


/tmp/ipykernel_4025/3539887745.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_pres20.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 5976 distinct dong-level observations.

--- Loading Data for 21st Presidential Election (2025) ---


/tmp/ipykernel_4025/3539887745.py:328: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['total_voting_pop'] = df[[c for c in voting_age_cols if c in df.columns]].sum(axis=1)


  > Census Data: Loaded successfully
  > APT Price Data: Loaded from Cache (output_data/processed_apt_data_pres21.csv)
  > Employment Data: Loaded successfully
  > Polling Stations: Loaded successfully
  > Processed 6050 distinct dong-level observations.


######################################################################
 PART A: ISOLATED SINGLE-ELECTION PIPELINE
######################################################################
  -> Saved PCA Loadings for isolated_21 to output_data/pca_loadings_isolated_21.csv

  SINGLE ELECTION ANALYSIS: Election_21

--- Sub-Target: turnout ---
[Formula]: turnout ~ pc1_dm + pc2_dm + pc3_dm + pc4_dm + pc5_dm + pc6_dm + pc7_dm + pc8_dm + pc9_dm + pc10_dm + pc11_dm + C(province_tag) + voters_per_station_dm
                            WLS Regression Results                            
Dep. Variable:                turnout   R-squared:                       0.291
Model:                            WLS   Adj. R-squared:                  0.288
Metho